In [7]:
import os
import re
import gc
import json
import tarfile
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from lxml import etree
from tqdm import tqdm


# =========================================================
# CONFIG
# =========================================================

TOPICS_TGZ_PATH = os.getenv(
    "TOPICS_TGZ_PATH",
    "/kaggle/input/datasets/djnhngocduc/patent-benchmark/02_topics.tgz"
)

BASE_DIR = Path(os.getenv("PAC_TOPIC_WORK_DIR", "/kaggle/working/clefip2011_pac_topics"))
RAW_DIR = BASE_DIR / "raw"
EXTRACTED_DIR = BASE_DIR / "extracted"
OUT_DIR = BASE_DIR / "processed"
LOG_DIR = BASE_DIR / "logs"

TRAIN_XML_DIR = EXTRACTED_DIR / "training-pac-xml"
TEST_XML_DIR = EXTRACTED_DIR / "test-pac-xml"

TRAIN_OUT = OUT_DIR / "pac_training_topics.parquet"
TEST_OUT = OUT_DIR / "pac_test_topics.parquet"
QRELS_OUT = OUT_DIR / "pac_test_qrels.parquet"
SUMMARY_OUT = LOG_DIR / "summary.json"

for d in [RAW_DIR, EXTRACTED_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("TOPICS_TGZ_PATH =", TOPICS_TGZ_PATH)
print("BASE_DIR        =", BASE_DIR)
print("RAW_DIR         =", RAW_DIR)
print("OUT_DIR         =", OUT_DIR)


# =========================================================
# BASIC HELPERS
# =========================================================

def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)
    path.mkdir(parents=True, exist_ok=True)


def normalize_space(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def safe_text_join(parts):
    parts = [normalize_space(x) for x in parts if x and normalize_space(x)]
    return " ".join(parts).strip()


def local_name(tag):
    if tag is None:
        return ""
    if not isinstance(tag, str):
        return ""
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def iter_real_elements(root):
    for elem in root.iter():
        if isinstance(getattr(elem, "tag", None), str):
            yield elem


def collect_elem_text(elem, max_words=None):
    parts = []
    total_words = 0

    try:
        iterator = elem.itertext()
    except Exception:
        return "", 0

    for chunk in iterator:
        chunk = normalize_space(chunk)
        if not chunk:
            continue

        words = chunk.split()

        if max_words is not None:
            remaining = max_words - total_words
            if remaining <= 0:
                break
            if len(words) > remaining:
                words = words[:remaining]

        if words:
            parts.append(" ".join(words))
            total_words += len(words)

        if max_words is not None and total_words >= max_words:
            break

    return " ".join(parts).strip(), total_words


def unique_list(seq):
    seen = set()
    out = []

    for x in seq:
        s = normalize_space(x)
        if s and s not in seen:
            seen.add(s)
            out.append(s)

    return out


def word_count(text):
    text = normalize_space(text)
    if not text:
        return 0
    return len(text.split())


def truncate_words(text, max_words):
    text = normalize_space(text)
    if not text:
        return ""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])


def build_retrieval_text(title, abstract, claims, description):
    return safe_text_join([
        f"[TITLE] {truncate_words(title, 80)}" if title else "",
        f"[ABSTRACT] {truncate_words(abstract, 300)}" if abstract else "",
        f"[CLAIMS] {truncate_words(claims, 500)}" if claims else "",
        f"[DESCRIPTION] {truncate_words(description, 300)}" if description else "",
    ])


def build_full_text(title, abstract, claims, description):
    return safe_text_join([
        f"[TITLE] {title}" if title else "",
        f"[ABSTRACT] {abstract}" if abstract else "",
        f"[CLAIMS] {claims}" if claims else "",
        f"[DESCRIPTION] {description}" if description else "",
    ])


def split_codes(text):
    if not text:
        return []
    parts = re.split(r"[;\n\r]+", text)
    return [normalize_space(p) for p in parts if normalize_space(p)]


def clean_valid_yyyymmdd(text):
    text = normalize_space(text)
    if not text:
        return ""

    m = re.search(r"\b(\d{8})\b", text)
    if not m:
        return ""

    s = m.group(1)
    try:
        datetime.strptime(s, "%Y%m%d")
        return s
    except ValueError:
        return ""


def get_direct_child_text(elem, child_name):
    child_name = child_name.lower()

    for child in elem:
        tag = local_name(getattr(child, "tag", "")).lower()
        if tag == child_name:
            text, _ = collect_elem_text(child)
            if text:
                return text

    return ""


def extract_document_id_fields(container_elem):
    best = {
        "country": "",
        "doc_number": "",
        "kind": "",
        "date": "",
    }
    best_score = -1

    for elem in container_elem.iter():
        if local_name(getattr(elem, "tag", "")).lower() != "document-id":
            continue

        country = get_direct_child_text(elem, "country")
        doc_number = get_direct_child_text(elem, "doc-number")
        kind = get_direct_child_text(elem, "kind")
        date = get_direct_child_text(elem, "date")

        score = int(bool(country)) + int(bool(doc_number)) + int(bool(kind)) + int(bool(date))

        if score > best_score:
            best = {
                "country": country,
                "doc_number": doc_number,
                "kind": kind,
                "date": date,
            }
            best_score = score

    return best


def extract_priority_date(container_elem):
    for elem in container_elem.iter():
        if local_name(getattr(elem, "tag", "")).lower() == "date":
            text, _ = collect_elem_text(elem)
            cleaned = clean_valid_yyyymmdd(text)
            if cleaned:
                return cleaned

    return ""


def has_ancestor_with_tag(elem, tag_set):
    try:
        for parent in elem.iterancestors():
            parent_tag = local_name(getattr(parent, "tag", "")).lower()
            if parent_tag in tag_set:
                return True
    except Exception:
        return False

    return False


# =========================================================
# ARCHIVE HELPERS
# =========================================================

def extract_pac_files_from_topics_tgz():
    """
    Extract toàn bộ 02_topics.tgz vào RAW_DIR, sau đó tìm trực tiếp:
    - training_clef-ip-2011_PAC.7z
    - topics_clef-ip-2011_PAC.7z
    - relass_clef-ip-2011-PAC.txt

    Cách này tránh lỗi bắt nhầm folder training-pac/test-pac rỗng.
    """
    reset_dir(RAW_DIR)

    tgz_path = Path(TOPICS_TGZ_PATH)
    if not tgz_path.exists():
        raise FileNotFoundError(f"Cannot find TOPICS_TGZ_PATH: {tgz_path}")

    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall(path=RAW_DIR, filter="data")

    print("Extracted 02_topics.tgz to:", RAW_DIR)

    all_files = list(RAW_DIR.rglob("*"))
    print("Total files/folders after extracting tgz:", len(all_files))

    print("Preview extracted paths:")
    for p in all_files[:50]:
        print(" ", p)

    train_7z_candidates = [
        p for p in RAW_DIR.rglob("*.7z")
        if "training" in p.name.lower()
        and "pac" in p.name.lower()
        and "img" not in p.name.lower()
    ]

    test_7z_candidates = [
        p for p in RAW_DIR.rglob("*.7z")
        if (
            "topics" in p.name.lower()
            or "test" in str(p).lower()
        )
        and "pac" in p.name.lower()
        and "img" not in p.name.lower()
    ]

    relass_candidates = [
        p for p in RAW_DIR.rglob("*.txt")
        if "relass" in p.name.lower()
        and "pac" in p.name.lower()
        and "img" not in p.name.lower()
    ]

    print("train_7z_candidates:", train_7z_candidates)
    print("test_7z_candidates:", test_7z_candidates)
    print("relass_candidates:", relass_candidates)

    if not train_7z_candidates:
        raise FileNotFoundError(f"Cannot find training PAC .7z under {RAW_DIR}")

    if not test_7z_candidates:
        raise FileNotFoundError(f"Cannot find test/topics PAC .7z under {RAW_DIR}")

    train_7z = sorted(train_7z_candidates)[0]
    test_7z = sorted(test_7z_candidates)[0]
    relass_file = sorted(relass_candidates)[0] if relass_candidates else None

    return train_7z, test_7z, relass_file


def extract_7z(archive_path: Path, out_dir: Path):
    reset_dir(out_dir)

    cmd = ["7z", "x", str(archive_path), f"-o{out_dir}", "-y"]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"7z extract failed: {archive_path}")

    xml_files = sorted(out_dir.rglob("*.xml"))
    print(f"Extracted XML from {archive_path.name}: {len(xml_files)}")

    return xml_files


# =========================================================
# PAC TOPIC XML PARSER
# =========================================================

def parse_pac_topic_xml(xml_path: Path, split: str):
    parser = etree.XMLParser(recover=True, huge_tree=True)
    root = etree.parse(str(xml_path), parser=parser).getroot()

    topic_id = xml_path.stem
    doc_id = topic_id

    lang = ""
    country = root.attrib.get("country", "").strip().lower()
    kind = root.attrib.get("kind", "").strip().lower()

    for attr in ["lang", "{http://www.w3.org/XML/1998/namespace}lang"]:
        val = root.attrib.get(attr)
        if val:
            lang = str(val).strip().lower()
            break

    title_tags = {"invention-title", "title", "b540", "ti"}
    abstract_tags = {"abstract", "abst", "sdoab"}
    claims_tags = {"claims", "claim", "clms"}
    description_tags = {"description", "desc", "detdesc", "txt"}

    ipc_tags = {
        "classification-ipc",
        "ipc",
        "main-classification",
        "further-classification",
        "classification-ipcr",
        "classifications-ipcr",
    }

    assignee_tags = {
        "applicant",
        "assignee",
        "applicants",
        "assignees",
        "orgname",
    }

    inventor_tags = {
        "inventor",
        "inventors",
    }

    citation_tags = {
        "citation",
        "patcit",
        "nplcit",
    }

    title = ""
    abstract_parts = []
    claims_parts = []
    description_parts = []

    publication_doc_number = ""
    publication_kind = ""
    publication_date = ""

    application_doc_number = ""
    application_kind = ""
    application_date = ""

    priority_date = ""

    ipc_codes = []
    assignees = []
    inventors = []
    citations = []

    for elem in iter_real_elements(root):
        tag_lower = local_name(elem.tag).lower()
        if not tag_lower:
            continue

        if not lang:
            for attr_name, attr_val in elem.attrib.items():
                if "lang" in str(attr_name).lower() and attr_val:
                    lang = str(attr_val).strip().lower()
                    break

        if not country and tag_lower == "country":
            text, _ = collect_elem_text(elem)
            if text:
                country = text.lower()
            continue

        if not kind and tag_lower == "kind":
            text, _ = collect_elem_text(elem)
            if text:
                kind = text.lower()
            continue

        if tag_lower in title_tags and not title and not has_ancestor_with_tag(elem, title_tags):
            text, _ = collect_elem_text(elem)
            if text:
                title = text
            continue

        if tag_lower in abstract_tags and not has_ancestor_with_tag(elem, abstract_tags):
            text, _ = collect_elem_text(elem)
            if text:
                abstract_parts.append(text)
            continue

        if tag_lower in claims_tags and not has_ancestor_with_tag(elem, claims_tags):
            text, _ = collect_elem_text(elem)
            if text:
                claims_parts.append(text)
            continue

        if tag_lower in description_tags and not has_ancestor_with_tag(elem, description_tags):
            text, _ = collect_elem_text(elem, max_words=None)
            if text:
                description_parts.append(text)
            continue

        if tag_lower == "publication-reference" and (not publication_doc_number and not publication_date):
            fields = extract_document_id_fields(elem)
            publication_doc_number = normalize_space(fields["doc_number"])
            publication_kind = normalize_space(fields["kind"]).lower()
            publication_date = clean_valid_yyyymmdd(fields["date"])
            continue

        if tag_lower == "application-reference" and (not application_doc_number and not application_date):
            fields = extract_document_id_fields(elem)
            application_doc_number = normalize_space(fields["doc_number"])
            application_kind = normalize_space(fields["kind"]).lower()
            application_date = clean_valid_yyyymmdd(fields["date"])
            continue

        if tag_lower == "priority-claim" and not priority_date:
            priority_date = extract_priority_date(elem)
            continue

        if tag_lower in ipc_tags:
            text, _ = collect_elem_text(elem)
            if text:
                ipc_codes.extend(split_codes(text))
            continue

        if tag_lower in assignee_tags:
            text, _ = collect_elem_text(elem)
            if text and len(text) >= 2:
                assignees.append(text)
            continue

        if tag_lower in inventor_tags:
            text, _ = collect_elem_text(elem)
            if text and len(text) >= 2:
                inventors.append(text)
            continue

        if tag_lower in citation_tags:
            text, _ = collect_elem_text(elem)
            if text and len(text) >= 2:
                citations.append(text)
            continue

    abstract = safe_text_join(abstract_parts)
    claims = safe_text_join(claims_parts)
    description = safe_text_join(description_parts)

    retrieval_text = build_retrieval_text(title, abstract, claims, description)
    full_text = build_full_text(title, abstract, claims, description)

    ipc_codes = unique_list(ipc_codes)
    assignees = unique_list(assignees)
    inventors = unique_list(inventors)
    citations = unique_list(citations)

    return {
        "split": split,
        "topic_id": topic_id,
        "doc_id": doc_id,

        "lang": lang,
        "country": country,
        "kind": kind,

        "publication_doc_number": publication_doc_number,
        "publication_kind": publication_kind,
        "publication_date": publication_date,

        "application_doc_number": application_doc_number,
        "application_kind": application_kind,
        "application_date": application_date,

        "priority_date": priority_date,

        "title": title,
        "abstract": abstract,
        "claims": claims,
        "description": description,

        "retrieval_text": retrieval_text,
        "full_text": full_text,

        "ipc_codes": ipc_codes,
        "assignees": assignees,
        "inventors": inventors,
        "citations": citations,

        "title_word_count": word_count(title),
        "abstract_word_count": word_count(abstract),
        "claims_word_count": word_count(claims),
        "description_word_count": word_count(description),
        "retrieval_text_word_count": word_count(retrieval_text),
        "full_text_word_count": word_count(full_text),

        "num_ipc_codes": len(ipc_codes),
        "num_assignees": len(assignees),
        "num_inventors": len(inventors),
        "num_citations": len(citations),

        "source_path": str(xml_path),
    }


def parse_topic_folder(xml_dir: Path, split: str):
    xml_files = sorted(xml_dir.rglob("*.xml"))
    records = []
    errors = []

    for p in tqdm(xml_files, desc=f"Parsing {split} PAC topics"):
        try:
            records.append(parse_pac_topic_xml(p, split=split))
        except Exception as e:
            errors.append({
                "split": split,
                "source_path": str(p),
                "error": repr(e),
            })

    return records, errors


# =========================================================
# QRELS / RELEVANCE PARSER
# =========================================================

def parse_relass_file(path: Path):
    records = []

    if path is None or not Path(path).exists():
        return pd.DataFrame()

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw or raw.startswith("#"):
                continue

            parts = raw.split()

            if len(parts) < 2:
                continue

            topic_id = ""
            candidate_doc_id = ""
            relevance = 1

            # Format qrels phổ biến: topic_id 0 doc_id relevance
            if len(parts) >= 4 and parts[1] == "0":
                topic_id = parts[0]
                candidate_doc_id = parts[2]
                try:
                    relevance = int(float(parts[3]))
                except Exception:
                    relevance = 1

            # Format: topic_id doc_id relevance
            elif len(parts) >= 3:
                topic_id = parts[0]
                candidate_doc_id = parts[1]
                try:
                    relevance = int(float(parts[2]))
                except Exception:
                    relevance = 1

            # Format: topic_id doc_id
            else:
                topic_id = parts[0]
                candidate_doc_id = parts[1]
                relevance = 1

            records.append({
                "topic_id": topic_id,
                "candidate_doc_id": candidate_doc_id,
                "relevance": relevance,
                "line_no": line_no,
                "raw_line": raw,
            })

    return pd.DataFrame(records)


# =========================================================
# MAIN
# =========================================================

def main():
    train_7z, test_7z, relass_file = extract_pac_files_from_topics_tgz()

    print("train_7z   =", train_7z)
    print("test_7z    =", test_7z)
    print("relass_file =", relass_file)

    train_xml_files = extract_7z(train_7z, TRAIN_XML_DIR)
    test_xml_files = extract_7z(test_7z, TEST_XML_DIR)

    train_records, train_errors = parse_topic_folder(TRAIN_XML_DIR, split="train")
    test_records, test_errors = parse_topic_folder(TEST_XML_DIR, split="test")

    df_train = pd.DataFrame(train_records)
    df_test = pd.DataFrame(test_records)

    df_train.to_parquet(TRAIN_OUT, index=False, compression="zstd")
    df_test.to_parquet(TEST_OUT, index=False, compression="zstd")

    df_qrels = parse_relass_file(relass_file)
    if len(df_qrels) > 0:
        df_qrels.to_parquet(QRELS_OUT, index=False, compression="zstd")

    all_errors = train_errors + test_errors
    error_path = LOG_DIR / "parse_errors.jsonl"

    with open(error_path, "w", encoding="utf-8") as f:
        for err in all_errors:
            f.write(json.dumps(err, ensure_ascii=False) + "\n")

    summary = {
        "topics_tgz_path": str(TOPICS_TGZ_PATH),
        "train_7z": str(train_7z),
        "test_7z": str(test_7z),
        "relass_file": str(relass_file) if relass_file else None,

        "num_train_xml": len(train_xml_files),
        "num_test_xml": len(test_xml_files),

        "num_train_records": len(df_train),
        "num_test_records": len(df_test),
        "num_qrels": len(df_qrels),

        "num_train_errors": len(train_errors),
        "num_test_errors": len(test_errors),

        "train_out": str(TRAIN_OUT),
        "test_out": str(TEST_OUT),
        "qrels_out": str(QRELS_OUT),
        "error_path": str(error_path),
    }

    with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nDONE")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nTrain sample:")
    if len(df_train) > 0:
        print(df_train[[
            "topic_id",
            "title_word_count",
            "abstract_word_count",
            "claims_word_count",
            "description_word_count",
            "full_text_word_count",
            "title"
        ]].head())

    print("\nTest sample:")
    if len(df_test) > 0:
        print(df_test[[
            "topic_id",
            "title_word_count",
            "abstract_word_count",
            "claims_word_count",
            "description_word_count",
            "full_text_word_count",
            "title"
        ]].head())

    print("\nQrels sample:")
    if len(df_qrels) > 0:
        print(df_qrels.head())


if __name__ == "__main__":
    main()

TOPICS_TGZ_PATH = /kaggle/input/datasets/djnhngocduc/patent-benchmark/02_topics.tgz
BASE_DIR        = /kaggle/working/clefip2011_pac_topics
RAW_DIR         = /kaggle/working/clefip2011_pac_topics/raw
OUT_DIR         = /kaggle/working/clefip2011_pac_topics/processed
Extracted 02_topics.tgz to: /kaggle/working/clefip2011_pac_topics/raw
Total files/folders after extracting tgz: 29
Preview extracted paths:
  /kaggle/working/clefip2011_pac_topics/raw/02_topics
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/test-img-pac
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/test-cls
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/test-img-cls
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/test-pac
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/training-img-cls_correction
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/training-pac
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/training-img-cls
  /kaggle/working/clefip2011_pac_topics/raw/02_topics/test-i

Parsing test PAC topics: 100%|██████████| 3974/3974 [01:48<00:00, 36.72it/s]



DONE
{
  "topics_tgz_path": "/kaggle/input/datasets/djnhngocduc/patent-benchmark/02_topics.tgz",
  "train_7z": "/kaggle/working/clefip2011_pac_topics/raw/02_topics/training-pac/training_clef-ip-2011_PAC.7z",
  "test_7z": "/kaggle/working/clefip2011_pac_topics/raw/02_topics/test-pac/topics_clef-ip-2011_PAC.7z",
  "relass_file": "/kaggle/working/clefip2011_pac_topics/raw/02_topics/test-pac/relass_clef-ip-2011-PAC.txt",
  "num_train_xml": 301,
  "num_test_xml": 3974,
  "num_train_records": 301,
  "num_test_records": 3974,
  "num_qrels": 28600,
  "num_train_errors": 0,
  "num_test_errors": 0,
  "train_out": "/kaggle/working/clefip2011_pac_topics/processed/pac_training_topics.parquet",
  "test_out": "/kaggle/working/clefip2011_pac_topics/processed/pac_test_topics.parquet",
  "qrels_out": "/kaggle/working/clefip2011_pac_topics/processed/pac_test_qrels.parquet",
  "error_path": "/kaggle/working/clefip2011_pac_topics/logs/parse_errors.jsonl"
}

Train sample:
        topic_id  title_word_count

In [8]:
import os
import re
import json
from pathlib import Path

import pandas as pd

OUT_DIR = Path("/kaggle/working/clefip2011_pac_topics/processed")

BENCHMARK_PRESET = os.getenv("BENCHMARK_PRESET", "quick").strip().lower()
if BENCHMARK_PRESET in {"quick", "quick_200", "200"}:
    DEFAULT_NON_CITATION_QUERY_COUNT = 100
    DEFAULT_CITATION_QUERY_COUNT = 100
else:
    DEFAULT_NON_CITATION_QUERY_COUNT = 200
    DEFAULT_CITATION_QUERY_COUNT = 200

TARGET_NON_CITATION_QUERY_COUNT = int(os.getenv("TARGET_NON_CITATION_QUERY_COUNT", str(DEFAULT_NON_CITATION_QUERY_COUNT)))
TARGET_CITATION_QUERY_COUNT = int(os.getenv("TARGET_CITATION_QUERY_COUNT", str(DEFAULT_CITATION_QUERY_COUNT)))
TARGET_BENCHMARK_QUERY_COUNT = TARGET_NON_CITATION_QUERY_COUNT + TARGET_CITATION_QUERY_COUNT

TRAIN_RAW_PATH = OUT_DIR / "pac_training_topics.parquet"
TEST_RAW_PATH = OUT_DIR / "pac_test_topics.parquet"
QRELS_RAW_PATH = OUT_DIR / "pac_test_qrels.parquet"

TRAIN_CLEAN_PATH = OUT_DIR / "pac_training_topics_clean.parquet"
TEST_CLEAN_PATH = OUT_DIR / "pac_test_topics_clean.parquet"
QRELS_CLEAN_PATH = OUT_DIR / "pac_test_qrels_clean.parquet"

BENCHMARK_TEST_PATH = OUT_DIR / f"pac_test_topics_benchmark_{TARGET_BENCHMARK_QUERY_COUNT}.parquet"
BENCHMARK_QRELS_PATH = OUT_DIR / f"pac_test_qrels_benchmark_{TARGET_BENCHMARK_QUERY_COUNT}.parquet"
BENCHMARK_DOC_IDS_PATH = OUT_DIR / f"benchmark_{TARGET_BENCHMARK_QUERY_COUNT}_queries_index_doc_ids.parquet"
BENCHMARK_SUMMARY_PATH = OUT_DIR / f"benchmark_{TARGET_BENCHMARK_QUERY_COUNT}_queries_summary.json"


def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value).strip()


def normalize_list(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if hasattr(value, "tolist") and not isinstance(value, str):
        value = value.tolist()
    if isinstance(value, list):
        out = []
        seen = set()
        for item in value:
            text = normalize_text(item)
            if text and text not in seen:
                seen.add(text)
                out.append(text)
        return out
    text = normalize_text(value)
    return [text] if text else []


def canonical_doc_id(value):
    text = normalize_text(value).upper().replace(" ", "-").replace("_", "-")
    text = re.sub(r"-+", "-", text).strip("-")
    match = re.match(r"^([A-Z]{2})-?(\d+)(?:-[A-Z][0-9A-Z]*)?$", text)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return text


def topic_has_citations(row):
    try:
        citation_count = int(row.get("num_citations", 0) or 0)
    except Exception:
        citation_count = 0
    return citation_count > 0 or bool(normalize_list(row.get("citations")))


def select_topic_subset(topic_rows, quota):
    selected_rows = []
    for topic_row in sorted(topic_rows, key=lambda item: (len(item["candidate_ids"]), item["topic_id"])):
        if len(selected_rows) >= quota:
            break
        if topic_row["candidate_ids"]:
            selected_rows.append(topic_row)
    return selected_rows


train = pd.read_parquet(TRAIN_RAW_PATH)
test = pd.read_parquet(TEST_RAW_PATH)
qrels = pd.read_parquet(QRELS_RAW_PATH)

print("Before:")
print("train:", train.shape)
print("test:", test.shape)
print("qrels:", qrels.shape)
print("TARGET_NON_CITATION_QUERY_COUNT:", TARGET_NON_CITATION_QUERY_COUNT)
print("TARGET_CITATION_QUERY_COUNT:", TARGET_CITATION_QUERY_COUNT)
print("TARGET_BENCHMARK_QUERY_COUNT:", TARGET_BENCHMARK_QUERY_COUNT)

train_clean = train[train["full_text_word_count"] > 0].copy()
test_clean = test[test["full_text_word_count"] > 0].copy()

train_clean["topic_id"] = train_clean["topic_id"].astype(str).str.strip()
test_clean["topic_id"] = test_clean["topic_id"].astype(str).str.strip()
qrels["topic_id"] = qrels["topic_id"].astype(str).str.strip()
qrels["candidate_doc_id"] = qrels["candidate_doc_id"].astype(str).str.strip()

test_clean["has_citations"] = test_clean.apply(topic_has_citations, axis=1)
test_clean["citation_group"] = test_clean["has_citations"].map({
    True: "with_citation",
    False: "without_citation",
})

valid_topics = set(test_clean["topic_id"])
qrels_clean = qrels[qrels["topic_id"].isin(valid_topics)].copy()
qrels_clean["candidate_canonical_doc_id"] = qrels_clean["candidate_doc_id"].map(canonical_doc_id)
qrels_clean = qrels_clean[qrels_clean["candidate_canonical_doc_id"] != ""].copy()
qrels_clean = qrels_clean.merge(
    test_clean[["topic_id", "has_citations", "num_citations"]],
    on="topic_id",
    how="left",
)
qrels_clean["has_citations"] = qrels_clean["has_citations"].fillna(False).astype(bool)
qrels_clean["requires_kind_code_expansion"] = qrels_clean["has_citations"]

print("After clean:")
print("train_clean:", train_clean.shape)
print("test_clean:", test_clean.shape)
print("qrels_clean:", qrels_clean.shape)
print("test topics with citation:", int(test_clean["has_citations"].sum()))
print("test topics without citation:", int((~test_clean["has_citations"]).sum()))

train_clean.to_parquet(TRAIN_CLEAN_PATH, index=False, compression="zstd")
test_clean.to_parquet(TEST_CLEAN_PATH, index=False, compression="zstd")
qrels_clean.to_parquet(QRELS_CLEAN_PATH, index=False, compression="zstd")

topic_to_candidate_ids = {
    topic_id: sorted(set(group["candidate_canonical_doc_id"]))
    for topic_id, group in qrels_clean.groupby("topic_id", sort=True)
}

topic_rows = []
for row in test_clean.to_dict(orient="records"):
    topic_id = normalize_text(row.get("topic_id"))
    candidate_ids = topic_to_candidate_ids.get(topic_id, [])
    if not candidate_ids:
        continue
    topic_rows.append({
        "topic_id": topic_id,
        "has_citations": bool(row.get("has_citations")),
        "candidate_ids": candidate_ids,
    })

non_citation_topic_rows = [row for row in topic_rows if not row["has_citations"]]
citation_topic_rows = [row for row in topic_rows if row["has_citations"]]

selected_non_citation_rows = select_topic_subset(
    non_citation_topic_rows,
    TARGET_NON_CITATION_QUERY_COUNT,
)
selected_citation_rows = select_topic_subset(
    citation_topic_rows,
    TARGET_CITATION_QUERY_COUNT,
)

if len(selected_non_citation_rows) < TARGET_NON_CITATION_QUERY_COUNT:
    raise ValueError(
        f"Only {len(selected_non_citation_rows)} non-citation topics are available. "
        f"Requested {TARGET_NON_CITATION_QUERY_COUNT}."
    )

if len(selected_citation_rows) < TARGET_CITATION_QUERY_COUNT:
    raise ValueError(
        f"Only {len(selected_citation_rows)} citation topics are available. "
        f"Requested {TARGET_CITATION_QUERY_COUNT}."
    )

selected_topic_rows = selected_non_citation_rows + selected_citation_rows
selected_topic_ids = [row["topic_id"] for row in selected_topic_rows]
selected_topic_set = set(selected_topic_ids)

selected_qrels = qrels_clean[qrels_clean["topic_id"].isin(selected_topic_set)].copy()
selected_doc_ids = sorted(set(selected_qrels["candidate_canonical_doc_id"]))
doc_requires_kind_code = (
    selected_qrels.groupby("candidate_canonical_doc_id")["requires_kind_code_expansion"]
    .max()
    .to_dict()
)

index_doc_ids_df = pd.DataFrame({"doc_id": selected_doc_ids})
index_doc_ids_df["canonical_doc_id"] = index_doc_ids_df["doc_id"]
index_doc_ids_df["is_qrel_doc"] = True
index_doc_ids_df["source_type"] = "qrel"
index_doc_ids_df["requires_kind_code_expansion"] = (
    index_doc_ids_df["canonical_doc_id"].map(doc_requires_kind_code).fillna(False).astype(bool)
)
index_doc_ids_df.to_parquet(BENCHMARK_DOC_IDS_PATH, index=False, compression="zstd")

benchmark_test = test_clean[test_clean["topic_id"].isin(selected_topic_set)].copy()
benchmark_qrels = selected_qrels.sort_values(
    ["has_citations", "topic_id", "candidate_canonical_doc_id", "line_no"]
).reset_index(drop=True)
benchmark_test = benchmark_test.sort_values(["has_citations", "topic_id"]).reset_index(drop=True)

benchmark_test.to_parquet(BENCHMARK_TEST_PATH, index=False, compression="zstd")
benchmark_qrels.to_parquet(BENCHMARK_QRELS_PATH, index=False, compression="zstd")

summary = {
    "benchmark_preset": BENCHMARK_PRESET,
    "target_non_citation_query_count": TARGET_NON_CITATION_QUERY_COUNT,
    "target_citation_query_count": TARGET_CITATION_QUERY_COUNT,
    "target_benchmark_query_count": TARGET_BENCHMARK_QUERY_COUNT,
    "num_selected_topics": int(len(selected_topic_ids)),
    "num_selected_non_citation_topics": int(len(selected_non_citation_rows)),
    "num_selected_citation_topics": int(len(selected_citation_rows)),
    "num_benchmark_topics_rows": int(len(benchmark_test)),
    "num_benchmark_qrels_rows": int(len(benchmark_qrels)),
    "num_benchmark_unique_relevant_docs": int(len(selected_doc_ids)),
    "num_kind_code_expansion_docs": int(index_doc_ids_df["requires_kind_code_expansion"].sum()),
    "train_clean_path": str(TRAIN_CLEAN_PATH),
    "test_clean_path": str(TEST_CLEAN_PATH),
    "qrels_clean_path": str(QRELS_CLEAN_PATH),
    "benchmark_test_path": str(BENCHMARK_TEST_PATH),
    "benchmark_qrels_path": str(BENCHMARK_QRELS_PATH),
    "benchmark_doc_ids_path": str(BENCHMARK_DOC_IDS_PATH),
}

with open(BENCHMARK_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("BENCHMARK_PRESET =", BENCHMARK_PRESET)
print("Benchmark summary:")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Sample selected non-citation topic_ids:", [row["topic_id"] for row in selected_non_citation_rows[:10]])
print("Sample selected citation topic_ids:", [row["topic_id"] for row in selected_citation_rows[:10]])


Before:
train: (301, 34)
test: (3974, 34)
qrels: (28600, 5)
TARGET_NON_CITATION_QUERY_COUNT: 100
TARGET_CITATION_QUERY_COUNT: 100
TARGET_BENCHMARK_QUERY_COUNT: 200
After clean:
train_clean: (300, 34)
test_clean: (3973, 36)
qrels_clean: (28600, 9)
test topics with citation: 530
test topics without citation: 3443
BENCHMARK_PRESET = quick
Benchmark summary:
{
  "benchmark_preset": "quick",
  "target_non_citation_query_count": 100,
  "target_citation_query_count": 100,
  "target_benchmark_query_count": 200,
  "num_selected_topics": 200,
  "num_selected_non_citation_topics": 100,
  "num_selected_citation_topics": 100,
  "num_benchmark_topics_rows": 200,
  "num_benchmark_qrels_rows": 630,
  "num_benchmark_unique_relevant_docs": 624,
  "num_kind_code_expansion_docs": 327,
  "train_clean_path": "/kaggle/working/clefip2011_pac_topics/processed/pac_training_topics_clean.parquet",
  "test_clean_path": "/kaggle/working/clefip2011_pac_topics/processed/pac_test_topics_clean.parquet",
  "qrels_clean_

In [9]:
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path(os.getenv("PAC_TOPIC_WORK_DIR", "/kaggle/working/clefip2011_pac_topics"))
IN_DIR = BASE_DIR / "processed"
OUT_DIR = BASE_DIR / "query_understanding"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_PRESET = os.getenv("BENCHMARK_PRESET", "quick").strip().lower()
if BENCHMARK_PRESET in {"quick", "quick_200", "200"}:
    DEFAULT_NON_CITATION_QUERY_COUNT = 100
    DEFAULT_CITATION_QUERY_COUNT = 100
else:
    DEFAULT_NON_CITATION_QUERY_COUNT = 200
    DEFAULT_CITATION_QUERY_COUNT = 200
TARGET_NON_CITATION_QUERY_COUNT = int(os.getenv("TARGET_NON_CITATION_QUERY_COUNT", str(DEFAULT_NON_CITATION_QUERY_COUNT)))
TARGET_CITATION_QUERY_COUNT = int(os.getenv("TARGET_CITATION_QUERY_COUNT", str(DEFAULT_CITATION_QUERY_COUNT)))
TARGET_BENCHMARK_QUERY_COUNT = TARGET_NON_CITATION_QUERY_COUNT + TARGET_CITATION_QUERY_COUNT

TRAIN_TOPICS_PATH = Path(
    os.getenv(
        "TRAIN_TOPICS_PATH",
        str(IN_DIR / "pac_training_topics_clean.parquet")
    )
)

TEST_TOPICS_PATH = Path(
    os.getenv(
        "TEST_TOPICS_PATH",
        str(IN_DIR / f"pac_test_topics_benchmark_{TARGET_BENCHMARK_QUERY_COUNT}.parquet")
    )
)

QRELS_PATH = Path(
    os.getenv(
        "QRELS_PATH",
        str(IN_DIR / f"pac_test_qrels_benchmark_{TARGET_BENCHMARK_QUERY_COUNT}.parquet")
    )
)

TRAIN_PROFILE_OUT = OUT_DIR / "pac_train_query_profiles.parquet"
TEST_PROFILE_OUT = OUT_DIR / "pac_test_query_profiles.parquet"

TRAIN_VIEWS_OUT = OUT_DIR / "pac_train_query_views.parquet"
TEST_VIEWS_OUT = OUT_DIR / "pac_test_query_views.parquet"

SUMMARY_OUT = OUT_DIR / "summary.json"

# Short views: dùng cho retrieval nhanh/ổn định.
# Full fields vẫn được giữ nguyên, không cắt.
SHORT_TITLE_WORDS = int(os.getenv("SHORT_TITLE_WORDS", "80"))
SHORT_ABSTRACT_WORDS = int(os.getenv("SHORT_ABSTRACT_WORDS", "300"))
SHORT_CLAIMS_WORDS = int(os.getenv("SHORT_CLAIMS_WORDS", "900"))
SHORT_DESCRIPTION_WORDS = int(os.getenv("SHORT_DESCRIPTION_WORDS", "700"))

# Query embedding text mirror document index retrieval_text: title + abstract + full claims.
# 0 means keep the full section.
QUERY_DOC_EMB_TITLE_WORDS = int(os.getenv("QUERY_DOC_EMB_TITLE_WORDS", "0"))
QUERY_DOC_EMB_ABSTRACT_WORDS = int(os.getenv("QUERY_DOC_EMB_ABSTRACT_WORDS", "0"))
QUERY_DOC_EMB_CLAIMS_WORDS = int(os.getenv("QUERY_DOC_EMB_CLAIMS_WORDS", "0"))
QUERY_DOC_EMB_INCLUDE_DESCRIPTION = os.getenv("QUERY_DOC_EMB_INCLUDE_DESCRIPTION", "false").lower() == "true"
QUERY_DOC_EMB_DESCRIPTION_WORDS = int(os.getenv("QUERY_DOC_EMB_DESCRIPTION_WORDS", "0"))

print("BASE_DIR          =", BASE_DIR)
print("TRAIN_TOPICS_PATH =", TRAIN_TOPICS_PATH)
print("TEST_TOPICS_PATH  =", TEST_TOPICS_PATH)
print("QRELS_PATH        =", QRELS_PATH)
print("OUT_DIR           =", OUT_DIR)


# =========================================================
# HELPERS
# =========================================================

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_list(x: Any) -> List[str]:
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, list):
        out = []
        seen = set()

        for item in x:
            s = normalize_text(item)
            if s and s not in seen:
                seen.add(s)
                out.append(s)

        return out

    s = normalize_text(x)
    return [s] if s else []


def safe_join(parts: List[str]) -> str:
    parts = [normalize_text(p) for p in parts if normalize_text(p)]
    return " ".join(parts).strip()


def truncate_words(text: Any, max_words: Optional[int]) -> str:
    text = normalize_text(text)
    if not text:
        return ""

    if max_words is None or int(max_words) <= 0:
        return text

    words = text.split()

    if len(words) <= max_words:
        return text

    return " ".join(words[:max_words])


def word_count(text: Any) -> int:
    text = normalize_text(text)
    if not text:
        return 0
    return len(text.split())


def normalize_date(x: Any) -> str:
    s = normalize_text(x)

    if not s:
        return ""

    if re.fullmatch(r"\d{8}", s):
        return s

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return s.replace("-", "")

    return ""


def build_section_text(label: str, text: Any) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    return f"[{label}] {text}"


def build_title_abstract_full(row: Dict[str, Any]) -> str:
    title = normalize_text(row.get("title"))
    abstract = normalize_text(row.get("abstract"))

    return safe_join([title, abstract])


def build_title_abstract_short(row: Dict[str, Any]) -> str:
    title = truncate_words(row.get("title"), SHORT_TITLE_WORDS)
    abstract = truncate_words(row.get("abstract"), SHORT_ABSTRACT_WORDS)

    return safe_join([title, abstract])


def build_claims_full(row: Dict[str, Any]) -> str:
    claims = normalize_text(row.get("claims"))
    return claims


def build_claims_short(row: Dict[str, Any]) -> str:
    claims = truncate_words(row.get("claims"), SHORT_CLAIMS_WORDS)
    return claims


def build_description_full(row: Dict[str, Any]) -> str:
    description = normalize_text(row.get("description"))
    return description


def build_description_short(row: Dict[str, Any]) -> str:
    description = truncate_words(row.get("description"), SHORT_DESCRIPTION_WORDS)
    return description


def build_combined_full(row: Dict[str, Any]) -> str:
    """
    combined_full = full_text nếu đã có.
    Nếu không có thì tự ghép từ title/abstract/claims/description.
    """
    full_text = normalize_text(row.get("full_text"))

    if full_text:
        return full_text

    return safe_join([
        build_title_abstract_full(row),
        build_claims_full(row),
        build_description_full(row),
    ])


def build_combined_short(row: Dict[str, Any]) -> str:
    """
    BM25 combined query view aligned with the index retrieval_text policy:
    title + abstract + claims, no description.
    """
    return safe_join([
        build_title_abstract_short(row),
        build_claims_short(row),
    ])


def build_query_embedding_text(row: Dict[str, Any]) -> str:
    """
    Query text for dense retrieval. Keep it aligned with document embeddings:
    title + abstract + full claims, no description by default.
    """
    parts = [
        truncate_words(row.get("title"), QUERY_DOC_EMB_TITLE_WORDS),
        truncate_words(row.get("abstract"), QUERY_DOC_EMB_ABSTRACT_WORDS),
        truncate_words(row.get("claims"), QUERY_DOC_EMB_CLAIMS_WORDS),
    ]

    if QUERY_DOC_EMB_INCLUDE_DESCRIPTION:
        parts.append(truncate_words(row.get("description"), QUERY_DOC_EMB_DESCRIPTION_WORDS))

    return safe_join(parts)


def build_metadata_text(row: Dict[str, Any]) -> str:
    ipc_codes = normalize_list(row.get("ipc_codes"))
    assignees = normalize_list(row.get("assignees"))
    inventors = normalize_list(row.get("inventors"))
    citations = normalize_list(row.get("citations"))

    country = normalize_text(row.get("country")).lower()
    kind = normalize_text(row.get("kind")).lower()
    lang = normalize_text(row.get("lang")).lower()

    parts = []

    if ipc_codes:
        parts.append("[IPC] " + " ; ".join(ipc_codes[:20]))
    if assignees:
        parts.append("[ASSIGNEE] " + " ; ".join(assignees[:20]))
    if inventors:
        parts.append("[INVENTOR] " + " ; ".join(inventors[:20]))
    if citations:
        parts.append("[CITATION] " + " ; ".join(citations[:30]))
    if country:
        parts.append(f"[COUNTRY] {country}")
    if kind:
        parts.append(f"[KIND] {kind}")
    if lang:
        parts.append(f"[LANG] {lang}")

    return safe_join(parts)


def extract_main_ipc(ipc_codes: List[str]) -> str:
    ipc_codes = normalize_list(ipc_codes)
    if not ipc_codes:
        return ""
    return ipc_codes[0]


def pick_prior_art_cutoff(row: Dict[str, Any]) -> str:
    """
    Mốc lọc prior art:
    ưu tiên priority_date > application_date > publication_date.
    """
    priority_date = normalize_date(row.get("priority_date"))
    application_date = normalize_date(row.get("application_date"))
    publication_date = normalize_date(row.get("publication_date"))

    if priority_date:
        return priority_date
    if application_date:
        return application_date
    if publication_date:
        return publication_date

    return ""


def classify_query_strength(row: Dict[str, Any]) -> str:
    title_wc = int(row.get("title_word_count", 0) or 0)
    abstract_wc = int(row.get("abstract_word_count", 0) or 0)
    claims_wc = int(row.get("claims_word_count", 0) or 0)
    desc_wc = int(row.get("description_word_count", 0) or 0)

    if title_wc > 0 and abstract_wc >= 80 and claims_wc >= 300:
        return "strong"

    if title_wc > 0 and abstract_wc >= 40 and claims_wc >= 100:
        return "medium"

    if title_wc > 0 or abstract_wc > 0 or claims_wc > 0 or desc_wc > 0:
        return "weak"

    return "empty"


# =========================================================
# PROFILE BUILDING
# =========================================================

def build_query_profile_row(row: Dict[str, Any], split: str) -> Dict[str, Any]:
    topic_id = normalize_text(row.get("topic_id"))
    query_doc_id = normalize_text(row.get("doc_id")) or topic_id

    title = normalize_text(row.get("title"))
    abstract = normalize_text(row.get("abstract"))
    claims = normalize_text(row.get("claims"))
    description = normalize_text(row.get("description"))

    retrieval_text = build_combined_short(row)
    full_text = build_combined_full(row)

    title_abstract_full = build_title_abstract_full(row)
    title_abstract_short = build_title_abstract_short(row)

    claims_full = build_claims_full(row)
    claims_short = build_claims_short(row)

    description_full = build_description_full(row)
    description_short = build_description_short(row)

    combined_full = full_text
    combined_short = retrieval_text

    query_embedding_text = build_query_embedding_text(row)
    metadata_text = build_metadata_text(row)

    ipc_codes = normalize_list(row.get("ipc_codes"))
    assignees = normalize_list(row.get("assignees"))
    inventors = normalize_list(row.get("inventors"))
    citations = normalize_list(row.get("citations"))

    prior_art_cutoff_date = pick_prior_art_cutoff(row)

    return {
        "split": split,
        "topic_id": topic_id,
        "query_doc_id": query_doc_id,
        "input_type": "patent_document",

        # Original/full source fields: không cắt
        "title": title,
        "abstract": abstract,
        "claims": claims,
        "description": description,
        "retrieval_text": retrieval_text,
        "full_text": full_text,

        # Full views: không cắt
        "query_title_abstract_full": title_abstract_full,
        "query_claims_full": claims_full,
        "query_description_full": description_full,
        "query_combined_full": combined_full,

        # Short views: dùng cho retrieval baseline
        "query_title_abstract_short": title_abstract_short,
        "query_claims_short": claims_short,
        "query_description_short": description_short,
        "query_combined_short": combined_short,
        "query_embedding_text": query_embedding_text,

        # Metadata view
        "query_metadata_text": metadata_text,

        # Metadata fields
        "lang": normalize_text(row.get("lang")).lower(),
        "country": normalize_text(row.get("country")).lower(),
        "kind": normalize_text(row.get("kind")).lower(),

        "publication_doc_number": normalize_text(row.get("publication_doc_number")),
        "publication_kind": normalize_text(row.get("publication_kind")).lower(),
        "publication_date": normalize_date(row.get("publication_date")),

        "application_doc_number": normalize_text(row.get("application_doc_number")),
        "application_kind": normalize_text(row.get("application_kind")).lower(),
        "application_date": normalize_date(row.get("application_date")),

        "priority_date": normalize_date(row.get("priority_date")),
        "prior_art_cutoff_date": prior_art_cutoff_date,

        "ipc_codes": ipc_codes,
        "main_ipc": extract_main_ipc(ipc_codes),
        "assignees": assignees,
        "inventors": inventors,
        "citations": citations,

        # Diagnostics
        "query_strength": classify_query_strength(row),

        "title_word_count": word_count(title),
        "abstract_word_count": word_count(abstract),
        "claims_word_count": word_count(claims),
        "description_word_count": word_count(description),
        "retrieval_text_word_count": word_count(retrieval_text),
        "full_text_word_count": word_count(full_text),

        "query_title_abstract_full_word_count": word_count(title_abstract_full),
        "query_claims_full_word_count": word_count(claims_full),
        "query_description_full_word_count": word_count(description_full),
        "query_combined_full_word_count": word_count(combined_full),

        "query_title_abstract_short_word_count": word_count(title_abstract_short),
        "query_claims_short_word_count": word_count(claims_short),
        "query_description_short_word_count": word_count(description_short),
        "query_combined_short_word_count": word_count(combined_short),
        "query_embedding_text_word_count": word_count(query_embedding_text),

        "metadata_word_count": word_count(metadata_text),

        "num_ipc_codes": len(ipc_codes),
        "num_assignees": len(assignees),
        "num_inventors": len(inventors),
        "num_citations": len(citations),

        # Retrieval constraints for next step
        "exclude_doc_id": query_doc_id,
        "has_date_filter": bool(prior_art_cutoff_date),
        "has_ipc_filter": bool(ipc_codes),
        "has_assignee_filter": bool(assignees),
    }


def build_profiles(input_path: Path, split: str) -> pd.DataFrame:
    df = pd.read_parquet(input_path)

    # Safety: bỏ wrapper rỗng nếu còn
    if "full_text_word_count" in df.columns:
        df = df[df["full_text_word_count"] > 0].copy()

    records = []
    for row in df.to_dict(orient="records"):
        records.append(build_query_profile_row(row, split=split))

    return pd.DataFrame(records)


# =========================================================
# QUERY VIEWS
# =========================================================

def add_view(
    records: List[Dict[str, Any]],
    base: Dict[str, Any],
    query_view: str,
    query_text: str,
    view_group: str,
    retrieval_use: str,
    weight: float,
    retrieval_active: bool,
):
    query_text = normalize_text(query_text)

    if not query_text:
        return

    rec = dict(base)
    rec.update({
        "query_view": query_view,
        "query_text": query_text,
        "view_group": view_group,
        "retrieval_use": retrieval_use,
        "weight": float(weight),
        "retrieval_active": bool(retrieval_active),
        "query_word_count": word_count(query_text),
    })

    records.append(rec)


def build_query_views(profile_df: pd.DataFrame) -> pd.DataFrame:
    """
    Long-format query views.

    retrieval_active=True:
      dùng cho baseline hybrid retrieval đầu tiên.

    retrieval_active=False:
      lưu lại để debug/rerank/query-chunking sau.
    """
    records = []

    for row in profile_df.to_dict(orient="records"):
        base = {
            "split": row["split"],
            "topic_id": row["topic_id"],
            "query_doc_id": row["query_doc_id"],
            "input_type": row["input_type"],

            "lang": row["lang"],
            "country": row["country"],
            "kind": row["kind"],

            "prior_art_cutoff_date": row["prior_art_cutoff_date"],
            "exclude_doc_id": row["exclude_doc_id"],

            "main_ipc": row["main_ipc"],
            "ipc_codes": row["ipc_codes"],
            "assignees": row["assignees"],
            "inventors": row["inventors"],

            "query_strength": row["query_strength"],
        }

        # Recommended baseline views
        add_view(
            records=records,
            base=base,
            query_view="title_abstract_full",
            query_text=row["query_title_abstract_full"],
            view_group="section_full",
            retrieval_use="bm25_only",
            weight=1.00,
            retrieval_active=True,
        )

        add_view(
            records=records,
            base=base,
            query_view="claims_short",
            query_text=row["query_claims_short"],
            view_group="section_short",
            retrieval_use="bm25_only",
            weight=0.90,
            retrieval_active=True,
        )

        add_view(
            records=records,
            base=base,
            query_view="combined_short",
            query_text=row["query_combined_short"],
            view_group="combined_short",
            retrieval_use="bm25_only",
            weight=1.10,
            retrieval_active=True,
        )

        add_view(
            records=records,
            base=base,
            query_view="embedding_text",
            query_text=row["query_embedding_text"],
            view_group="embedding_text",
            retrieval_use="vector_only",
            weight=1.10,
            retrieval_active=True,
        )

        add_view(
            records=records,
            base=base,
            query_view="description_short",
            query_text=row["query_description_short"],
            view_group="section_short",
            retrieval_use="rerank_or_later",
            weight=0.0,
            retrieval_active=False,
        )

        add_view(
            records=records,
            base=base,
            query_view="metadata",
            query_text=row["query_metadata_text"],
            view_group="metadata",
            retrieval_use="filter_or_boost",
            weight=0.30,
            retrieval_active=True,
        )

        # Full views for rerank / later experiments
        add_view(
            records=records,
            base=base,
            query_view="claims_full",
            query_text=row["query_claims_full"],
            view_group="section_full",
            retrieval_use="rerank_or_chunked_query",
            weight=0.0,
            retrieval_active=False,
        )

        add_view(
            records=records,
            base=base,
            query_view="description_full",
            query_text=row["query_description_full"],
            view_group="section_full",
            retrieval_use="rerank_or_chunked_query",
            weight=0.0,
            retrieval_active=False,
        )

        add_view(
            records=records,
            base=base,
            query_view="combined_full",
            query_text=row["query_combined_full"],
            view_group="combined_full",
            retrieval_use="rerank_or_chunked_query",
            weight=0.0,
            retrieval_active=False,
        )

        # Short title/abstract is usually redundant because title_abstract_full is not too long,
        # but keep for ablation.
        add_view(
            records=records,
            base=base,
            query_view="title_abstract_short",
            query_text=row["query_title_abstract_short"],
            view_group="section_short",
            retrieval_use="ablation",
            weight=0.0,
            retrieval_active=False,
        )

    return pd.DataFrame(records)


# =========================================================
# MAIN
# =========================================================

def main():
    if not TRAIN_TOPICS_PATH.exists():
        raise FileNotFoundError(f"Missing train topics: {TRAIN_TOPICS_PATH}")
    if not TEST_TOPICS_PATH.exists():
        raise FileNotFoundError(f"Missing test topics: {TEST_TOPICS_PATH}")
    if not QRELS_PATH.exists():
        raise FileNotFoundError(f"Missing qrels: {QRELS_PATH}")

    train_profiles = build_profiles(TRAIN_TOPICS_PATH, split="train")
    test_profiles = build_profiles(TEST_TOPICS_PATH, split="test")

    train_views = build_query_views(train_profiles)
    test_views = build_query_views(test_profiles)

    train_profiles.to_parquet(TRAIN_PROFILE_OUT, index=False, compression="zstd")
    test_profiles.to_parquet(TEST_PROFILE_OUT, index=False, compression="zstd")

    train_views.to_parquet(TRAIN_VIEWS_OUT, index=False, compression="zstd")
    test_views.to_parquet(TEST_VIEWS_OUT, index=False, compression="zstd")

    qrels = pd.read_parquet(QRELS_PATH)
    qrels["topic_id"] = qrels["topic_id"].astype(str).str.strip()

    test_topic_ids = set(test_profiles["topic_id"].astype(str).str.strip())
    qrels = qrels[qrels["topic_id"].isin(test_topic_ids)].copy()

    summary = {
        "num_train_profiles": int(len(train_profiles)),
        "num_test_profiles": int(len(test_profiles)),
        "num_train_views": int(len(train_views)),
        "num_test_views": int(len(test_views)),
        "num_test_topics_with_qrels": int(qrels["topic_id"].nunique()) if len(qrels) else 0,
        "train_profile_out": str(TRAIN_PROFILE_OUT),
        "test_profile_out": str(TEST_PROFILE_OUT),
        "train_views_out": str(TRAIN_VIEWS_OUT),
        "test_views_out": str(TEST_VIEWS_OUT),
        "active_retrieval_views": [
            "title_abstract_full",
            "claims_short",
            "combined_short",
            "embedding_text",
            "metadata",
        ],
                "target_benchmark_query_count": TARGET_BENCHMARK_QUERY_COUNT,
    }

    with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("DONE")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nTrain profiles sample:")
    print(train_profiles[[
        "topic_id",
        "query_strength",
        "prior_art_cutoff_date",
        "main_ipc",
        "num_assignees",
        "query_combined_short_word_count",
    ]].head())

    print("\nTest profiles sample:")
    print(test_profiles[[
        "topic_id",
        "query_strength",
        "prior_art_cutoff_date",
        "main_ipc",
        "num_assignees",
        "query_combined_short_word_count",
    ]].head())

    print("\nTest views distribution:")
    if len(test_views):
        print(test_views["query_view"].value_counts())


if __name__ == "__main__":
    main()



BASE_DIR          = /kaggle/working/clefip2011_pac_topics
TRAIN_TOPICS_PATH = /kaggle/working/clefip2011_pac_topics/processed/pac_training_topics_clean.parquet
TEST_TOPICS_PATH  = /kaggle/working/clefip2011_pac_topics/processed/pac_test_topics_benchmark_200.parquet
QRELS_PATH        = /kaggle/working/clefip2011_pac_topics/processed/pac_test_qrels_benchmark_200.parquet
OUT_DIR           = /kaggle/working/clefip2011_pac_topics/query_understanding
DONE
{
  "num_train_profiles": 300,
  "num_test_profiles": 200,
  "num_train_views": 3000,
  "num_test_views": 2000,
  "num_test_topics_with_qrels": 200,
  "train_profile_out": "/kaggle/working/clefip2011_pac_topics/query_understanding/pac_train_query_profiles.parquet",
  "test_profile_out": "/kaggle/working/clefip2011_pac_topics/query_understanding/pac_test_query_profiles.parquet",
  "train_views_out": "/kaggle/working/clefip2011_pac_topics/query_understanding/pac_train_query_views.parquet",
  "test_views_out": "/kaggle/working/clefip2011_pac_t

In [10]:
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path(os.getenv("PAC_TOPIC_WORK_DIR", "/kaggle/working/clefip2011_pac_topics"))
QUERY_DIR = BASE_DIR / "query_understanding"
OUT_DIR = BASE_DIR / "metadata_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PROFILE_PATH = QUERY_DIR / "pac_train_query_profiles.parquet"
TEST_PROFILE_PATH = QUERY_DIR / "pac_test_query_profiles.parquet"

TRAIN_META_OUT = OUT_DIR / "pac_train_metadata_features.parquet"
TEST_META_OUT = OUT_DIR / "pac_test_metadata_features.parquet"
SUMMARY_OUT = OUT_DIR / "summary.json"

print("TRAIN_PROFILE_PATH =", TRAIN_PROFILE_PATH)
print("TEST_PROFILE_PATH  =", TEST_PROFILE_PATH)
print("OUT_DIR            =", OUT_DIR)


# =========================================================
# BASIC HELPERS
# =========================================================

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_list(x: Any) -> List[str]:
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, list):
        out = []
        seen = set()
        for item in x:
            s = normalize_text(item)
            if s and s not in seen:
                seen.add(s)
                out.append(s)
        return out

    s = normalize_text(x)
    return [s] if s else []


def normalize_date_yyyymmdd(x: Any) -> str:
    s = normalize_text(x)
    if not s:
        return ""

    if re.fullmatch(r"\d{8}", s):
        return s

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return s.replace("-", "")

    m = re.search(r"\b(\d{8})\b", s)
    if m:
        return m.group(1)

    return ""


def parse_date_int(x: Any) -> int:
    s = normalize_date_yyyymmdd(x)
    if not s:
        return 0
    try:
        return int(s)
    except Exception:
        return 0


def clean_party_name(name: Any) -> str:
    s = normalize_text(name).lower()

    # Bỏ vài hậu tố pháp nhân phổ biến để assignee matching đỡ nhiễu.
    s = re.sub(r"\b(ltd|limited|inc|corp|corporation|gmbh|ag|sa|nv|llc|co)\b\.?", " ", s)
    s = re.sub(r"[^a-z0-9äöüßéèàçñ\s\-&]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


# =========================================================
# IPC PARSING
# =========================================================

# Bắt IPC dạng:
# B41J 2/05
# A61K 31/715
# G06F 17/30
IPC_PATTERN = re.compile(
    r"\b([A-HY])\s*([0-9]{2})\s*([A-Z])\s*([0-9]{1,4})\s*/\s*([0-9]{1,6})\b"
)


def format_ipc_group(section: str, cls_num: str, subclass_letter: str, main_group: str, subgroup: str) -> Dict[str, str]:
    """
    Giữ định dạng IPC chuẩn:
    - class: A61
    - subclass: A61K
    - group: A61K 6/02, A23G 3/00, A61K 31/715

    Quan trọng:
    - main_group bỏ zero đầu: 0006 -> 6
    - subgroup giữ/pad tối thiểu 2 chữ số: 5 -> 05, 0 -> 00, 715 -> 715
    """
    section = section.upper()
    cls_num = cls_num.zfill(2)
    subclass_letter = subclass_letter.upper()

    try:
        main_group_norm = str(int(main_group))
    except Exception:
        main_group_norm = main_group.lstrip("0") or "0"

    subgroup = str(subgroup)
    if subgroup.isdigit():
        subgroup_norm = subgroup.zfill(2)
    else:
        subgroup_norm = subgroup

    ipc_class = f"{section}{cls_num}"
    ipc_subclass = f"{section}{cls_num}{subclass_letter}"
    ipc_group = f"{ipc_subclass} {main_group_norm}/{subgroup_norm}"
    ipc_main_group = f"{ipc_subclass} {main_group_norm}"

    return {
        "ipc_normalized": ipc_group,
        "ipc_section": section,
        "ipc_class": ipc_class,
        "ipc_subclass": ipc_subclass,
        "ipc_group": ipc_group,
        "ipc_main_group": ipc_main_group,
    }


def extract_ipc_candidates(text: str) -> List[str]:
    """
    Lấy IPC code chuẩn từ chuỗi nhiễu.
    Ví dụ:
      B41J 2/05 20060101A...
    → B41J 2/05

    Sửa quan trọng:
    - Không dùng int(subgroup) làm mất zero.
    """
    text = normalize_text(text).upper()
    out = []

    for m in IPC_PATTERN.finditer(text):
        parsed = format_ipc_group(
            section=m.group(1),
            cls_num=m.group(2),
            subclass_letter=m.group(3),
            main_group=m.group(4),
            subgroup=m.group(5),
        )
        out.append(parsed["ipc_normalized"])

    # fallback cho dạng tương đối chuẩn
    if not out:
        fallback = re.findall(r"\b([A-HY][0-9]{2}[A-Z]\s+[0-9]{1,4}/[0-9]{1,6})\b", text)
        for code in fallback:
            parsed = parse_ipc_code(code)
            if parsed["ipc_normalized"]:
                out.append(parsed["ipc_normalized"])

    seen = set()
    uniq = []
    for code in out:
        code = normalize_text(code.upper())
        if code and code not in seen:
            seen.add(code)
            uniq.append(code)

    return uniq


def parse_ipc_code(code: str) -> Dict[str, str]:
    code = normalize_text(code).upper()

    m = IPC_PATTERN.search(code)

    if not m:
        candidates = extract_ipc_candidates(code)
        if candidates:
            code = candidates[0]
            m = IPC_PATTERN.search(code)

    if not m:
        return {
            "ipc_normalized": code,
            "ipc_section": "",
            "ipc_class": "",
            "ipc_subclass": "",
            "ipc_group": "",
            "ipc_main_group": "",
        }

    return format_ipc_group(
        section=m.group(1),
        cls_num=m.group(2),
        subclass_letter=m.group(3),
        main_group=m.group(4),
        subgroup=m.group(5),
    )


def normalize_ipc_list(ipc_codes: Any) -> List[str]:
    raw_list = normalize_list(ipc_codes)
    found = []

    for item in raw_list:
        candidates = extract_ipc_candidates(item)

        if candidates:
            found.extend(candidates)
        else:
            parsed = parse_ipc_code(item)
            if parsed["ipc_normalized"]:
                found.append(parsed["ipc_normalized"])

    seen = set()
    out = []
    for code in found:
        code = normalize_text(code.upper())
        if code and code not in seen:
            seen.add(code)
            out.append(code)

    return out


# =========================================================
# FEATURE BUILDING
# =========================================================

def build_metadata_row(row: Dict[str, Any]) -> Dict[str, Any]:
    topic_id = normalize_text(row.get("topic_id"))
    query_doc_id = normalize_text(row.get("query_doc_id")) or topic_id

    raw_ipc_codes = normalize_list(row.get("ipc_codes"))
    ipc_codes_norm = normalize_ipc_list(raw_ipc_codes)

    main_ipc_norm = ipc_codes_norm[0] if ipc_codes_norm else ""
    main_ipc_parts = parse_ipc_code(main_ipc_norm) if main_ipc_norm else {
        "ipc_normalized": "",
        "ipc_section": "",
        "ipc_class": "",
        "ipc_subclass": "",
        "ipc_group": "",
        "ipc_main_group": "",
    }

    assignees_raw = normalize_list(row.get("assignees"))
    inventors_raw = normalize_list(row.get("inventors"))
    citations_raw = normalize_list(row.get("citations"))

    assignees_clean = []
    seen_assignee = set()
    for a in assignees_raw:
        c = clean_party_name(a)
        if c and c not in seen_assignee:
            seen_assignee.add(c)
            assignees_clean.append(c)

    inventors_clean = []
    seen_inventor = set()
    for inv in inventors_raw:
        c = clean_party_name(inv)
        if c and c not in seen_inventor:
            seen_inventor.add(c)
            inventors_clean.append(c)

    publication_date = normalize_date_yyyymmdd(row.get("publication_date"))
    application_date = normalize_date_yyyymmdd(row.get("application_date"))
    priority_date = normalize_date_yyyymmdd(row.get("priority_date"))
    prior_art_cutoff_date = normalize_date_yyyymmdd(row.get("prior_art_cutoff_date"))

    if not prior_art_cutoff_date:
        prior_art_cutoff_date = priority_date or application_date or publication_date

    title_wc = int(row.get("title_word_count", 0) or 0)
    abstract_wc = int(row.get("abstract_word_count", 0) or 0)
    claims_wc = int(row.get("claims_word_count", 0) or 0)
    description_wc = int(row.get("description_word_count", 0) or 0)

    return {
        "split": normalize_text(row.get("split")),
        "topic_id": topic_id,
        "query_doc_id": query_doc_id,
        "input_type": normalize_text(row.get("input_type")) or "patent_document",

        "lang": normalize_text(row.get("lang")).lower(),
        "country": normalize_text(row.get("country")).lower(),
        "kind": normalize_text(row.get("kind")).lower(),

        "publication_date": publication_date,
        "application_date": application_date,
        "priority_date": priority_date,
        "prior_art_cutoff_date": prior_art_cutoff_date,
        "prior_art_cutoff_date_int": parse_date_int(prior_art_cutoff_date),

        "publication_doc_number": normalize_text(row.get("publication_doc_number")),
        "application_doc_number": normalize_text(row.get("application_doc_number")),

        # IPC normalized
        "ipc_codes_raw": raw_ipc_codes,
        "ipc_codes_norm": ipc_codes_norm,
        "main_ipc_norm": main_ipc_norm,
        "ipc_section": main_ipc_parts["ipc_section"],
        "ipc_class": main_ipc_parts["ipc_class"],
        "ipc_subclass": main_ipc_parts["ipc_subclass"],
        "ipc_group": main_ipc_parts["ipc_group"],
        "ipc_main_group": main_ipc_parts["ipc_main_group"],

        # Parties
        "assignees_raw": assignees_raw,
        "assignees_clean": assignees_clean,
        "inventors_raw": inventors_raw,
        "inventors_clean": inventors_clean,

        # Citations in query patent
        "citations_raw": citations_raw,

        # Boolean flags for planner
        "has_ipc": bool(ipc_codes_norm),
        "has_ipc_subclass": bool(main_ipc_parts["ipc_subclass"]),
        "has_assignee": bool(assignees_clean),
        "has_inventor": bool(inventors_clean),
        "has_citations": bool(citations_raw),
        "has_date_filter": bool(prior_art_cutoff_date),

        # Query/text statistics for planner
        "query_strength": normalize_text(row.get("query_strength")),
        "title_word_count": title_wc,
        "abstract_word_count": abstract_wc,
        "claims_word_count": claims_wc,
        "description_word_count": description_wc,

        "is_claims_long": claims_wc >= 800,
        "is_description_long": description_wc >= 3000,
        "is_text_sparse": (title_wc + abstract_wc + claims_wc) < 100,
    }


def build_metadata_features(profile_path: Path) -> pd.DataFrame:
    df = pd.read_parquet(profile_path)

    records = []
    for row in df.to_dict(orient="records"):
        records.append(build_metadata_row(row))

    return pd.DataFrame(records)


# =========================================================
# MAIN
# =========================================================

def main():
    if not TRAIN_PROFILE_PATH.exists():
        raise FileNotFoundError(f"Missing train profile: {TRAIN_PROFILE_PATH}")
    if not TEST_PROFILE_PATH.exists():
        raise FileNotFoundError(f"Missing test profile: {TEST_PROFILE_PATH}")

    train_meta = build_metadata_features(TRAIN_PROFILE_PATH)
    test_meta = build_metadata_features(TEST_PROFILE_PATH)

    train_meta.to_parquet(TRAIN_META_OUT, index=False, compression="zstd")
    test_meta.to_parquet(TEST_META_OUT, index=False, compression="zstd")

    summary = {
        "num_train": int(len(train_meta)),
        "num_test": int(len(test_meta)),

        "train_meta_out": str(TRAIN_META_OUT),
        "test_meta_out": str(TEST_META_OUT),

        "train_has_ipc_rate": float(train_meta["has_ipc"].mean()) if len(train_meta) else 0,
        "test_has_ipc_rate": float(test_meta["has_ipc"].mean()) if len(test_meta) else 0,

        "train_has_date_filter_rate": float(train_meta["has_date_filter"].mean()) if len(train_meta) else 0,
        "test_has_date_filter_rate": float(test_meta["has_date_filter"].mean()) if len(test_meta) else 0,

        "train_has_assignee_rate": float(train_meta["has_assignee"].mean()) if len(train_meta) else 0,
        "test_has_assignee_rate": float(test_meta["has_assignee"].mean()) if len(test_meta) else 0,

        "ipc_parser": "regex_extract_ipc_section_class_subclass_group_keep_leading_zero",
    }

    with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("DONE")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nTrain metadata sample:")
    print(train_meta[[
        "topic_id",
        "main_ipc_norm",
        "ipc_section",
        "ipc_class",
        "ipc_subclass",
        "ipc_group",
        "prior_art_cutoff_date",
        "has_assignee",
        "query_strength"
    ]].head())

    print("\nTest metadata sample:")
    print(test_meta[[
        "topic_id",
        "main_ipc_norm",
        "ipc_section",
        "ipc_class",
        "ipc_subclass",
        "ipc_group",
        "prior_art_cutoff_date",
        "has_assignee",
        "query_strength"
    ]].head())

    print("\nTop test IPC subclasses:")
    print(test_meta["ipc_subclass"].value_counts().head(20))


if __name__ == "__main__":
    main()

TRAIN_PROFILE_PATH = /kaggle/working/clefip2011_pac_topics/query_understanding/pac_train_query_profiles.parquet
TEST_PROFILE_PATH  = /kaggle/working/clefip2011_pac_topics/query_understanding/pac_test_query_profiles.parquet
OUT_DIR            = /kaggle/working/clefip2011_pac_topics/metadata_features
DONE
{
  "num_train": 300,
  "num_test": 200,
  "train_meta_out": "/kaggle/working/clefip2011_pac_topics/metadata_features/pac_train_metadata_features.parquet",
  "test_meta_out": "/kaggle/working/clefip2011_pac_topics/metadata_features/pac_test_metadata_features.parquet",
  "train_has_ipc_rate": 1.0,
  "test_has_ipc_rate": 1.0,
  "train_has_date_filter_rate": 1.0,
  "test_has_date_filter_rate": 1.0,
  "train_has_assignee_rate": 1.0,
  "test_has_assignee_rate": 1.0,
  "ipc_parser": "regex_extract_ipc_section_class_subclass_group_keep_leading_zero"
}

Train metadata sample:
        topic_id main_ipc_norm ipc_section ipc_class ipc_subclass   ipc_group  \
0  EP-1222910-A2     A61K 6/02         

In [11]:
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path(os.getenv("PAC_TOPIC_WORK_DIR", "/kaggle/working/clefip2011_pac_topics"))

QUERY_DIR = BASE_DIR / "query_understanding"
META_DIR = BASE_DIR / "metadata_features"
OUT_DIR = BASE_DIR / "retrieval_plans"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_VIEWS_PATH = QUERY_DIR / "pac_train_query_views.parquet"
TEST_VIEWS_PATH = QUERY_DIR / "pac_test_query_views.parquet"

TRAIN_META_PATH = META_DIR / "pac_train_metadata_features.parquet"
TEST_META_PATH = META_DIR / "pac_test_metadata_features.parquet"

TRAIN_PLAN_OUT = OUT_DIR / "pac_train_retrieval_plans.parquet"
TEST_PLAN_OUT = OUT_DIR / "pac_test_retrieval_plans.parquet"

TRAIN_PLAN_VIEWS_OUT = OUT_DIR / "pac_train_plan_views.parquet"
TEST_PLAN_VIEWS_OUT = OUT_DIR / "pac_test_plan_views.parquet"

SUMMARY_OUT = OUT_DIR / "summary.json"

# Retrieval defaults. Đây chỉ là plan, chưa search thật.
DEFAULT_RRF_K = int(os.getenv("DEFAULT_RRF_K", "60"))
DEFAULT_FINAL_TOP_K = int(os.getenv("DEFAULT_FINAL_TOP_K", "100"))
DEFAULT_VIEW_TOP_K = int(os.getenv("DEFAULT_VIEW_TOP_K", "100"))
DEFAULT_CANDIDATE_POOL_SIZE = int(os.getenv("DEFAULT_CANDIDATE_POOL_SIZE", "500"))

# Per-view retrieval size. Có thể tune sau bằng train set.
STRONG_BM25_TOP_K = int(os.getenv("STRONG_BM25_TOP_K", str(DEFAULT_VIEW_TOP_K)))
STRONG_VECTOR_TOP_K = int(os.getenv("STRONG_VECTOR_TOP_K", str(DEFAULT_VIEW_TOP_K)))

MEDIUM_BM25_TOP_K = int(os.getenv("MEDIUM_BM25_TOP_K", str(DEFAULT_VIEW_TOP_K)))
MEDIUM_VECTOR_TOP_K = int(os.getenv("MEDIUM_VECTOR_TOP_K", str(DEFAULT_VIEW_TOP_K)))

WEAK_BM25_TOP_K = int(os.getenv("WEAK_BM25_TOP_K", str(DEFAULT_VIEW_TOP_K)))
WEAK_VECTOR_TOP_K = int(os.getenv("WEAK_VECTOR_TOP_K", str(DEFAULT_VIEW_TOP_K)))

# Description thường nhiễu hơn title/abstract/claims, nên lấy ít hơn.
DESCRIPTION_BM25_TOP_K_FACTOR = float(os.getenv("DESCRIPTION_BM25_TOP_K_FACTOR", "0.50"))
DESCRIPTION_VECTOR_TOP_K_FACTOR = float(os.getenv("DESCRIPTION_VECTOR_TOP_K_FACTOR", "0.75"))

# Metadata không dùng như text search chính, chỉ dùng filter/boost.
USE_METADATA_AS_TEXT_SEARCH = os.getenv("USE_METADATA_AS_TEXT_SEARCH", "false").lower() == "true"

print("BASE_DIR         =", BASE_DIR)
print("TRAIN_VIEWS_PATH =", TRAIN_VIEWS_PATH)
print("TEST_VIEWS_PATH  =", TEST_VIEWS_PATH)
print("TRAIN_META_PATH  =", TRAIN_META_PATH)
print("TEST_META_PATH   =", TEST_META_PATH)
print("OUT_DIR          =", OUT_DIR)


# =========================================================
# HELPERS
# =========================================================

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def canonical_doc_id(value: Any) -> str:
    text = normalize_text(value).upper()
    if not text:
        return ""
    text = text.replace(" ", "-").replace("_", "-")
    text = re.sub(r"-+", "-", text).strip("-")
    match = re.match(r"^([A-Z]{2})-?(\d+)(?:-[A-Z][0-9A-Z]*)?$", text)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return text


def normalize_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    if isinstance(x, float) and pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s in {"true", "1", "yes", "y"}


def normalize_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        out = []
        seen = set()
        for item in x:
            s = normalize_text(item)
            if s and s not in seen:
                seen.add(s)
                out.append(s)
        return out
    s = normalize_text(x)
    return [s] if s else []


def to_json(x: Any) -> str:
    return json.dumps(x, ensure_ascii=False)


def safe_int(x: Any, default: int = 0) -> int:
    try:
        if x is None:
            return default
        if isinstance(x, float) and pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        if x is None:
            return default
        if isinstance(x, float) and pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


# =========================================================
# STRATEGY RULES
# =========================================================

def choose_strategy(meta: Dict[str, Any]) -> str:
    """
    Planner rule-based.
    Chưa dùng LLM, vì đây là benchmark patent document.
    """
    query_strength = normalize_text(meta.get("query_strength")).lower()

    has_ipc = normalize_bool(meta.get("has_ipc"))
    has_date_filter = normalize_bool(meta.get("has_date_filter"))
    has_assignee = normalize_bool(meta.get("has_assignee"))

    is_text_sparse = normalize_bool(meta.get("is_text_sparse"))
    is_claims_long = normalize_bool(meta.get("is_claims_long"))
    is_description_long = normalize_bool(meta.get("is_description_long"))

    if is_text_sparse:
        return "sparse_metadata_recall_heavy"

    if query_strength == "strong" and has_ipc and has_date_filter:
        return "standard_hybrid_ipc_date"

    if query_strength == "strong":
        return "standard_hybrid"

    if query_strength == "medium" and has_ipc:
        return "balanced_hybrid_ipc"

    if query_strength == "medium":
        return "balanced_hybrid"

    if has_assignee or has_ipc:
        return "weak_metadata_assisted"

    return "weak_recall_heavy"


def strategy_topk(strategy: str) -> Dict[str, int]:
    if strategy in {"standard_hybrid_ipc_date", "standard_hybrid"}:
        return {
            "bm25_top_k": STRONG_BM25_TOP_K,
            "vector_top_k": STRONG_VECTOR_TOP_K,
            "candidate_pool_size": DEFAULT_CANDIDATE_POOL_SIZE,
            "final_top_k": DEFAULT_FINAL_TOP_K,
        }

    if strategy in {"balanced_hybrid_ipc", "balanced_hybrid"}:
        return {
            "bm25_top_k": MEDIUM_BM25_TOP_K,
            "vector_top_k": MEDIUM_VECTOR_TOP_K,
            "candidate_pool_size": max(DEFAULT_CANDIDATE_POOL_SIZE, 750),
            "final_top_k": DEFAULT_FINAL_TOP_K,
        }

    return {
        "bm25_top_k": WEAK_BM25_TOP_K,
        "vector_top_k": WEAK_VECTOR_TOP_K,
        "candidate_pool_size": max(DEFAULT_CANDIDATE_POOL_SIZE, 1000),
        "final_top_k": DEFAULT_FINAL_TOP_K,
    }


def build_filter_config(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Filter config cho bước retrieval thật.

    Lưu ý:
    - Date filter nên dùng cẩn thận. Prior art thường cần trước priority/application date.
    - Lúc retrieval thật có thể dùng publication_date < cutoff.
    - Nếu corpus thiếu date ở nhiều docs, ta có thể dùng soft filter/boost thay vì hard filter.
    """
    topic_id = normalize_text(meta.get("topic_id"))
    query_doc_id = normalize_text(meta.get("query_doc_id")) or topic_id

    prior_art_cutoff_date = normalize_text(meta.get("prior_art_cutoff_date"))
    prior_art_cutoff_date_int = safe_int(meta.get("prior_art_cutoff_date_int"), 0)

    has_date_filter = normalize_bool(meta.get("has_date_filter"))

    return {
        "exclude_doc_id": query_doc_id,
        "exclude_canonical_doc_id": canonical_doc_id(query_doc_id),
        "use_date_filter": bool(has_date_filter and prior_art_cutoff_date),
        "date_filter_field": "publication_date",
        "date_filter_operator": "lt",
        "prior_art_cutoff_date": prior_art_cutoff_date,
        "prior_art_cutoff_date_int": prior_art_cutoff_date_int,

        # Để retrieval implementation sau này chọn:
        # strict: loại bỏ doc sau cutoff
        # soft: vẫn lấy nhưng trừ điểm
        "date_filter_mode": "strict_if_field_available",
    }


def build_boost_config(meta: Dict[str, Any]) -> Dict[str, Any]:
    ipc_codes_norm = normalize_list(meta.get("ipc_codes_norm"))
    assignees_clean = normalize_list(meta.get("assignees_clean"))
    inventors_clean = normalize_list(meta.get("inventors_clean"))
    assignees_raw = normalize_list(meta.get("assignees_raw"))
    inventors_raw = normalize_list(meta.get("inventors_raw"))
    citations_raw = normalize_list(meta.get("citations_raw"))

    ipc_section = normalize_text(meta.get("ipc_section"))
    ipc_class = normalize_text(meta.get("ipc_class"))
    ipc_subclass = normalize_text(meta.get("ipc_subclass"))
    ipc_group = normalize_text(meta.get("ipc_group"))
    ipc_main_group = normalize_text(meta.get("ipc_main_group"))

    return {
        "use_ipc_boost": bool(ipc_subclass),
        "ipc_boost_fields": {
            "ipc_section": ipc_section,
            "ipc_class": ipc_class,
            "ipc_subclass": ipc_subclass,
            "ipc_main_group": ipc_main_group,
            "ipc_group": ipc_group,
            "ipc_codes_norm": ipc_codes_norm[:20],
        },
        "ipc_boost_weights": {
            "same_ipc_group": 1.35,
            "same_ipc_main_group": 1.25,
            "same_ipc_subclass": 1.18,
            "same_ipc_class": 1.10,
            "same_ipc_section": 1.04,
        },

        "use_assignee_boost": bool(assignees_clean),
        "assignees_clean": assignees_clean[:20],
        "assignees_raw": assignees_raw[:20],
        "assignee_boost_weight": 1.08,

        "use_inventor_boost": bool(inventors_clean),
        "inventors_clean": inventors_clean[:20],
        "inventors_raw": inventors_raw[:20],
        "inventor_boost_weight": 1.04,

        "use_query_citation_seeds": bool(citations_raw),
        "query_citation_seeds": citations_raw[:100],
        "citation_seed_boost_weight": 1.20,
    }


def build_graph_plan(meta: Dict[str, Any], strategy: str) -> Dict[str, Any]:
    """
    Chưa chạy graph retrieval ở bước 4.
    Chỉ quyết định khi nào graph expansion nên được gọi ở bước sau.
    """
    has_citations = normalize_bool(meta.get("has_citations"))
    has_ipc = normalize_bool(meta.get("has_ipc"))
    is_text_sparse = normalize_bool(meta.get("is_text_sparse"))

    enable_graph_expansion = bool(has_citations or is_text_sparse or strategy in {
        "weak_metadata_assisted",
        "weak_recall_heavy",
        "sparse_metadata_recall_heavy",
    })

    return {
        "enable_graph_expansion": enable_graph_expansion,
        "graph_expansion_reason": (
            "citations_or_sparse_or_weak_query"
            if enable_graph_expansion else
            "hybrid_first"
        ),
        "graph_seed_sources": [
            "top_hybrid_results",
            "query_citations" if has_citations else "",
            "ipc_neighbors" if has_ipc else "",
        ],
        "graph_max_seed_docs": 100,
        "graph_max_expanded_docs": 1000,
    }


def select_views_for_plan(topic_views: pd.DataFrame, meta: Dict[str, Any], strategy: str) -> pd.DataFrame:
    """
    Chọn query views cho retrieval plan.

    Mặc định dùng các active views từ bước 2:
    - title_abstract_full
    - claims_short
    - combined_short
    - embedding_text
    - metadata

    Nhưng metadata không dùng làm search text chính, chỉ dùng filter/boost.
    """
    df = topic_views.copy()

    if "retrieval_active" in df.columns:
        df = df[df["retrieval_active"].apply(normalize_bool)].copy()

    allowed = {
        "title_abstract_full",
        "claims_short",
        "combined_short",
        "embedding_text",
        "metadata",
    }

    df = df[df["query_view"].isin(allowed)].copy()

    is_text_sparse = normalize_bool(meta.get("is_text_sparse"))
    query_strength = normalize_text(meta.get("query_strength")).lower()

    # Dense retrieval uses embedding_text only, aligned with document embeddings.
    if query_strength == "strong" and not is_text_sparse:
        pass

    return df


def build_view_plan_rows(
    topic_views: pd.DataFrame,
    meta: Dict[str, Any],
    topic_plan: Dict[str, Any],
) -> List[Dict[str, Any]]:
    rows = []

    base_bm25_top_k = topic_plan["bm25_top_k"]
    base_vector_top_k = topic_plan["vector_top_k"]

    strategy = topic_plan["strategy_name"]

    for view in topic_views.to_dict(orient="records"):
        query_view = normalize_text(view.get("query_view"))
        retrieval_use = normalize_text(view.get("retrieval_use"))
        query_text = normalize_text(view.get("query_text"))

        if not query_text:
            continue

        base_weight = safe_float(view.get("weight"), 1.0)

        use_bm25 = False
        use_vector = False
        bm25_top_k = 0
        vector_top_k = 0

        if query_view == "metadata":
            use_bm25 = bool(USE_METADATA_AS_TEXT_SEARCH)
            use_vector = False
            bm25_top_k = int(base_bm25_top_k * 0.25) if use_bm25 else 0
            vector_top_k = 0

        elif query_view == "description_short":
            # Description có thể nhiễu, nhưng hữu ích cho recall.
            use_bm25 = False
            use_vector = True
            bm25_top_k = 0
            vector_top_k = max(50, int(base_vector_top_k * DESCRIPTION_VECTOR_TOP_K_FACTOR))

        else:
            use_bm25 = retrieval_use in {"bm25_and_vector", "bm25_only"}
            use_vector = retrieval_use in {"bm25_and_vector", "vector_only", "vector_optional"}

            bm25_top_k = base_bm25_top_k if use_bm25 else 0
            vector_top_k = base_vector_top_k if use_vector else 0

        # Tinh chỉnh weight
        weight = base_weight

        if query_view == "claims_short" and normalize_bool(meta.get("is_claims_long")):
            weight *= 1.05

        if query_view == "description_short" and normalize_bool(meta.get("is_description_long")):
            weight *= 1.05

        if query_view == "title_abstract_full":
            weight *= 1.05

        rows.append({
            "split": normalize_text(meta.get("split")),
            "topic_id": normalize_text(meta.get("topic_id")),
            "query_doc_id": normalize_text(meta.get("query_doc_id")),

            "strategy_name": strategy,
            "query_view": query_view,
            "query_text": query_text,
            "query_word_count": safe_int(view.get("query_word_count"), 0),

            "retrieval_use": retrieval_use,
            "use_bm25": bool(use_bm25),
            "use_vector": bool(use_vector),

            "bm25_top_k": int(bm25_top_k),
            "vector_top_k": int(vector_top_k),

            "view_weight": float(round(weight, 4)),
            "rrf_k": int(topic_plan["rrf_k"]),

            "filter_config_json": topic_plan["filter_config_json"],
            "boost_config_json": topic_plan["boost_config_json"],
        })

    return rows


def build_topic_plan(meta: Dict[str, Any], topic_views: pd.DataFrame) -> Dict[str, Any]:
    strategy = choose_strategy(meta)
    topk = strategy_topk(strategy)

    filter_config = build_filter_config(meta)
    boost_config = build_boost_config(meta)
    graph_plan = build_graph_plan(meta, strategy)

    selected_views = select_views_for_plan(topic_views, meta, strategy)

    active_query_views = selected_views["query_view"].tolist() if len(selected_views) else []

    if len(selected_views) and "retrieval_use" in selected_views.columns:
        text_search_views = selected_views[
            selected_views["retrieval_use"].isin(["bm25_and_vector", "bm25_only"])
        ]["query_view"].tolist()
    else:
        text_search_views = []

    topic_id = normalize_text(meta.get("topic_id"))
    query_doc_id = normalize_text(meta.get("query_doc_id")) or topic_id

    return {
        "split": normalize_text(meta.get("split")),
        "topic_id": topic_id,
        "query_doc_id": query_doc_id,
        "input_type": normalize_text(meta.get("input_type")) or "patent_document",

        "strategy_name": strategy,
        "query_strength": normalize_text(meta.get("query_strength")),

        "active_query_views": active_query_views,
        "active_query_views_json": to_json(active_query_views),
        "text_search_views": text_search_views,
        "text_search_views_json": to_json(text_search_views),

        "use_bm25": True,
        "use_vector": True,
        "use_metadata_boost": True,
        "use_graph_expansion_later": bool(graph_plan["enable_graph_expansion"]),

        "bm25_top_k": int(topk["bm25_top_k"]),
        "vector_top_k": int(topk["vector_top_k"]),
        "candidate_pool_size": int(topk["candidate_pool_size"]),
        "final_top_k": int(topk["final_top_k"]),
        "rrf_k": int(DEFAULT_RRF_K),

        "prior_art_cutoff_date": normalize_text(meta.get("prior_art_cutoff_date")),
        "prior_art_cutoff_date_int": safe_int(meta.get("prior_art_cutoff_date_int"), 0),
        "has_date_filter": normalize_bool(meta.get("has_date_filter")),
        "has_ipc": normalize_bool(meta.get("has_ipc")),
        "has_assignee": normalize_bool(meta.get("has_assignee")),
        "has_citations": normalize_bool(meta.get("has_citations")),

        "ipc_section": normalize_text(meta.get("ipc_section")),
        "ipc_class": normalize_text(meta.get("ipc_class")),
        "ipc_subclass": normalize_text(meta.get("ipc_subclass")),
        "ipc_group": normalize_text(meta.get("ipc_group")),
        "ipc_main_group": normalize_text(meta.get("ipc_main_group")),

        "is_claims_long": normalize_bool(meta.get("is_claims_long")),
        "is_description_long": normalize_bool(meta.get("is_description_long")),
        "is_text_sparse": normalize_bool(meta.get("is_text_sparse")),

        "filter_config_json": to_json(filter_config),
        "boost_config_json": to_json(boost_config),
        "graph_plan_json": to_json(graph_plan),
    }


# =========================================================
# PLAN BUILDING
# =========================================================

def build_retrieval_plans(views_path: Path, meta_path: Path) -> Dict[str, pd.DataFrame]:
    views = pd.read_parquet(views_path)
    meta = pd.read_parquet(meta_path)

    if "topic_id" not in views.columns:
        raise ValueError(f"Missing topic_id in views: {views_path}")
    if "topic_id" not in meta.columns:
        raise ValueError(f"Missing topic_id in metadata: {meta_path}")

    topic_to_views = {
        topic_id: g.copy()
        for topic_id, g in views.groupby("topic_id")
    }

    plan_rows = []
    plan_view_rows = []

    for row in meta.to_dict(orient="records"):
        topic_id = normalize_text(row.get("topic_id"))
        topic_views = topic_to_views.get(topic_id, pd.DataFrame())

        topic_plan = build_topic_plan(row, topic_views)
        plan_rows.append(topic_plan)

        selected_views = select_views_for_plan(topic_views, row, topic_plan["strategy_name"])
        view_rows = build_view_plan_rows(selected_views, row, topic_plan)
        plan_view_rows.extend(view_rows)

    plans = pd.DataFrame(plan_rows)
    plan_views = pd.DataFrame(plan_view_rows)

    return {
        "plans": plans,
        "plan_views": plan_views,
    }


# =========================================================
# SUMMARY
# =========================================================

def make_summary(train_plans: pd.DataFrame, test_plans: pd.DataFrame,
                 train_plan_views: pd.DataFrame, test_plan_views: pd.DataFrame) -> Dict[str, Any]:
    def strategy_counts(df: pd.DataFrame) -> Dict[str, int]:
        if len(df) == 0:
            return {}
        return {str(k): int(v) for k, v in df["strategy_name"].value_counts().to_dict().items()}

    def view_counts(df: pd.DataFrame) -> Dict[str, int]:
        if len(df) == 0:
            return {}
        return {str(k): int(v) for k, v in df["query_view"].value_counts().to_dict().items()}

    return {
        "num_train_plans": int(len(train_plans)),
        "num_test_plans": int(len(test_plans)),
        "num_train_plan_views": int(len(train_plan_views)),
        "num_test_plan_views": int(len(test_plan_views)),

        "train_plan_out": str(TRAIN_PLAN_OUT),
        "test_plan_out": str(TEST_PLAN_OUT),
        "train_plan_views_out": str(TRAIN_PLAN_VIEWS_OUT),
        "test_plan_views_out": str(TEST_PLAN_VIEWS_OUT),

        "default_rrf_k": DEFAULT_RRF_K,
        "default_final_top_k": DEFAULT_FINAL_TOP_K,
        "default_view_top_k": DEFAULT_VIEW_TOP_K,
        "default_candidate_pool_size": DEFAULT_CANDIDATE_POOL_SIZE,

        "strategy_counts_train": strategy_counts(train_plans),
        "strategy_counts_test": strategy_counts(test_plans),

        "query_view_counts_train": view_counts(train_plan_views),
        "query_view_counts_test": view_counts(test_plan_views),

        "planner_type": "rule_based_patent_document_planner",
        "uses_natural_language_or_idea_input": False,

        "next_step": "fixed_retrieval_experiments_bm25_knn_hybrid",
    }


# =========================================================
# MAIN
# =========================================================

def main():
    required = [
        TRAIN_VIEWS_PATH,
        TEST_VIEWS_PATH,
        TRAIN_META_PATH,
        TEST_META_PATH,
    ]

    for p in required:
        if not p.exists():
            raise FileNotFoundError(f"Missing required input: {p}")

    train_result = build_retrieval_plans(TRAIN_VIEWS_PATH, TRAIN_META_PATH)
    test_result = build_retrieval_plans(TEST_VIEWS_PATH, TEST_META_PATH)

    train_plans = train_result["plans"]
    train_plan_views = train_result["plan_views"]

    test_plans = test_result["plans"]
    test_plan_views = test_result["plan_views"]

    train_plans.to_parquet(TRAIN_PLAN_OUT, index=False, compression="zstd")
    test_plans.to_parquet(TEST_PLAN_OUT, index=False, compression="zstd")

    train_plan_views.to_parquet(TRAIN_PLAN_VIEWS_OUT, index=False, compression="zstd")
    test_plan_views.to_parquet(TEST_PLAN_VIEWS_OUT, index=False, compression="zstd")

    summary = make_summary(
        train_plans=train_plans,
        test_plans=test_plans,
        train_plan_views=train_plan_views,
        test_plan_views=test_plan_views,
    )

    with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("DONE")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nTrain plans sample:")
    print(train_plans[[
        "topic_id",
        "strategy_name",
        "query_strength",
        "bm25_top_k",
        "vector_top_k",
        "final_top_k",
        "has_date_filter",
        "ipc_subclass",
        "use_graph_expansion_later"
    ]].head())

    print("\nTest plans sample:")
    print(test_plans[[
        "topic_id",
        "strategy_name",
        "query_strength",
        "bm25_top_k",
        "vector_top_k",
        "final_top_k",
        "has_date_filter",
        "ipc_subclass",
        "use_graph_expansion_later"
    ]].head())

    print("\nTest plan views sample:")
    print(test_plan_views[[
        "topic_id",
        "query_view",
        "use_bm25",
        "use_vector",
        "bm25_top_k",
        "vector_top_k",
        "view_weight",
    ]].head(20))

    print("\nTest strategy counts:")
    print(test_plans["strategy_name"].value_counts())

    print("\nTest query view counts:")
    print(test_plan_views["query_view"].value_counts())


if __name__ == "__main__":
    main()


BASE_DIR         = /kaggle/working/clefip2011_pac_topics
TRAIN_VIEWS_PATH = /kaggle/working/clefip2011_pac_topics/query_understanding/pac_train_query_views.parquet
TEST_VIEWS_PATH  = /kaggle/working/clefip2011_pac_topics/query_understanding/pac_test_query_views.parquet
TRAIN_META_PATH  = /kaggle/working/clefip2011_pac_topics/metadata_features/pac_train_metadata_features.parquet
TEST_META_PATH   = /kaggle/working/clefip2011_pac_topics/metadata_features/pac_test_metadata_features.parquet
OUT_DIR          = /kaggle/working/clefip2011_pac_topics/retrieval_plans
DONE
{
  "num_train_plans": 300,
  "num_test_plans": 200,
  "num_train_plan_views": 1500,
  "num_test_plan_views": 1000,
  "train_plan_out": "/kaggle/working/clefip2011_pac_topics/retrieval_plans/pac_train_retrieval_plans.parquet",
  "test_plan_out": "/kaggle/working/clefip2011_pac_topics/retrieval_plans/pac_test_retrieval_plans.parquet",
  "train_plan_views_out": "/kaggle/working/clefip2011_pac_topics/retrieval_plans/pac_train_plan

In [12]:
!pip install elasticsearch "transformers==4.45.2" "sentence-transformers==3.3.1" "einops" "peft" "accelerate"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 62.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.7/992.7 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installat

In [13]:
import os
import json
import re
import numpy as np
import torch
from collections import Counter
from typing import Dict, Any, List
from elasticsearch import Elasticsearch
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

# ================= CONFIG =================
ES_CLOUD_ID = os.getenv("ES_CLOUD_ID", "")
ES_API_KEY = os.getenv("ES_API_KEY", "")

BM25_INDEX = os.getenv("BM25_INDEX", "clef_ip_patents_v1_mini")
KNN_INDEX = os.getenv("KNN_INDEX", "clef_ip_patents_v1_mini_jina")
VECTOR_FIELD = os.getenv("ES_VECTOR_FIELD", "content_vector")
QUERY_EMB_TASK = os.getenv("QUERY_EMB_TASK", "retrieval.query")
QUERY_EMB_MAX_LEN = int(os.getenv("QUERY_EMB_MAX_LEN", "0"))
QUERY_EMB_MODEL_MAX_LEN = int(os.getenv("QUERY_EMB_MODEL_MAX_LEN", "0"))
BM25_FIELDS = [
    os.getenv("BM25_FIELD_TITLE", "title^3"),
    os.getenv("BM25_FIELD_ABSTRACT", "abstract^2"),
    os.getenv("BM25_FIELD_RETRIEVAL_TEXT", "retrieval_text^2"),
    os.getenv("BM25_FIELD_CLAIMS", "claims^1.2"),
    os.getenv("BM25_FIELD_DESCRIPTION", "description^0.3"),
]
BM25_INCLUDE_DESCRIPTION = os.getenv("BM25_INCLUDE_DESCRIPTION", "false").lower() == "true"
BM25_FIELD_PROFILES = {
    "title_abstract_full": [
        os.getenv("BM25_TA_FIELD_TITLE", "title^4.0"),
        os.getenv("BM25_TA_FIELD_ABSTRACT", "abstract^2.5"),
    ],
    "claims_short": [
        os.getenv("BM25_CLAIMS_FIELD_CLAIMS", "claims^2.6"),
        os.getenv("BM25_CLAIMS_FIELD_ABSTRACT", "abstract^0.5"),
    ],
    "combined_short": [
        os.getenv("BM25_COMBINED_FIELD_TITLE", "title^4.0"),
        os.getenv("BM25_COMBINED_FIELD_ABSTRACT", "abstract^2.5"),
        os.getenv("BM25_COMBINED_FIELD_CLAIMS", "claims^1.6"),
    ],
    "combined_fields_only": [
        os.getenv("BM25_COMBINED_FIELD_TITLE", "title^4.0"),
        os.getenv("BM25_COMBINED_FIELD_ABSTRACT", "abstract^2.5"),
        os.getenv("BM25_COMBINED_FIELD_CLAIMS", "claims^1.6"),
        os.getenv("BM25_COMBINED_FIELD_RETRIEVAL_TEXT", "retrieval_text^1.0"),
        os.getenv("BM25_COMBINED_FIELD_DESCRIPTION", "description^0.35"),
    ],
    "retrieval_text_only": [
        os.getenv("BM25_RETRIEVAL_TEXT_ONLY_FIELD", "retrieval_text^2.5"),
    ],
    "description_only": [
        os.getenv("BM25_DESCRIPTION_ONLY_FIELD", "description^2.0"),
        os.getenv("BM25_DESCRIPTION_ABSTRACT_FIELD", "abstract^0.3"),
    ],
    "all_text_fields": [
        os.getenv("BM25_ALL_FIELD_TITLE", "title^5.0"),
        os.getenv("BM25_ALL_FIELD_ABSTRACT", "abstract^3.0"),
        os.getenv("BM25_ALL_FIELD_CLAIMS", "claims^2.0"),
        os.getenv("BM25_ALL_FIELD_RETRIEVAL_TEXT", "retrieval_text^1.2"),
        os.getenv("BM25_ALL_FIELD_DESCRIPTION", "description^0.45"),
    ],
}
if BM25_INCLUDE_DESCRIPTION:
    BM25_FIELD_PROFILES["combined_short"].append(os.getenv("BM25_COMBINED_FIELD_DESCRIPTION", "description^0.15"))
BM25_TIE_BREAKER = float(os.getenv("BM25_TIE_BREAKER", "0.1"))
BM25_SELECTED_TERMS = int(os.getenv("BM25_SELECTED_TERMS", "80"))
BM25_PHRASE_TERMS = int(os.getenv("BM25_PHRASE_TERMS", "6"))
BM25_PRF_TOP_DOCS = int(os.getenv("BM25_PRF_TOP_DOCS", "20"))
BM25_PRF_EXPANSION_TERMS = int(os.getenv("BM25_PRF_EXPANSION_TERMS", "40"))
BM25_PRF_INITIAL_FETCH = int(os.getenv("BM25_PRF_INITIAL_FETCH", "150"))
# Keep a wide raw candidate pool for rerank; metrics canonical-dedup later.
RETRIEVAL_POOL_TOP_K = int(os.getenv("RETRIEVAL_POOL_TOP_K", "300"))
BM25_RAW_FETCH_K = int(os.getenv("BM25_RAW_FETCH_K", "300"))
BM25_CANONICAL_DEDUP_OUTPUT = os.getenv("BM25_CANONICAL_DEDUP_OUTPUT", "false").lower() == "true"
USE_SOFT_METADATA_BOOST = os.getenv("USE_SOFT_METADATA_BOOST", "true").lower() == "true"
VARIANT_FETCH_MULTIPLIER = int(os.getenv("VARIANT_FETCH_MULTIPLIER", "3"))
# KNN index is chunk-level: fetch more chunk hits, then aggregate back to canonical patent IDs.
KNN_CHUNK_FETCH_MULTIPLIER_ENV = os.getenv("KNN_CHUNK_FETCH_MULTIPLIER")
KNN_MAX_FETCH_SIZE_ENV = os.getenv("KNN_MAX_FETCH_SIZE")
KNN_CHUNK_FETCH_MULTIPLIER = int(KNN_CHUNK_FETCH_MULTIPLIER_ENV or "12")
KNN_MAX_FETCH_SIZE = int(KNN_MAX_FETCH_SIZE_ENV or "1500")
KNN_SCORE_AGG = os.getenv("KNN_SCORE_AGG", "max").lower()
BM25_PRF_GRID_CONFIGS = []
HYBRID_SCORE_ALPHAS = [
    float(x.strip())
    for x in os.getenv("HYBRID_SCORE_ALPHAS", "0.01,0.03,0.05,0.08,0.10,0.15,0.20").split(",")
    if x.strip()
]
HYBRID_RRF_K_VALUES = [int(x.strip()) for x in os.getenv("HYBRID_RRF_K_VALUES", "20,40,60,100").split(",") if x.strip()]
HYBRID_RRF_BM25_WEIGHTS = [float(x.strip()) for x in os.getenv("HYBRID_RRF_BM25_WEIGHTS", "0.1,0.2,0.3,0.5").split(",") if x.strip()]
HYBRID_SCORE_ALPHA = float(os.getenv("HYBRID_SCORE_ALPHA", str(HYBRID_SCORE_ALPHAS[0] if HYBRID_SCORE_ALPHAS else 0.2)))
HYBRID_SCORE_EPS = 1e-12
QUERY_CHUNKING_ENABLED = os.getenv("QUERY_CHUNKING_ENABLED", "true").lower() == "true"
QUERY_CHUNK_TOKENS = int(os.getenv("QUERY_CHUNK_TOKENS", "4096"))
QUERY_CHUNK_OVERLAP_TOKENS = int(os.getenv("QUERY_CHUNK_OVERLAP_TOKENS", "256"))
QUERY_MAX_CHUNKS = int(os.getenv("QUERY_MAX_CHUNKS", "0"))
AUTO_TUNE_RETRIEVAL = os.getenv("AUTO_TUNE_RETRIEVAL", "true").lower() == "true"
FULL_CORPUS_CHUNK_THRESHOLD = int(os.getenv("FULL_CORPUS_CHUNK_THRESHOLD", "1000000"))
RETRIEVAL_RUNTIME_CONFIG = {}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ================= ES CLIENT =================
def get_es_client():
    return Elasticsearch(
        cloud_id=ES_CLOUD_ID,
        api_key=ES_API_KEY,
        request_timeout=120
    )

def safe_count_index(es, index_name):
    try:
        if not es.indices.exists(index=index_name):
            return None
        return int(es.count(index=index_name)["count"])
    except Exception:
        return None


def apply_retrieval_autotune(es):
    global KNN_CHUNK_FETCH_MULTIPLIER, KNN_MAX_FETCH_SIZE, RETRIEVAL_RUNTIME_CONFIG

    bm25_count = safe_count_index(es, BM25_INDEX)
    knn_count = safe_count_index(es, KNN_INDEX)
    max_count = max([count for count in [bm25_count, knn_count] if count is not None] or [0])
    corpus_mode = "full" if max_count >= FULL_CORPUS_CHUNK_THRESHOLD else "mini"

    if AUTO_TUNE_RETRIEVAL:
        if corpus_mode == "full":
            if KNN_CHUNK_FETCH_MULTIPLIER_ENV is None:
                KNN_CHUNK_FETCH_MULTIPLIER = 8
            if KNN_MAX_FETCH_SIZE_ENV is None:
                KNN_MAX_FETCH_SIZE = 3000
        else:
            if KNN_CHUNK_FETCH_MULTIPLIER_ENV is None:
                KNN_CHUNK_FETCH_MULTIPLIER = 12
            if KNN_MAX_FETCH_SIZE_ENV is None:
                KNN_MAX_FETCH_SIZE = 1500

    RETRIEVAL_RUNTIME_CONFIG = {
        "auto_tune_retrieval": AUTO_TUNE_RETRIEVAL,
        "corpus_mode": corpus_mode,
        "full_corpus_chunk_threshold": FULL_CORPUS_CHUNK_THRESHOLD,
        "bm25_index": BM25_INDEX,
        "knn_index": KNN_INDEX,
        "bm25_doc_count": bm25_count,
        "knn_doc_count": knn_count,
        "knn_chunk_fetch_multiplier": KNN_CHUNK_FETCH_MULTIPLIER,
        "knn_max_fetch_size": KNN_MAX_FETCH_SIZE,
        "knn_score_agg": KNN_SCORE_AGG,
        "retrieval_pool_top_k": RETRIEVAL_POOL_TOP_K,
        "bm25_raw_fetch_k": BM25_RAW_FETCH_K,
        "bm25_canonical_dedup_output": BM25_CANONICAL_DEDUP_OUTPUT,
        "hybrid_score_alpha": HYBRID_SCORE_ALPHA,
        "hybrid_score_alphas": HYBRID_SCORE_ALPHAS,
        "query_chunking_enabled": QUERY_CHUNKING_ENABLED,
        "query_chunk_tokens": QUERY_CHUNK_TOKENS,
        "query_chunk_overlap_tokens": QUERY_CHUNK_OVERLAP_TOKENS,
        "query_max_chunks": QUERY_MAX_CHUNKS,
    }
    return RETRIEVAL_RUNTIME_CONFIG


# ================= FILTER =================
def canonical_doc_id_runtime(value):
    text = str(value or "").strip().upper()
    if not text:
        return ""
    text = text.replace(" ", "-").replace("_", "-")
    text = re.sub(r"-+", "-", text).strip("-")
    match = re.match(r"^([A-Z]{2})-?(\d+)(?:-[A-Z][0-9A-Z]*)?$", text)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return text


def build_es_filters(filter_conf: Dict[str, Any]) -> List[Dict[str, Any]]:
    filters = []

    exclude_shoulds = []
    if filter_conf.get("exclude_doc_id"):
        exclude_shoulds.append({"term": {"doc_id": filter_conf["exclude_doc_id"]}})
    exclude_canonical_doc_id = filter_conf.get("exclude_canonical_doc_id") or canonical_doc_id_runtime(filter_conf.get("exclude_doc_id"))
    if exclude_canonical_doc_id:
        exclude_shoulds.append({"term": {"canonical_doc_id": exclude_canonical_doc_id}})
    if exclude_shoulds:
        filters.append({
            "bool": {
                "must_not": [{"bool": {"should": exclude_shoulds, "minimum_should_match": 1}}]
            }
        })

    if filter_conf.get("use_date_filter") and filter_conf.get("prior_art_cutoff_date"):
        date_range = {
            "range": {
                "publication_date": {
                    "lt": filter_conf["prior_art_cutoff_date"],
                    "format": "yyyyMMdd||yyyy-MM-dd"
                }
            }
        }
        if filter_conf.get("date_filter_mode") == "strict_if_field_available":
            filters.append({
                "bool": {
                    "should": [
                        date_range,
                        {"bool": {"must_not": {"exists": {"field": "publication_date"}}}}
                    ],
                    "minimum_should_match": 1
                }
            })
        else:
            filters.append(date_range)

    return filters


def terms_boost(field, values, boost):
    values = [str(v).strip() for v in values if str(v).strip()]
    if not values:
        return None
    return {"terms": {field: values, "boost": float(boost)}}


def term_boost(field, value, boost):
    value = str(value).strip()
    if not value:
        return None
    return {"term": {field: {"value": value, "boost": float(boost)}}}


def prefix_boost(field, value, boost):
    value = str(value).strip()
    if not value:
        return None
    return {"prefix": {field: {"value": value, "boost": float(boost)}}}


def build_metadata_shoulds(boost_conf):
    if not USE_SOFT_METADATA_BOOST or not boost_conf:
        return []

    should = []

    ipc_fields = boost_conf.get("ipc_boost_fields", {}) or {}
    ipc_weights = boost_conf.get("ipc_boost_weights", {}) or {}
    if boost_conf.get("use_ipc_boost"):
        should.extend([
            terms_boost("ipc_codes", ipc_fields.get("ipc_codes_norm", []), ipc_weights.get("same_ipc_group", 1.35)),
            prefix_boost("ipc_codes", ipc_fields.get("ipc_main_group", ""), ipc_weights.get("same_ipc_main_group", 1.25)),
            prefix_boost("ipc_codes", ipc_fields.get("ipc_subclass", ""), ipc_weights.get("same_ipc_subclass", 1.18)),
            prefix_boost("ipc_codes", ipc_fields.get("ipc_class", ""), ipc_weights.get("same_ipc_class", 1.10)),
            prefix_boost("ipc_codes", ipc_fields.get("ipc_section", ""), ipc_weights.get("same_ipc_section", 1.04)),
        ])

    if boost_conf.get("use_assignee_boost"):
        should.append(terms_boost("assignees", boost_conf.get("assignees_raw", []), boost_conf.get("assignee_boost_weight", 1.08)))

    if boost_conf.get("use_inventor_boost"):
        should.append(terms_boost("inventors", boost_conf.get("inventors_raw", []), boost_conf.get("inventor_boost_weight", 1.04)))

    return [clause for clause in should if clause]


def resolve_bm25_fields(query_view):
    return BM25_FIELD_PROFILES.get(str(query_view), BM25_FIELDS)


def resolve_bm25_query_type(query_view):
    return "most_fields" if str(query_view) in {"combined_short", "combined_fields_only"} else "best_fields"


BM25_STOPWORDS = {
    "the", "and", "or", "for", "with", "from", "that", "this", "these", "those", "into", "onto", "wherein",
    "thereof", "therein", "thereby", "therefor", "said", "which", "when", "then", "than", "such", "have", "has",
    "having", "comprising", "comprises", "comprised", "including", "includes", "included", "using", "used", "use",
    "method", "apparatus", "device", "system", "means", "portion", "member", "plurality", "first", "second", "third",
    "one", "more", "least", "between", "within", "without", "about", "above", "below", "each", "other", "same",
    "can", "may", "are", "was", "were", "been", "being", "their", "its", "his", "her", "our", "your", "not",
}


def bm25_tokenize(text):
    return [
        tok.lower()
        for tok in re.findall(r"[A-Za-z][A-Za-z0-9\-]{2,}", str(text or ""))
        if tok.lower() not in BM25_STOPWORDS and not tok.isdigit()
    ]


def select_bm25_terms(text, max_terms=80):
    counts = Counter(bm25_tokenize(text))
    if not counts:
        return ""
    ranked = sorted(counts.items(), key=lambda item: (-item[1], -len(item[0]), item[0]))[:int(max_terms)]
    return " ".join(term for term, _ in ranked)


def select_bm25_phrases(text, max_phrases=6):
    tokens = bm25_tokenize(text)
    phrases = []
    seen = set()
    for n in (4, 3, 2):
        for i in range(0, max(0, len(tokens) - n + 1)):
            phrase_terms = tokens[i:i+n]
            if len(set(phrase_terms)) < max(2, n - 1):
                continue
            phrase = " ".join(phrase_terms)
            if phrase not in seen:
                seen.add(phrase)
                phrases.append(phrase)
            if len(phrases) >= int(max_phrases):
                return phrases
    return phrases


def build_enhanced_bm25_shoulds(query_text, fields, selected_terms="", expansion_terms="", selected_terms_count=None):
    selected_terms = selected_terms or select_bm25_terms(query_text, selected_terms_count or BM25_SELECTED_TERMS)
    clauses = [
        {
            "multi_match": {
                "query": query_text,
                "fields": fields,
                "type": "most_fields",
                "tie_breaker": BM25_TIE_BREAKER,
                "operator": "or",
                "boost": 1.0,
            }
        }
    ]
    if selected_terms:
        clauses.append({
            "multi_match": {
                "query": selected_terms,
                "fields": fields,
                "type": "most_fields",
                "tie_breaker": BM25_TIE_BREAKER,
                "operator": "or",
                "boost": 2.2,
            }
        })
    if expansion_terms:
        clauses.append({
            "multi_match": {
                "query": expansion_terms,
                "fields": fields,
                "type": "most_fields",
                "tie_breaker": BM25_TIE_BREAKER,
                "operator": "or",
                "boost": 0.8,
            }
        })
    for phrase in select_bm25_phrases(query_text, BM25_PHRASE_TERMS):
        clauses.append({"match_phrase": {"title": {"query": phrase, "slop": 1, "boost": 4.0}}})
        clauses.append({"match_phrase": {"abstract": {"query": phrase, "slop": 2, "boost": 2.0}}})
        clauses.append({"match_phrase": {"claims": {"query": phrase, "slop": 3, "boost": 1.2}}})
        clauses.append({"match_phrase": {"retrieval_text": {"query": phrase, "slop": 3, "boost": 1.0}}})
    return clauses

# ================= BM25 =================
def bm25_search_items(es, query_text, weight, top_k, filters, boost_conf=None, query_view="", mode="standard", expansion_terms="", selected_terms_count=None):
    if not query_text or top_k <= 0:
        return []

    top_k = int(top_k)
    fetch_size = max(top_k, top_k * max(1, VARIANT_FETCH_MULTIPLIER))
    fields = resolve_bm25_fields(query_view)
    query_type = resolve_bm25_query_type(query_view)
    # BM25 benchmark is text-only: title, abstract, claims, description. No citation/metadata boosts.
    metadata_shoulds = []
    if str(mode) in {"enhanced", "prf"}:
        text_shoulds = build_enhanced_bm25_shoulds(
            query_text,
            fields,
            expansion_terms=expansion_terms,
            selected_terms_count=selected_terms_count,
        )
        bool_query = {
            "must": [{"bool": {"should": text_shoulds, "minimum_should_match": 1}}],
            "filter": filters,
        }
    else:
        bool_query = {
            "must": [
                {
                    "multi_match": {
                        "query": query_text,
                        "fields": fields,
                        "type": query_type,
                        "tie_breaker": BM25_TIE_BREAKER,
                        "operator": "or",
                        "boost": 1.0
                    }
                }
            ],
            "filter": filters,
        }
    if metadata_shoulds:
        bool_query.setdefault("should", [])
        bool_query["should"].extend(metadata_shoulds)

    body = {
        "query": {
            "bool": bool_query
        },
        "size": fetch_size,
        "_source": ["doc_id", "canonical_doc_id"],
    }

    resp = es.search(index=BM25_INDEX, body=body)

    doc_best = {}
    for rank, hit in enumerate(resp["hits"]["hits"], start=1):
        src = hit.get("_source", {})
        doc_id = src.get("doc_id") or src.get("canonical_doc_id")
        if doc_id and doc_id not in doc_best:
            doc_best[doc_id] = {"score": float(hit.get("_score") or 0.0), "rank": rank}
            if len(doc_best) >= top_k:
                break

    ranked = sorted(doc_best.items(), key=lambda item: (-item[1]["score"], item[1]["rank"], item[0]))
    return [(doc_id, float(payload["score"]) * float(weight)) for doc_id, payload in ranked]


def bm25_search(es, query_text, weight, top_k, filters, boost_conf=None, query_view=""):
    ranked_items = bm25_search_items(es, query_text, weight, top_k, filters, boost_conf, query_view)
    return ranked_items_to_ranks(ranked_items)


def fetch_bm25_texts(es, doc_ids):
    if not doc_ids:
        return []
    resp = es.mget(
        index=BM25_INDEX,
        ids=[str(doc_id) for doc_id in doc_ids],
        _source=["title", "abstract", "claims", "retrieval_text"],
    )
    texts = []
    for doc in resp.get("docs", []):
        if not doc.get("found"):
            continue
        src = doc.get("_source", {}) or {}
        texts.append(" ".join(str(src.get(field) or "") for field in ["title", "abstract", "claims", "retrieval_text"]))
    return texts


def fetch_bm25_sources_by_id(es, doc_ids):
    if not doc_ids:
        return {}
    resp = es.mget(
        index=BM25_INDEX,
        ids=[str(doc_id) for doc_id in doc_ids],
        _source=["title", "abstract", "claims", "description", "retrieval_text"],
    )
    sources = {}
    for doc in resp.get("docs", []):
        if not doc.get("found"):
            continue
        sources[str(doc.get("_id"))] = doc.get("_source", {}) or {}
    return sources


def bm25_rescore_ranked_items(
    es,
    ranked_items,
    query_text,
    top_k=None,
    base_weight=0.60,
    overlap_weight=0.30,
    phrase_weight=0.10,
):
    if not ranked_items:
        return []
    limit = int(top_k) if top_k is not None else len(ranked_items)
    candidates = ranked_items[:max(limit, len(ranked_items))]
    doc_ids = [str(doc_id) for doc_id, _ in candidates]
    sources = fetch_bm25_sources_by_id(es, doc_ids)

    query_terms = select_bm25_terms(query_text, BM25_SELECTED_TERMS)
    query_term_set = set(bm25_tokenize(query_terms))
    query_phrases = select_bm25_phrases(query_text, BM25_PHRASE_TERMS)
    if not query_term_set and not query_phrases:
        return ranked_items[:limit]

    base_norm = minmax_normalize_ranked_items(candidates)
    rescored = []
    for rank, (doc_id, score) in enumerate(candidates, start=1):
        doc_id = str(doc_id)
        src = sources.get(doc_id, {})
        title = normalize_text(src.get("title"))
        abstract = normalize_text(src.get("abstract"))
        claims = normalize_text(src.get("claims"))
        retrieval_text = normalize_text(src.get("retrieval_text"))
        description = normalize_text(src.get("description"))

        field_score = 0.0
        denom = max(1, len(query_term_set))
        for field_text, field_weight in [
            (title, 3.0),
            (abstract, 2.0),
            (claims, 1.2),
            (retrieval_text, 0.8),
            (description, 0.25),
        ]:
            if not field_text:
                continue
            field_terms = set(bm25_tokenize(field_text))
            field_score += field_weight * (len(query_term_set & field_terms) / denom)

        combined_text = " ".join([title, abstract, claims, retrieval_text, description]).lower()
        phrase_score = sum(1.0 for phrase in query_phrases if phrase.lower() in combined_text)
        phrase_score = phrase_score / max(1, len(query_phrases))

        final_score = (
            float(base_weight) * base_norm.get(doc_id, 0.0)
            + float(overlap_weight) * field_score
            + float(phrase_weight) * phrase_score
        )
        rescored.append((doc_id, float(final_score), rank))

    rescored.sort(key=lambda item: (-item[1], item[2], item[0]))
    return [(doc_id, score) for doc_id, score, _ in rescored[:limit]]


def build_prf_expansion_terms(es, initial_items, query_text, prf_top_docs=None, prf_expansion_terms=None):
    prf_top_docs = int(prf_top_docs or BM25_PRF_TOP_DOCS)
    prf_expansion_terms = int(prf_expansion_terms or BM25_PRF_EXPANSION_TERMS)
    seed_ids = [doc_id for doc_id, _ in initial_items[:prf_top_docs]]
    feedback_text = " ".join(fetch_bm25_texts(es, seed_ids))
    if not feedback_text:
        return ""
    query_terms = set(bm25_tokenize(query_text))
    counts = Counter(bm25_tokenize(feedback_text))
    for term in list(counts):
        if term in query_terms:
            counts[term] *= 0.5
    ranked = sorted(counts.items(), key=lambda item: (-item[1], -len(item[0]), item[0]))[:prf_expansion_terms]
    return " ".join(term for term, _ in ranked)


def bm25_prf_search_items(
    es, query_text, weight, top_k, filters, boost_conf=None, query_view="",
    prf_top_docs=None, prf_expansion_terms=None, prf_initial_fetch=None, selected_terms_count=None,
):
    initial_top_k = max(int(top_k), int(prf_initial_fetch or BM25_PRF_INITIAL_FETCH))
    initial_items = bm25_search_items(
        es, query_text, weight, initial_top_k, filters, boost_conf, query_view, mode="enhanced",
        selected_terms_count=selected_terms_count,
    )
    expansion_terms = build_prf_expansion_terms(
        es, initial_items, query_text, prf_top_docs=prf_top_docs, prf_expansion_terms=prf_expansion_terms
    )
    if not expansion_terms:
        return initial_items[:int(top_k)]
    return bm25_search_items(
        es, query_text, weight, top_k, filters, boost_conf, query_view,
        mode="prf", expansion_terms=expansion_terms, selected_terms_count=selected_terms_count,
    )

# ================= EMBEDDING =================
def sanitize_length_limit(value):
    try:
        value = int(value)
    except Exception:
        return None

    if value <= 0 or value >= 1_000_000:
        return None
    return value


def resolve_model_max_length(tokenizer, model):
    candidates = [
        getattr(tokenizer, "model_max_length", None),
        getattr(getattr(model, "config", None), "max_position_embeddings", None),
        getattr(getattr(getattr(model, "config", None), "text_config", None), "max_position_embeddings", None),
        QUERY_EMB_MODEL_MAX_LEN,
        8192,
    ]

    cleaned = [limit for limit in (sanitize_length_limit(x) for x in candidates) if limit is not None]
    if not cleaned:
        raise RuntimeError("Cannot resolve a valid query embedding max length.")
    return min(cleaned)


print(f"Loading embedding model on device={DEVICE} | cuda_available={torch.cuda.is_available()}")
query_tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v3", use_fast=True, trust_remote_code=True)
jina_model = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
jina_model.eval()

QUERY_MODEL_LIMIT = resolve_model_max_length(query_tokenizer, jina_model)
QUERY_REQUESTED_MAX_LEN = sanitize_length_limit(QUERY_EMB_MAX_LEN)
QUERY_TRUNCATION_WARNED = False
QUERY_EMBED_COUNT = 0
QUERY_TRUNCATED_COUNT = 0
QUERY_MAX_OBSERVED_TOKENS = 0

print(
    f"[QUERY EMB] device={DEVICE} | task={QUERY_EMB_TASK} | "
    f"requested_max_len={QUERY_REQUESTED_MAX_LEN or 'auto'} | model_limit={QUERY_MODEL_LIMIT}"
)

embedding_cache = {}


def get_query_embedding_stats():
    return {
        "query_embedding_task": QUERY_EMB_TASK,
        "query_embedding_max_length": QUERY_REQUESTED_MAX_LEN,
        "query_embedding_model_max_length": QUERY_MODEL_LIMIT,
        "query_embedding_count": QUERY_EMBED_COUNT,
        "query_truncated_count": QUERY_TRUNCATED_COUNT,
        "query_max_observed_tokens": QUERY_MAX_OBSERVED_TOKENS,
        "query_chunking_enabled": QUERY_CHUNKING_ENABLED,
        "query_chunk_tokens": QUERY_CHUNK_TOKENS,
        "query_chunk_overlap_tokens": QUERY_CHUNK_OVERLAP_TOKENS,
        "query_max_chunks": QUERY_MAX_CHUNKS,
    }


def get_query_content_token_ids(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []
    try:
        backend = getattr(query_tokenizer, "backend_tokenizer", None)
        if backend is not None:
            return list(backend.encode(text, add_special_tokens=False).ids)
    except Exception:
        pass

    tokenized = query_tokenizer(
        text,
        padding=False,
        truncation=True,
        max_length=max(1, QUERY_MODEL_LIMIT - 2),
        add_special_tokens=False,
        return_overflowing_tokens=True,
    )
    input_ids = tokenized.get("input_ids", [])
    if input_ids and isinstance(input_ids[0], list):
        flattened = []
        for ids in input_ids:
            flattened.extend(ids)
        return flattened
    return list(input_ids)


def resolve_query_max_length(text):
    # Count via backend tokenizer to avoid the HF warning for sequences > model_max_length.
    token_length = len(get_query_content_token_ids(text)) + 2

    effective_max_length = token_length
    if QUERY_REQUESTED_MAX_LEN is not None:
        effective_max_length = min(effective_max_length, QUERY_REQUESTED_MAX_LEN)
    effective_max_length = min(effective_max_length, QUERY_MODEL_LIMIT)

    truncated = token_length > effective_max_length
    return max(1, int(effective_max_length)), int(token_length), bool(truncated)



def resolve_query_chunk_size():
    model_limit = min(QUERY_MODEL_LIMIT, QUERY_REQUESTED_MAX_LEN or QUERY_MODEL_LIMIT)
    max_content_tokens = max(1, model_limit - 2)
    if not QUERY_CHUNKING_ENABLED:
        return max_content_tokens
    return max(1, min(int(QUERY_CHUNK_TOKENS), max_content_tokens))


def build_query_text_chunks(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []

    token_ids = get_query_content_token_ids(text)
    if not token_ids:
        return []

    if not QUERY_CHUNKING_ENABLED:
        return [text]

    chunk_size = resolve_query_chunk_size()
    overlap = max(0, min(int(QUERY_CHUNK_OVERLAP_TOKENS), chunk_size - 1))
    step = max(1, chunk_size - overlap)

    chunks = []
    start = 0
    while start < len(token_ids):
        end = min(len(token_ids), start + chunk_size)
        chunk_text = " ".join(
            query_tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            ).split()
        ).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if QUERY_MAX_CHUNKS > 0 and len(chunks) >= QUERY_MAX_CHUNKS:
            break
        if end >= len(token_ids):
            break
        start += step

    return chunks or [text]

@torch.no_grad()
def embed_query(text):
    global QUERY_TRUNCATION_WARNED, QUERY_EMBED_COUNT, QUERY_TRUNCATED_COUNT, QUERY_MAX_OBSERVED_TOKENS
    if text in embedding_cache:
        return embedding_cache[text]

    text = " ".join(str(text).split()).strip()
    effective_max_length, token_length, truncated = resolve_query_max_length(text)
    QUERY_EMBED_COUNT += 1
    QUERY_MAX_OBSERVED_TOKENS = max(QUERY_MAX_OBSERVED_TOKENS, token_length)

    if truncated:
        QUERY_TRUNCATED_COUNT += 1
        if not QUERY_TRUNCATION_WARNED:
            print(
                f"[WARN] Query truncation detected | token_length={token_length} | "
                f"effective_max_length={effective_max_length}"
            )
            QUERY_TRUNCATION_WARNED = True

    # `text` should already be chunked, but pass max_length defensively for any direct calls.
    emb = jina_model.encode([text], task=QUERY_EMB_TASK, max_length=effective_max_length)

    if isinstance(emb, torch.Tensor):
        emb = emb.cpu().numpy()

    emb = np.asarray(emb, dtype=np.float32)
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)

    vec = emb[0].tolist()
    embedding_cache[text] = vec
    return vec

# ================= KNN =================
def knn_search_items(es, query_text, weight, top_k, num_candidates, filters):
    if not query_text or top_k <= 0:
        return []

    top_k = int(top_k)
    base_fetch = max(top_k, top_k * max(1, VARIANT_FETCH_MULTIPLIER))
    fetch_size = base_fetch * max(1, KNN_CHUNK_FETCH_MULTIPLIER)
    if KNN_MAX_FETCH_SIZE > 0:
        fetch_size = min(fetch_size, KNN_MAX_FETCH_SIZE)
    fetch_size = max(top_k, fetch_size)
    num_candidates = max(fetch_size, int(num_candidates) * max(1, KNN_CHUNK_FETCH_MULTIPLIER))
    if KNN_MAX_FETCH_SIZE > 0:
        num_candidates = max(fetch_size, min(num_candidates, KNN_MAX_FETCH_SIZE))

    query_chunks = build_query_text_chunks(query_text)
    query_vectors = [embed_query(chunk_text) for chunk_text in query_chunks]

    # Chunk-level hits are aggregated back to parent variant doc_id.
    # Evaluation canonicalizes later; retrieval output keeps the actual index row ID.
    doc_best = {}
    global_rank = 0
    for query_chunk_index, query_vector in enumerate(query_vectors):
        body = {
            "knn": {
                "field": VECTOR_FIELD,
                "query_vector": query_vector,
                "k": fetch_size,
                "num_candidates": num_candidates,
                "boost": 1.0,
                "filter": filters
            },
            "size": fetch_size,
            "_source": ["doc_id", "canonical_doc_id", "parent_doc_id", "chunk_id", "chunk_index"]
        }

        resp = es.search(index=KNN_INDEX, body=body)
        for chunk_rank, hit in enumerate(resp["hits"]["hits"], start=1):
            global_rank += 1
            src = hit.get("_source", {})
            doc_id = src.get("parent_doc_id") or src.get("doc_id") or src.get("canonical_doc_id")
            if not doc_id:
                continue

            score = float(hit.get("_score") or 0.0)
            if KNN_SCORE_AGG == "sum":
                current = doc_best.setdefault(doc_id, {"score": 0.0, "first_rank": global_rank, "hits": 0})
                current["score"] += score
                current["hits"] += 1
                current["first_rank"] = min(current["first_rank"], global_rank)
            else:
                current = doc_best.get(doc_id)
                if current is None or score > current["score"]:
                    doc_best[doc_id] = {"score": score, "first_rank": global_rank, "hits": 1}
                else:
                    current["hits"] += 1
                    current["first_rank"] = min(current["first_rank"], global_rank)

    ranked_docs = sorted(
        doc_best.items(),
        key=lambda item: (-item[1]["score"], item[1]["first_rank"], item[0])
    )[:top_k]

    return [(doc_id, float(payload["score"]) * float(weight)) for doc_id, payload in ranked_docs]


def knn_search(es, query_text, weight, top_k, num_candidates, filters):
    ranked_items = knn_search_items(es, query_text, weight, top_k, num_candidates, filters)
    return ranked_items_to_ranks(ranked_items)

# ================= RRF =================
def calculate_rrf(rankings, rrf_k=60):
    scores = {}
    for _, source in rankings.items():
        if isinstance(source, dict) and "ranks" in source:
            doc_ranks = source.get("ranks", {})
            weight = float(source.get("weight", 1.0))
        else:
            doc_ranks = source
            weight = 1.0

        for doc, rank in doc_ranks.items():
            scores.setdefault(doc, 0.0)
            scores[doc] += weight / (rrf_k + rank)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def minmax_normalize_ranked_items(ranked_items):
    if not ranked_items:
        return {}
    pairs = [(str(doc_id), float(score or 0.0)) for doc_id, score in ranked_items]
    scores = [score for _, score in pairs]
    score_min = min(scores)
    score_max = max(scores)
    if abs(score_max - score_min) <= HYBRID_SCORE_EPS:
        return {doc_id: 1.0 for doc_id, _ in pairs}
    return {
        doc_id: (score - score_min) / (score_max - score_min + HYBRID_SCORE_EPS)
        for doc_id, score in pairs
    }


def hybrid_score_fusion(bm25_items, knn_items, alpha=None, top_k=150):
    alpha = HYBRID_SCORE_ALPHA if alpha is None else float(alpha)
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("HYBRID_SCORE_ALPHA must be in [0, 1]")

    bm25_norm = minmax_normalize_ranked_items(bm25_items)
    knn_norm = minmax_normalize_ranked_items(knn_items)
    representatives = {}
    for doc_id, _ in knn_items:
        representatives.setdefault(str(doc_id), str(doc_id))
    for doc_id, _ in bm25_items:
        representatives.setdefault(str(doc_id), str(doc_id))

    fused = []
    for doc_id in representatives:
        score = alpha * bm25_norm.get(doc_id, 0.0) + (1.0 - alpha) * knn_norm.get(doc_id, 0.0)
        fused.append((doc_id, float(score)))

    fused.sort(key=lambda item: item[1], reverse=True)
    return fused[:int(top_k)]


def hybrid_rrf_fusion(bm25_items, knn_items, bm25_weight=0.2, rrf_k=60, top_k=150):
    sources = {
        "knn": {"ranks": ranked_items_to_ranks(knn_items), "weight": 1.0},
        "bm25": {"ranks": ranked_items_to_ranks(bm25_items), "weight": float(bm25_weight)},
    }
    return calculate_rrf(sources, rrf_k=int(rrf_k))[:int(top_k)]


def hybrid_knn_primary_union(knn_items, bm25_items, top_k=300, knn_first=200):
    top_k = int(top_k)
    knn_first = min(int(knn_first), top_k)
    merged = []
    seen = set()

    for doc_id, score in knn_items[:knn_first]:
        doc_id = str(doc_id)
        if doc_id in seen:
            continue
        seen.add(doc_id)
        merged.append((doc_id, float(score)))

    bm25_score_offset = -1.0
    for rank, (doc_id, score) in enumerate(bm25_items, start=1):
        if len(merged) >= top_k:
            break
        doc_id = str(doc_id)
        if doc_id in seen:
            continue
        seen.add(doc_id)
        merged.append((doc_id, bm25_score_offset - rank * 1e-6))

    for doc_id, score in knn_items[knn_first:]:
        if len(merged) >= top_k:
            break
        doc_id = str(doc_id)
        if doc_id in seen:
            continue
        seen.add(doc_id)
        merged.append((doc_id, float(score) - 2.0))

    return merged[:top_k]


# ================= BM25 ABLATION =================
def view_row_by_name(topic_views, query_view):
    matches = topic_views[topic_views["query_view"] == query_view]
    if matches.empty:
        return None
    return matches.iloc[0]


def retrieval_pool_top_k(plan=None, override_top_k=None):
    limits = [int(override_top_k) if override_top_k is not None else RETRIEVAL_POOL_TOP_K]
    if plan is not None and override_top_k is None:
        limits.append(int(plan.get("final_top_k", 0)))
    return max(limits)


def bm25_raw_fetch_k(base_top_k, output_top_k=None):
    limits = [int(base_top_k), BM25_RAW_FETCH_K]
    if output_top_k is not None:
        limits.append(int(output_top_k))
    return max(limits)


def canonical_dedup_runtime_ranked_items(ranked_items, top_k=None):
    deduped = []
    seen = set()
    limit = int(top_k) if top_k is not None else None
    for doc_id, score in ranked_items:
        doc_id = str(doc_id)
        canonical_id = canonical_doc_id_runtime(doc_id)
        if not canonical_id or canonical_id in seen:
            continue
        seen.add(canonical_id)
        deduped.append((doc_id, float(score)))
        if limit is not None and len(deduped) >= limit:
            break
    return deduped


def bm25_finalize_ranked_items(ranked_items, final_top_k):
    if BM25_CANONICAL_DEDUP_OUTPUT:
        return canonical_dedup_runtime_ranked_items(ranked_items, top_k=final_top_k)
    return ranked_items[:int(final_top_k)]


def run_bm25_sources(es, plan, topic_views, source_specs, output_top_k_override=None):
    filter_conf = json.loads(plan["filter_config_json"])
    boost_conf = json.loads(plan.get("boost_config_json", "{}"))
    es_filters = build_es_filters(filter_conf)

    sources = {}
    rescore_query_parts = []
    should_rescore = any(bool(spec.get("rescore")) for spec in source_specs)
    rescore_specs = [spec for spec in source_specs if bool(spec.get("rescore"))]
    rescore_conf = rescore_specs[0] if rescore_specs else {}
    output_top_k = retrieval_pool_top_k(plan, override_top_k=output_top_k_override)
    top_k = bm25_raw_fetch_k(plan["bm25_top_k"], output_top_k)

    for spec in source_specs:
        source_name = spec["name"]
        search_mode = spec.get("search_mode", "standard")
        weight = float(spec.get("weight", 1.0))

        view = view_row_by_name(topic_views, spec["query_view"])
        if view is None:
            continue

        q_text = view["query_text"]
        if not q_text:
            continue
        if should_rescore:
            rescore_query_parts.append(q_text)

        field_profile = spec.get("field_profile", spec["query_view"])
        weight = float(spec.get("weight", view.get("view_weight", 1.0)))
        if search_mode == "prf":
            ranked_items = bm25_prf_search_items(
                es, q_text, weight, top_k, es_filters, boost_conf, query_view=field_profile,
                prf_top_docs=spec.get("prf_top_docs"),
                prf_expansion_terms=spec.get("prf_expansion_terms"),
                prf_initial_fetch=spec.get("prf_initial_fetch"),
                selected_terms_count=spec.get("selected_terms"),
            )
        else:
            ranked_items = bm25_search_items(
                es, q_text, weight, top_k, es_filters, boost_conf,
                query_view=field_profile,
                mode=search_mode,
                selected_terms_count=spec.get("selected_terms"),
            )
        ranks = ranked_items_to_ranks(ranked_items)
        sources[source_name] = {"ranks": ranks, "weight": weight}

    fused = bm25_finalize_ranked_items(calculate_rrf(sources, plan["rrf_k"]), output_top_k)
    if should_rescore:
        return bm25_rescore_ranked_items(
            es,
            fused,
            " ".join(rescore_query_parts),
            top_k=output_top_k,
            base_weight=float(rescore_conf.get("rescore_base_weight", 0.60)),
            overlap_weight=float(rescore_conf.get("rescore_overlap_weight", 0.30)),
            phrase_weight=float(rescore_conf.get("rescore_phrase_weight", 0.10)),
        )
    return fused


def run_bm25_ablation_for_topic(es, plan, topic_views, output_top_k_override=None):
    # Best observed BM25 baseline: title/abstract + full claims, enhanced BM25, RRF merge.
    strategies = {
        "bm25_enhanced_2view_claims_full_rrf": [
            {"name": "enhanced_ta", "query_view": "title_abstract_full", "field_profile": "all_text_fields", "weight": 1.10, "search_mode": "enhanced"},
            {"name": "enhanced_claims_full", "query_view": "claims_full", "field_profile": "all_text_fields", "weight": 1.00, "search_mode": "enhanced"},
        ],
    }

    return {
        name: run_bm25_sources(es, plan, topic_views, specs, output_top_k_override=output_top_k_override)
        for name, specs in strategies.items()
    }

# ================= KNN =================
def run_knn_for_topic(es, plan, topic_views, output_top_k_override=None):
    filter_conf = json.loads(plan["filter_config_json"])
    es_filters = build_es_filters(filter_conf)

    rankings = {"knn": {}}

    for _, view in topic_views.iterrows():
        q_text = view["query_text"]
        weight = view["view_weight"]
        query_view = view["query_view"]

        if view["use_vector"]:
            knn_items = knn_search_items(
                es, q_text, weight,
                retrieval_pool_top_k(plan, override_top_k=output_top_k_override),
                max(int(plan["candidate_pool_size"]), retrieval_pool_top_k(plan, override_top_k=output_top_k_override)),
                es_filters
            )
            knn_ranks = ranked_items_to_ranks(knn_items)
            source_name = f"knn:{query_view}"
            rankings["knn"][source_name] = {"ranks": knn_ranks, "weight": weight}

    output_top_k = retrieval_pool_top_k(plan, override_top_k=output_top_k_override)
    knn_full = calculate_rrf(rankings["knn"], plan["rrf_k"])

    return {
        "knn": knn_full[:output_top_k],
    }



Loading embedding model on device=cuda | cuda_available=True


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- stochastic_depth.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


mha.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- mha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- block.py
- stochastic_depth.py
- mha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- modeling_xlm_roberta.py
- embedding.py
- mlp.py
- xlm_padding.py
- rotary.py
- block.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downl

model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

[QUERY EMB] device=cuda | task=retrieval.query | requested_max_len=auto | model_limit=8192


In [14]:
# ================= RETRIEVAL EXPERIMENTS: BM25 / KNN / HYBRID =================
# No agent routing here: every benchmark query runs all retrieval modes.
from pathlib import Path
import time
import re

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

BASE_DIR = Path(os.getenv("PAC_TOPIC_WORK_DIR", "/kaggle/working/clefip2011_pac_topics"))
PLAN_DIR = BASE_DIR / "retrieval_plans"
PROCESSED_DIR = BASE_DIR / "processed"
OUT_DIR = BASE_DIR / "retrieval_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_PRESET = os.getenv("BENCHMARK_PRESET", "quick").strip().lower()
if BENCHMARK_PRESET in {"quick", "quick_200", "200"}:
    DEFAULT_NON_CITATION_QUERY_COUNT = 100
    DEFAULT_CITATION_QUERY_COUNT = 100
else:
    DEFAULT_NON_CITATION_QUERY_COUNT = 200
    DEFAULT_CITATION_QUERY_COUNT = 200
TARGET_NON_CITATION_QUERY_COUNT = int(os.getenv("TARGET_NON_CITATION_QUERY_COUNT", str(DEFAULT_NON_CITATION_QUERY_COUNT)))
TARGET_CITATION_QUERY_COUNT = int(os.getenv("TARGET_CITATION_QUERY_COUNT", str(DEFAULT_CITATION_QUERY_COUNT)))
TARGET_BENCHMARK_QUERY_COUNT = TARGET_NON_CITATION_QUERY_COUNT + TARGET_CITATION_QUERY_COUNT

TEST_PLAN_PATH = Path(os.getenv("TEST_PLAN_PATH", str(PLAN_DIR / "pac_test_retrieval_plans.parquet")))
TEST_PLAN_VIEWS_PATH = Path(os.getenv("TEST_PLAN_VIEWS_PATH", str(PLAN_DIR / "pac_test_plan_views.parquet")))
QRELS_PATH = Path(os.getenv(
    "QRELS_PATH",
    str(PROCESSED_DIR / f"pac_test_qrels_benchmark_{TARGET_BENCHMARK_QUERY_COUNT}.parquet")
))

RESULTS_OUT = OUT_DIR / "retrieval_rankings_final_bm25_knn_hybrid.parquet"
TOPIC_METRICS_OUT = OUT_DIR / "retrieval_topic_metrics_final_bm25_knn_hybrid.parquet"
SUMMARY_OUT = OUT_DIR / "retrieval_summary_final_bm25_knn_hybrid.csv"
SUMMARY_JSON_OUT = OUT_DIR / "retrieval_summary_final_bm25_knn_hybrid.json"

EVAL_KS = [1, 2, 3, 5, 10, 20, 30, 50, 100]


def canonical_doc_id(value):
    text = str(value or "").strip().upper()
    if not text:
        return ""
    text = text.replace(" ", "-").replace("_", "-")
    text = re.sub(r"-+", "-", text).strip("-")
    match = re.match(r"^([A-Z]{2})-?(\d+)(?:-[A-Z][0-9A-Z]*)?$", text)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return text


BM25_METHODS = ["bm25_enhanced_2view_claims_full_rrf"]
def alpha_method_name(base_name, alpha):
    return f"{base_name}_a{int(round(float(alpha) * 100)):02d}"


def parse_float_list_env(name, default):
    text = os.getenv(name, default)
    values = []
    for part in str(text).split(","):
        part = part.strip()
        if not part:
            continue
        values.append(float(part))
    return values


HYBRID_NORM_KNN_ALPHA = max(0.0, min(1.0, float(os.getenv("HYBRID_NORM_KNN_ALPHA", "0.99"))))
HYBRID_DEEP_POOL_TOP_K = int(os.getenv("HYBRID_DEEP_POOL_TOP_K", "1000"))
HYBRID_DEEP_METHOD = f"hybrid_deep_norm_knn{int(round(HYBRID_NORM_KNN_ALPHA * 100)):02d}_bm25{int(round((1.0 - HYBRID_NORM_KNN_ALPHA) * 100)):02d}"
HYBRID_DEEP_RRF_METHOD = "hybrid_deep_canonical_rrf_bm25_knn"
HYBRID_METHODS = [HYBRID_DEEP_METHOD, HYBRID_DEEP_RRF_METHOD]
METHODS = BM25_METHODS + ["knn"] + HYBRID_METHODS
ACTIVE_RETRIEVAL_VIEWS = {
    "title_abstract_full",
    "claims_short",
    "combined_short",
    "claims_full",
    "combined_full",
    "description_short",
    "embedding_text",
    "metadata",
}


def ranked_items_to_ranks(ranked_items):
    return {str(doc_id): rank for rank, (doc_id, _) in enumerate(ranked_items, start=1)}


def fuse_ranked_items(source_specs, rrf_k, top_k):
    sources = {}
    for name, ranked_items, weight in source_specs:
        if not ranked_items:
            continue
        sources[name] = {"ranks": ranked_items_to_ranks(ranked_items), "weight": float(weight)}
    return calculate_rrf(sources, rrf_k)[:int(top_k)] if sources else []


def normalize_source_scores_by_canonical(ranked_items):
    best_by_canonical = {}
    for rank, (doc_id, score) in enumerate(ranked_items or [], start=1):
        doc_id = str(doc_id)
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id:
            continue
        try:
            score = float(score)
        except Exception:
            score = 0.0
        current = best_by_canonical.get(canonical_id)
        if current is None or score > current["score"] or (score == current["score"] and rank < current["rank"]):
            best_by_canonical[canonical_id] = {"doc_id": doc_id, "score": score, "rank": rank}

    if not best_by_canonical:
        return {}

    scores = [item["score"] for item in best_by_canonical.values()]
    min_score = min(scores)
    max_score = max(scores)
    denom = max_score - min_score
    normalized = {}
    for canonical_id, item in best_by_canonical.items():
        if denom > 1e-12:
            norm_score = (item["score"] - min_score) / denom
        else:
            # If all scores are tied, use reciprocal rank as a stable fallback signal.
            norm_score = 1.0 / (1.0 + float(item["rank"] - 1))
        normalized[canonical_id] = {
            "doc_id": item["doc_id"],
            "score": float(norm_score),
            "rank": int(item["rank"]),
        }
    return normalized


def canonical_rank_map_for_fusion(ranked_items):
    ranks = {}
    doc_ids = {}
    for rank, (doc_id, _score) in enumerate(ranked_items or [], start=1):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in ranks:
            continue
        ranks[canonical_id] = rank
        doc_ids[canonical_id] = str(doc_id)
    return ranks, doc_ids


def hybrid_deep_canonical_rrf(knn_ranked, bm25_ranked, top_k=300, rrf_k=60, knn_weight=1.0, bm25_weight=1.0):
    knn_ranks, knn_doc_ids = canonical_rank_map_for_fusion(knn_ranked)
    bm25_ranks, bm25_doc_ids = canonical_rank_map_for_fusion(bm25_ranked)
    all_canonicals = set(knn_ranks) | set(bm25_ranks)
    scored = []
    for canonical_id in all_canonicals:
        score = 0.0
        best_rank = 10**12
        source_hit_count = 0
        if canonical_id in knn_ranks:
            score += float(knn_weight) / (float(rrf_k) + float(knn_ranks[canonical_id]))
            best_rank = min(best_rank, knn_ranks[canonical_id])
            source_hit_count += 1
        if canonical_id in bm25_ranks:
            score += float(bm25_weight) / (float(rrf_k) + float(bm25_ranks[canonical_id]))
            best_rank = min(best_rank, bm25_ranks[canonical_id])
            source_hit_count += 1
        doc_id = knn_doc_ids.get(canonical_id) or bm25_doc_ids.get(canonical_id) or canonical_id
        scored.append((doc_id, float(score), source_hit_count, best_rank))
    scored.sort(key=lambda item: (item[1], item[2], -item[3], item[0]), reverse=True)
    return [(doc_id, score) for doc_id, score, _source_hit_count, _best_rank in scored[:int(top_k)]]


def hybrid_normalized_weighted_sum(knn_ranked, bm25_ranked, knn_alpha=0.80, top_k=300):
    knn_alpha = max(0.0, min(1.0, float(knn_alpha)))
    bm25_alpha = 1.0 - knn_alpha
    sources = [
        ("knn", normalize_source_scores_by_canonical(knn_ranked), knn_alpha),
        ("bm25", normalize_source_scores_by_canonical(bm25_ranked), bm25_alpha),
    ]
    fused = {}
    for source_name, source_scores, weight in sources:
        if not source_scores or weight <= 0:
            continue
        for canonical_id, item in source_scores.items():
            entry = fused.setdefault(canonical_id, {
                "doc_id": item["doc_id"],
                "score": 0.0,
                "best_rank": item["rank"],
                "source_hit_count": 0,
            })
            entry["score"] += weight * item["score"]
            entry["source_hit_count"] += 1
            if item["rank"] < entry["best_rank"]:
                entry["best_rank"] = item["rank"]
                entry["doc_id"] = item["doc_id"]

    ranked = sorted(
        fused.values(),
        key=lambda item: (item["score"], item["source_hit_count"], -item["best_rank"]),
        reverse=True,
    )
    return [(item["doc_id"], float(item["score"])) for item in ranked[:int(top_k)]]


def build_hybrid_ablation_results(result, bm25_ablation, plan):
    top_k = retrieval_pool_top_k(plan)
    rrf_k = int(plan["rrf_k"])
    knn_ranked = result.get("knn", [])
    bm25_enhanced_2view_claims_full = bm25_ablation.get("bm25_enhanced_2view_claims_full_rrf", [])

    knn_alpha = max(0.0, min(1.0, HYBRID_NORM_KNN_ALPHA))
    method_name = f"hybrid_deep_norm_knn{int(round(knn_alpha * 100)):02d}_bm25{int(round((1.0 - knn_alpha) * 100)):02d}"
    return {
        method_name: hybrid_normalized_weighted_sum(
            knn_ranked,
            bm25_enhanced_2view_claims_full,
            knn_alpha=knn_alpha,
            top_k=top_k,
        ),
        HYBRID_DEEP_RRF_METHOD: hybrid_deep_canonical_rrf(
            knn_ranked,
            bm25_enhanced_2view_claims_full,
            top_k=top_k,
            rrf_k=int(plan.get("rrf_k", 60)),
            knn_weight=1.0,
            bm25_weight=1.0,
        ),
    }


def load_retrieval_inputs():
    for p in [TEST_PLAN_PATH, TEST_PLAN_VIEWS_PATH, QRELS_PATH]:
        if not p.exists():
            raise FileNotFoundError(f"Missing required retrieval input: {p}")

    plans = pd.read_parquet(TEST_PLAN_PATH)
    views = pd.read_parquet(TEST_PLAN_VIEWS_PATH)
    qrels = pd.read_parquet(QRELS_PATH)

    plans["topic_id"] = plans["topic_id"].astype(str).str.strip()
    views["topic_id"] = views["topic_id"].astype(str).str.strip()
    qrels["topic_id"] = qrels["topic_id"].astype(str).str.strip()
    qrels["candidate_doc_id_raw"] = qrels["candidate_doc_id"].astype(str).str.strip()
    if "candidate_canonical_doc_id" in qrels.columns:
        qrels["candidate_doc_id"] = qrels["candidate_canonical_doc_id"].astype(str).str.strip().map(canonical_doc_id)
    else:
        qrels["candidate_doc_id"] = qrels["candidate_doc_id_raw"].map(canonical_doc_id)

    if "relevance" in qrels.columns:
        qrels = qrels[pd.to_numeric(qrels["relevance"], errors="coerce").fillna(0) > 0].copy()

    benchmark_topic_ids = sorted(qrels["topic_id"].unique().tolist())
    plans = plans[plans["topic_id"].isin(benchmark_topic_ids)].copy()
    views = views[views["topic_id"].isin(benchmark_topic_ids)].copy()
    views = views[views["query_view"].isin(ACTIVE_RETRIEVAL_VIEWS)].copy()

    missing_plans = sorted(set(benchmark_topic_ids) - set(plans["topic_id"]))
    missing_views = sorted(set(benchmark_topic_ids) - set(views["topic_id"]))
    if missing_plans or missing_views:
        raise ValueError(
            "Retrieval input mismatch. Run/rebuild the retrieval-plan cell after changing BENCHMARK_PRESET or QRELS_PATH. "
            f"BENCHMARK_PRESET={BENCHMARK_PRESET!r}; "
            f"TEST_PLAN_PATH={TEST_PLAN_PATH}; TEST_PLAN_VIEWS_PATH={TEST_PLAN_VIEWS_PATH}; QRELS_PATH={QRELS_PATH}; "
            f"plans_topics={plans['topic_id'].nunique()}; views_topics={views['topic_id'].nunique()}; qrels_topics={len(benchmark_topic_ids)}; "
            f"missing_plans={len(missing_plans)}, sample={missing_plans[:5]}; "
            f"missing_views={len(missing_views)}, sample={missing_views[:5]}"
        )

    plans = plans.sort_values("topic_id").drop_duplicates("topic_id", keep="first").reset_index(drop=True)
    views = views.sort_values(["topic_id", "query_view"]).reset_index(drop=True)

    relevant_by_topic = {
        topic_id: set(group["candidate_doc_id"].astype(str))
        for topic_id, group in qrels.groupby("topic_id")
    }
    topic_groups = (
        qrels[["topic_id", "has_citations"]]
        .drop_duplicates("topic_id")
        .assign(citation_group=lambda df: df["has_citations"].map({True: "with_citation", False: "without_citation"}))
        [["topic_id", "citation_group"]]
    ) if "has_citations" in qrels.columns else pd.DataFrame({
        "topic_id": benchmark_topic_ids,
        "citation_group": "unknown",
    })

    return plans, views, qrels, relevant_by_topic, topic_groups


def ranked_doc_ids(ranked_items):
    return [str(doc_id) for doc_id, _ in ranked_items]


def canonical_dedup_ranked_items(ranked_items, top_k=None):
    """Metric-only adapter: collapse publication-kind variants to canonical qrels IDs.

    Retrieval/rerank keep the real indexed doc_id, e.g. EP-xxxx-B1. Before scoring,
    EP-xxxx-A1/B1/B2 are mapped to EP-xxxx and counted once at the best rank.
    """
    deduped = []
    seen = set()
    limit = int(top_k) if top_k is not None else None
    for doc_id, score in ranked_items:
        doc_id = str(doc_id)
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            continue
        seen.add(canonical_id)
        deduped.append((doc_id, float(score)))
        if limit is not None and len(deduped) >= limit:
            break
    return deduped


def canonical_ranked_doc_ids(ranked_items):
    return [canonical_doc_id(doc_id) for doc_id, _ in canonical_dedup_ranked_items(ranked_items)]


def recall_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    return len(set(ranked[:k]) & relevant) / len(relevant)


def precision_at_k(ranked, relevant, k):
    if k <= 0:
        return 0.0
    retrieved = ranked[:k]
    if not retrieved:
        return 0.0
    return len(set(retrieved) & relevant) / float(k)


def f1_at_k(ranked, relevant, k):
    precision = precision_at_k(ranked, relevant, k)
    recall = recall_at_k(ranked, relevant, k)
    denom = precision + recall
    return 0.0 if denom <= 0 else (2.0 * precision * recall / denom)


def hit_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    return 1.0 if set(ranked[:k]) & relevant else 0.0


def evaluate_ranking(topic_id, method, ranked_items, relevant):
    ranked_variants = ranked_doc_ids(ranked_items)
    metric_ranked_items = canonical_dedup_ranked_items(ranked_items)
    ranked = canonical_ranked_doc_ids(metric_ranked_items)
    row = {
        "topic_id": topic_id,
        "method": method,
        "num_relevant": len(relevant),
        "num_retrieved": len(ranked_variants),
        "num_retrieved_canonical": len(ranked),
        "num_duplicate_variants_removed": len(ranked_variants) - len(ranked),
    }
    for k in EVAL_KS:
        row[f"precision@{k}"] = precision_at_k(ranked, relevant, k)
        row[f"recall@{k}"] = recall_at_k(ranked, relevant, k)
        row[f"f1@{k}"] = f1_at_k(ranked, relevant, k)
        row[f"hit@{k}"] = hit_at_k(ranked, relevant, k)
    return row


def ranking_rows(topic_id, method, ranked_items):
    return [
        {
            "topic_id": topic_id,
            "method": method,
            "rank": rank,
            "doc_id": str(doc_id),
            "score": float(score),
        }
        for rank, (doc_id, score) in enumerate(ranked_items, start=1)
    ]


def summarize_metrics(topic_metrics, group_cols=None):
    if group_cols is None:
        group_cols = ["method"]
    metric_cols = [c for c in topic_metrics.columns if "@" in c]
    optional_mean_cols = [
        "num_retrieved_canonical",
        "num_duplicate_variants_removed",
    ]
    agg_spec = {
        "topic_id": "nunique",
        "num_relevant": "mean",
        "num_retrieved": "mean",
        **{col: "mean" for col in optional_mean_cols if col in topic_metrics.columns},
        **{col: "mean" for col in metric_cols},
    }
    summary = (
        topic_metrics
        .groupby(group_cols, as_index=False)
        .agg(agg_spec)
        .rename(columns={
            "topic_id": "num_topics",
            "num_relevant": "avg_relevant_per_topic",
            "num_retrieved": "avg_retrieved_per_topic",
            "num_retrieved_canonical": "avg_retrieved_canonical_per_topic",
            "num_duplicate_variants_removed": "avg_duplicate_variants_removed",
        })
    )

    sort_cols = ["f1@100", "recall@100", "precision@100", "hit@100"]
    sort_cols = [c for c in sort_cols if c in summary.columns]
    return summary.sort_values(sort_cols, ascending=False).reset_index(drop=True)


def run_retrieval_experiments():
    plans, views, qrels, relevant_by_topic, topic_groups = load_retrieval_inputs()
    topic_to_views = {topic_id: group.copy() for topic_id, group in views.groupby("topic_id")}

    print("[INPUT]")
    print("plans:", plans.shape, "| views:", views.shape, "| qrels:", qrels.shape)
    print("benchmark topics:", len(relevant_by_topic))
    print("methods:", METHODS)
    print("eval ks:", EVAL_KS)

    es = get_es_client()
    if not es.ping():
        raise RuntimeError("Cannot connect to Elasticsearch.")

    runtime_config = apply_retrieval_autotune(es)
    print("[RETRIEVAL RUNTIME]")
    print(json.dumps(runtime_config, ensure_ascii=False, indent=2))

    all_rank_rows = []
    metric_rows = []
    t0 = time.time()

    for plan in tqdm(plans.to_dict(orient="records"), desc="Retrieval experiments"):
        topic_id = str(plan["topic_id"])
        topic_views = topic_to_views.get(topic_id, pd.DataFrame())
        if topic_views.empty:
            continue

        result = run_knn_for_topic(es, plan, topic_views)
        bm25_ablation = run_bm25_ablation_for_topic(es, plan, topic_views)
        result.update(bm25_ablation)

        deep_result = run_knn_for_topic(es, plan, topic_views, output_top_k_override=HYBRID_DEEP_POOL_TOP_K)
        deep_bm25_ablation = run_bm25_ablation_for_topic(es, plan, topic_views, output_top_k_override=HYBRID_DEEP_POOL_TOP_K)
        result.update(build_hybrid_ablation_results(deep_result, deep_bm25_ablation, plan))
        relevant = relevant_by_topic.get(topic_id, set())

        for method in METHODS:
            ranked_items = result.get(method, [])
            all_rank_rows.extend(ranking_rows(topic_id, method, ranked_items))
            metric_rows.append(evaluate_ranking(topic_id, method, ranked_items, relevant))

    rankings = pd.DataFrame(all_rank_rows)
    topic_metrics = pd.DataFrame(metric_rows)
    topic_metrics = topic_metrics.merge(topic_groups, on="topic_id", how="left")
    topic_metrics["citation_group"] = topic_metrics["citation_group"].fillna("unknown")
    summary = summarize_metrics(topic_metrics)
    rankings.to_parquet(RESULTS_OUT, index=False, compression="zstd")
    topic_metrics.to_parquet(TOPIC_METRICS_OUT, index=False, compression="zstd")
    summary.to_csv(SUMMARY_OUT, index=False)

    payload = {
        "methods": METHODS,
        "eval_ks": EVAL_KS,
        "num_topics": int(len(relevant_by_topic)),
        "num_rank_rows": int(len(rankings)),
        "elapsed_sec": round(time.time() - t0, 2),
        "results_out": str(RESULTS_OUT),
        "topic_metrics_out": str(TOPIC_METRICS_OUT),
        "summary_out": str(SUMMARY_OUT),
        "summary": summary.to_dict(orient="records"),
        "retrieval_runtime_config": RETRIEVAL_RUNTIME_CONFIG,
        "hybrid_norm_knn_alpha": HYBRID_NORM_KNN_ALPHA,
        "hybrid_deep_pool_top_k": HYBRID_DEEP_POOL_TOP_K,
        "query_embedding_stats": get_query_embedding_stats(),
    }
    with open(SUMMARY_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("\n[SUMMARY]")
    display_cols = [
        "method",
        "num_topics",
        "avg_retrieved_per_topic",
        "avg_retrieved_canonical_per_topic",
        "avg_duplicate_variants_removed",
        "hit@1",
        "precision@1",
        "recall@1",
        "f1@1",
        "hit@3",
        "precision@3",
        "recall@2",
        "recall@3",
        "f1@3",
        "hit@5",
        "precision@5",
        "recall@5",
        "f1@5",
        "hit@10",
        "precision@10",
        "recall@10",
        "f1@10",
        "hit@20",
        "recall@20",
        "f1@20",
        "hit@30",
        "recall@30",
        "f1@30",
        "hit@50",
        "recall@50",
        "f1@50",
        "hit@100",
        "precision@100",
        "recall@100",
        "f1@100",
    ]
    display_cols = [c for c in display_cols if c in summary.columns]
    print(summary[display_cols].to_string(index=False))

    best = summary.iloc[0]["method"] if len(summary) else ""
    print(f"\n[BEST_BY_SORT] {best}")
    bm25_summary = summary[summary["method"].isin(BM25_METHODS)].copy()
    if len(bm25_summary):
        best_bm25 = bm25_summary.iloc[0]["method"]
        print(f"[BEST_BM25] {best_bm25}")
    hybrid_summary = summary[summary["method"].isin(HYBRID_METHODS)].copy()
    if len(hybrid_summary):
        best_hybrid = hybrid_summary.iloc[0]["method"]
        print(f"[BEST_HYBRID] {best_hybrid}")
    print("[SAVED]", RESULTS_OUT)
    print("[SAVED]", TOPIC_METRICS_OUT)
    print("[SAVED]", SUMMARY_OUT)

    return rankings, topic_metrics, summary


rankings_df, topic_metrics_df, retrieval_summary_df = run_retrieval_experiments()


[INPUT]
plans: (200, 36) | views: (1000, 16) | qrels: (630, 10)
benchmark topics: 200
methods: ['bm25_enhanced_2view_claims_full_rrf', 'knn', 'hybrid_deep_norm_knn99_bm2501', 'hybrid_deep_canonical_rrf_bm25_knn']
eval ks: [1, 2, 3, 5, 10, 20, 30, 50, 100]
[RETRIEVAL RUNTIME]
{
  "auto_tune_retrieval": true,
  "corpus_mode": "mini",
  "full_corpus_chunk_threshold": 1000000,
  "bm25_index": "clef_ip_patents_v1_mini",
  "knn_index": "clef_ip_patents_v1_mini_jina",
  "bm25_doc_count": 2752,
  "knn_doc_count": 3932,
  "knn_chunk_fetch_multiplier": 12,
  "knn_max_fetch_size": 1500,
  "knn_score_agg": "max",
  "retrieval_pool_top_k": 300,
  "bm25_raw_fetch_k": 300,
  "bm25_canonical_dedup_output": false,
  "hybrid_score_alpha": 0.01,
  "hybrid_score_alphas": [
    0.01,
    0.03,
    0.05,
    0.08,
    0.1,
    0.15,
    0.2
  ],
  "query_chunking_enabled": true,
  "query_chunk_tokens": 4096,
  "query_chunk_overlap_tokens": 256,
  "query_max_chunks": 0
}


Retrieval experiments:   0%|          | 0/200 [00:00<?, ?it/s]

2026-05-12 23:19:54.985980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778627995.211271      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778627995.274364      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778627995.814396      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778627995.814434      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778627995.814437      57 computation_placer.cc:177] computation placer alr


[SUMMARY]
                             method  num_topics  avg_retrieved_per_topic  avg_retrieved_canonical_per_topic  avg_duplicate_variants_removed  hit@1  precision@1  recall@1   f1@1  hit@3  precision@3  recall@2  recall@3     f1@3  hit@5  precision@5  recall@5     f1@5  hit@10  precision@10  recall@10    f1@10  hit@20  recall@20    f1@20  hit@30  recall@30    f1@30  hit@50  recall@50    f1@50  hit@100  precision@100  recall@100   f1@100
      hybrid_deep_norm_knn99_bm2501         200                    300.0                             300.00                            0.00  0.825        0.825  0.264167 0.3995  0.925     0.670000  0.472083  0.641667 0.653810  0.965        0.476  0.756250 0.582639   0.985         0.265   0.840833 0.402088   0.995   0.895000 0.243261   1.000   0.921667 0.175062   1.000   0.940833 0.111488     1.00         0.0302    0.958750 0.058533
                                knn         200                    300.0                             207.02          

In [15]:
# ================= GRAPH RETRIEVAL EXPERIMENTS =================
# Run this after the BM25/KNN/hybrid retrieval cell above.
import os
import gc
import math
import time
import json
import re
from pathlib import Path
from collections import defaultdict
from typing import Any, List

import numpy as np
import pandas as pd
from elasticsearch import helpers

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

def graph_dir_has_edge_files(graph_dir: Path) -> bool:
    required = ["edges_citation", "edges_inventor", "edges_assignee", "edges_ipc"]
    return graph_dir.exists() and all(any((graph_dir / name).glob("*.parquet")) for name in required)


def discover_graph_dir() -> Path:
    explicit = os.getenv("GRAPH_DIR")
    if explicit:
        return Path(explicit)
    candidates = [
        Path("/kaggle/input/datasets/djnhngocduc/graph-parquet"),
        Path("/kaggle/input/datasets/djnhngocduc/graph-parquet/graph_parquet"),
        Path("/kaggle/working/graph_parquet"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(sorted(input_root.glob("*/graph_parquet")))
        candidates.extend([p for p in sorted(input_root.glob("*")) if graph_dir_has_edge_files(p)])
    for candidate in candidates:
        if graph_dir_has_edge_files(candidate):
            return candidate
    return candidates[0]


GRAPH_DIR = discover_graph_dir()
GRAPH_SOURCE_PARQUET_DIR = Path(os.getenv(
    "GRAPH_SOURCE_PARQUET_DIR",
    "/kaggle/input/datasets/djnhngocduc/patent-v2-parquet-shard-index/clefip_es_chunked/es_ready_shards",
))
GRAPH_ENABLED = os.getenv("GRAPH_ENABLED", "true").lower() == "true"
GRAPH_BUILD_IF_MISSING = os.getenv("GRAPH_BUILD_IF_MISSING", "true").lower() == "true"
print(f"[GRAPH] GRAPH_DIR={GRAPH_DIR}")
GRAPH_FILTER_TO_BM25_INDEX = os.getenv("GRAPH_FILTER_TO_BM25_INDEX", "true").lower() == "true"
GRAPH_SEED_TOP_K = int(os.getenv("GRAPH_SEED_TOP_K", "75"))
GRAPH_WITH_CITATION_SEED_TOP_K = int(os.getenv("GRAPH_WITH_CITATION_SEED_TOP_K", "50"))
GRAPH_NO_CITATION_SEED_TOP_K = int(os.getenv("GRAPH_NO_CITATION_SEED_TOP_K", "100"))
GRAPH_CANDIDATE_TOP_K = int(os.getenv("GRAPH_CANDIDATE_TOP_K", "300"))
GRAPH_ADAPTIVE_WITH_CITATION_WEIGHT = float(os.getenv("GRAPH_ADAPTIVE_WITH_CITATION_WEIGHT", "0.15"))
GRAPH_ADAPTIVE_NO_CITATION_WEIGHT = float(os.getenv("GRAPH_ADAPTIVE_NO_CITATION_WEIGHT", "0.05"))
GRAPH_ADAPTIVE_WITH_CITATION_RRF_K = int(os.getenv("GRAPH_ADAPTIVE_WITH_CITATION_RRF_K", "40"))
GRAPH_ADAPTIVE_NO_CITATION_RRF_K = int(os.getenv("GRAPH_ADAPTIVE_NO_CITATION_RRF_K", "20"))
GRAPH_GENERATION_HYBRID_KEEP = int(os.getenv("GRAPH_GENERATION_HYBRID_KEEP", "150"))
GRAPH_GENERATION_GRAPH_KEEP = int(os.getenv("GRAPH_GENERATION_GRAPH_KEEP", "150"))
GRAPH_RAG_SECOND_HOP_TOP_K = int(os.getenv("GRAPH_RAG_SECOND_HOP_TOP_K", "50"))
GRAPH_RAG_BACKFILL_TOP_K = int(os.getenv("GRAPH_RAG_BACKFILL_TOP_K", "100"))
GRAPH_RAG_PRESERVE_TOP_20 = int(os.getenv("GRAPH_RAG_PRESERVE_TOP_20", "20"))
GRAPH_RAG_PRESERVE_TOP_50 = int(os.getenv("GRAPH_RAG_PRESERVE_TOP_50", "50"))
GRAPH_ONLY_RAG_SEED_TOP_K = int(os.getenv("GRAPH_ONLY_RAG_SEED_TOP_K", "80"))
GRAPH_ONLY_RAG_SECOND_HOP_TOP_K = int(os.getenv("GRAPH_ONLY_RAG_SECOND_HOP_TOP_K", "60"))
GRAPH_ONLY_RAG_QUERY_WEIGHT = float(os.getenv("GRAPH_ONLY_RAG_QUERY_WEIGHT", "1.35"))
GRAPH_ONLY_RAG_HOP1_WEIGHT = float(os.getenv("GRAPH_ONLY_RAG_HOP1_WEIGHT", "0.90"))
GRAPH_ONLY_RAG_HOP2_WEIGHT = float(os.getenv("GRAPH_ONLY_RAG_HOP2_WEIGHT", "0.45"))
GRAPH_HYBRID_RAG_WEIGHT = float(os.getenv("GRAPH_HYBRID_RAG_WEIGHT", "0.08"))
GRAPH_HYBRID_RAG_RRF_K = int(os.getenv("GRAPH_HYBRID_RAG_RRF_K", "80"))
GRAPH_ONLY_BALANCED_PRESERVE_TOP = int(os.getenv("GRAPH_ONLY_BALANCED_PRESERVE_TOP", "20"))
GRAPH_ONLY_BALANCED_BACKFILL_TOP = int(os.getenv("GRAPH_ONLY_BALANCED_BACKFILL_TOP", "120"))
GRAPH_SAFE_RERANK_PRESERVE_TOP = int(os.getenv("GRAPH_SAFE_RERANK_PRESERVE_TOP", "10"))
GRAPH_SAFE_RERANK_TOP_K = int(os.getenv("GRAPH_SAFE_RERANK_TOP_K", "100"))
GRAPH_SAFE_RERANK_WEIGHT = float(os.getenv("GRAPH_SAFE_RERANK_WEIGHT", "0.12"))
GRAPH_PATH_RAG_CANDIDATE_TOP_K = int(os.getenv("GRAPH_PATH_RAG_CANDIDATE_TOP_K", "1000"))
GRAPH_PATH_RAG_SEED_TOP_K = int(os.getenv("GRAPH_PATH_RAG_SEED_TOP_K", "120"))
GRAPH_PATH_RAG_BASE_WEIGHT = float(os.getenv("GRAPH_PATH_RAG_BASE_WEIGHT", "0.35"))
GRAPH_PATH_RAG_EVIDENCE_WEIGHT = float(os.getenv("GRAPH_PATH_RAG_EVIDENCE_WEIGHT", "1.00"))
GRAPH_PATH_RAG_DIVERSITY_WEIGHT = float(os.getenv("GRAPH_PATH_RAG_DIVERSITY_WEIGHT", "0.16"))
GRAPH_PATH_RERANK_WEIGHT = float(os.getenv("GRAPH_PATH_RERANK_WEIGHT", "0.16"))
GRAPH_PPR_STEPS = int(os.getenv("GRAPH_PPR_STEPS", "3"))
GRAPH_PPR_ALPHA = float(os.getenv("GRAPH_PPR_ALPHA", "0.82"))
GRAPH_PPR_FRONTIER_TOP_K = int(os.getenv("GRAPH_PPR_FRONTIER_TOP_K", "2500"))
GRAPH_PPR_CANDIDATE_TOP_K = int(os.getenv("GRAPH_PPR_CANDIDATE_TOP_K", "1000"))
GRAPH_PPR_MAX_CITATION_DEGREE = int(os.getenv("GRAPH_PPR_MAX_CITATION_DEGREE", "900"))
GRAPH_PPR_MAX_ENTITY_DEGREE = int(os.getenv("GRAPH_PPR_MAX_ENTITY_DEGREE", "600"))
GRAPH_PPR_QUERY_WEIGHT = float(os.getenv("GRAPH_PPR_QUERY_WEIGHT", "1.15"))
GRAPH_PPR_PATH_WEIGHT = float(os.getenv("GRAPH_PPR_PATH_WEIGHT", "0.75"))
GRAPH_PPR_BACKFILL_PRESERVE_TOP = int(os.getenv("GRAPH_PPR_BACKFILL_PRESERVE_TOP", "20"))
GRAPH_PPR_BACKFILL_UNTIL = int(os.getenv("GRAPH_PPR_BACKFILL_UNTIL", "100"))
GRAPH_PPR_RERANK_WEIGHT = float(os.getenv("GRAPH_PPR_RERANK_WEIGHT", "0.10"))
GRAPH_CONFIDENT_EXPAND_PRESERVE_TOP = int(os.getenv("GRAPH_CONFIDENT_EXPAND_PRESERVE_TOP", "50"))
GRAPH_CONFIDENT_EXPAND_UNTIL = int(os.getenv("GRAPH_CONFIDENT_EXPAND_UNTIL", "100"))
GRAPH_CONFIDENT_GRAPH_TOP_K = int(os.getenv("GRAPH_CONFIDENT_GRAPH_TOP_K", "200"))
GRAPH_CONFIDENT_DIRECT_TOP_K = int(os.getenv("GRAPH_CONFIDENT_DIRECT_TOP_K", "15"))
GRAPH_CONFIDENT_MIN_SOURCES = int(os.getenv("GRAPH_CONFIDENT_MIN_SOURCES", "2"))
GRAPH_CONFIDENT_MAX_INSERTS = int(os.getenv("GRAPH_CONFIDENT_MAX_INSERTS", "25"))
GRAPH_RESCORER_SEED_TOP_K = int(os.getenv("GRAPH_RESCORER_SEED_TOP_K", "150"))
GRAPH_RESCORER_EXPAND_TOP_K = int(os.getenv("GRAPH_RESCORER_EXPAND_TOP_K", "700"))
GRAPH_RESCORER_PRESERVE_TOP = int(os.getenv("GRAPH_RESCORER_PRESERVE_TOP", "10"))
GRAPH_RESCORER_BASE_WEIGHT = float(os.getenv("GRAPH_RESCORER_BASE_WEIGHT", "1.00"))
GRAPH_RESCORER_GRAPH_WEIGHT = float(os.getenv("GRAPH_RESCORER_GRAPH_WEIGHT", "0.18"))
GRAPH_RESCORER_EVIDENCE_WEIGHT = float(os.getenv("GRAPH_RESCORER_EVIDENCE_WEIGHT", "0.08"))
GRAPH_RESCORER_NEW_DOC_WEIGHT = float(os.getenv("GRAPH_RESCORER_NEW_DOC_WEIGHT", "0.025"))
GRAPH_RANK_RESCORER_GRAPH_WEIGHT = float(os.getenv("GRAPH_RANK_RESCORER_GRAPH_WEIGHT", "0.22"))
GRAPH_RANK_RESCORER_EVIDENCE_WEIGHT = float(os.getenv("GRAPH_RANK_RESCORER_EVIDENCE_WEIGHT", "0.10"))
GRAPH_RANK_RESCORER_NEW_DOC_WEIGHT = float(os.getenv("GRAPH_RANK_RESCORER_NEW_DOC_WEIGHT", "0.015"))
GRAPH_RANK_RESCORER_EVIDENCE_GATE_FLOOR = float(os.getenv("GRAPH_RANK_RESCORER_EVIDENCE_GATE_FLOOR", "0.25"))
GRAPH_RANK_RESCORER_VARIANTS = [
    ("hybrid_graph_rank_rescore_g030_e015_gate010", 0.30, 0.15, 0.015, 0.10),
    ("hybrid_graph_rank_rescore_g040_e020_gate010", 0.40, 0.20, 0.015, 0.10),
    ("hybrid_graph_rank_rescore_g030_e020_gate000", 0.30, 0.20, 0.015, 0.00),
    ("hybrid_graph_rank_rescore_g050_e020_gate010", 0.50, 0.20, 0.015, 0.10),
]
GRAPH_MAX_CITATION_DEGREE = int(os.getenv("GRAPH_MAX_CITATION_DEGREE", "1500"))
GRAPH_MAX_IPC_DEGREE = int(os.getenv("GRAPH_MAX_IPC_DEGREE", "800"))
GRAPH_MAX_IPC_MAIN_DEGREE = int(os.getenv("GRAPH_MAX_IPC_MAIN_DEGREE", "1500"))
GRAPH_MAX_IPC_SUBCLASS_DEGREE = int(os.getenv("GRAPH_MAX_IPC_SUBCLASS_DEGREE", "3000"))
GRAPH_MAX_IPC_CLASS_DEGREE = int(os.getenv("GRAPH_MAX_IPC_CLASS_DEGREE", "5000"))
GRAPH_MAX_IPC_SECTION_DEGREE = int(os.getenv("GRAPH_MAX_IPC_SECTION_DEGREE", "0"))
GRAPH_MAX_ASSIGNEE_DEGREE = int(os.getenv("GRAPH_MAX_ASSIGNEE_DEGREE", "400"))
GRAPH_MAX_INVENTOR_DEGREE = int(os.getenv("GRAPH_MAX_INVENTOR_DEGREE", "250"))

GRAPH_METHODS = [
    "graph_query",
    "graph_only_rag",
    "graph_only_balanced",
    "graph_path_rag",
    "graph_ppr",
    "graph_ppr_rag",
    "graph_expanded",
    "graph_rag_expanded",
    "graph_generation_pool",
]
HYBRID_GRAPH_METHODS = [
    "hybrid_graph_adaptive",
    "hybrid_graph_safe_rerank",
    "hybrid_graph_path_rerank",
    "hybrid_graph_confident_expand",
    "hybrid_graph_rescore_expand",
    "hybrid_graph_rank_rescore_expand",
    "hybrid_graph_rank_rescore_g030_e015_gate010",
    "hybrid_graph_rank_rescore_g040_e020_gate010",
    "hybrid_graph_rank_rescore_g030_e020_gate000",
    "hybrid_graph_rank_rescore_g050_e020_gate010",
    "hybrid_graph_ppr_rerank",
    "hybrid_graph_ppr_backfill",
    "hybrid_graph_hybrid_rag",
    "hybrid_graph_rag_backfill_top20",
    "hybrid_graph_rag_backfill_top50",
]
GRAPH_EVAL_METHODS = GRAPH_METHODS + HYBRID_GRAPH_METHODS

GRAPH_RANKINGS_OUT = OUT_DIR / "retrieval_rankings_graph_methods.parquet"
GRAPH_TOPIC_METRICS_OUT = OUT_DIR / "retrieval_topic_metrics_with_graph.parquet"
GRAPH_SUMMARY_OUT = OUT_DIR / "retrieval_summary_with_graph.csv"
GRAPH_SUMMARY_JSON_OUT = OUT_DIR / "retrieval_summary_with_graph.json"


def graph_is_nan_like(x: Any) -> bool:
    try:
        return isinstance(x, float) and np.isnan(x)
    except Exception:
        return False


def graph_safe_str(x: Any) -> str:
    if x is None or graph_is_nan_like(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def graph_safe_list(x: Any) -> List[Any]:
    if x is None or graph_is_nan_like(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, tuple):
        x = list(x)
    if isinstance(x, list):
        return x
    return [x]


def graph_norm_text(value):
    return graph_safe_str(value)


def graph_norm_doc_id(value):
    return graph_safe_str(value).upper().replace("_", "-")


def graph_norm_ipc(value):
    return re.sub(r"\s+", "", graph_safe_str(value).upper())


def graph_ipc_level_keys(value):
    ipc = graph_norm_ipc(value)
    if not ipc:
        return []

    keys = []
    # Full key keeps the most specific notation available in the shard.
    keys.append(f"IPC_FULL:{ipc}")

    compact = ipc.replace("/", "")
    section = compact[:1] if re.match(r"^[A-Z]", compact) else ""
    ipc_class = compact[:3] if re.match(r"^[A-Z][0-9]{2}", compact) else ""
    subclass = compact[:4] if re.match(r"^[A-Z][0-9]{2}[A-Z]", compact) else ""

    main_group = ""
    if "/" in ipc:
        main_group = ipc.split("/", 1)[0]
    else:
        m = re.match(r"^([A-Z][0-9]{2}[A-Z][0-9]+)", ipc)
        if m:
            main_group = m.group(1)

    if main_group:
        keys.append(f"IPC_MAIN:{main_group}")
    if subclass:
        keys.append(f"IPC_SUBCLASS:{subclass}")
    if ipc_class:
        keys.append(f"IPC_CLASS:{ipc_class}")
    if section:
        keys.append(f"IPC_SECTION:{section}")

    seen = set()
    return [key for key in keys if key and not (key in seen or seen.add(key))]


def graph_norm_name(value):
    return graph_safe_str(value).upper().strip(" ,;|")


def graph_date_key(value):
    digits = re.sub(r"\D", "", graph_safe_str(value))
    return digits[:8]


def graph_rank_weight(rank):
    return 1.0 / math.log2(float(rank) + 2.0)


def graph_degree_weight(degree):
    return 1.0 / math.log2(float(max(2, degree)) + 1.0)


def parse_json_object(value, default=None):
    if default is None:
        default = {}
    if isinstance(value, dict):
        return value
    try:
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return default
        text = str(value).strip()
        return json.loads(text) if text else default
    except Exception:
        return default


def graph_parquet_ready(graph_dir: Path) -> bool:
    required = ["edges_citation", "edges_inventor", "edges_assignee", "edges_ipc"]
    return graph_dir.exists() and all(any((graph_dir / name).glob("*.parquet")) for name in required)


def graph_build_edges(df: pd.DataFrame, src_col: str, list_col: str, edge_type: str, dst_normalizer) -> pd.DataFrame:
    if src_col not in df.columns or list_col not in df.columns:
        return pd.DataFrame(columns=["src", "dst", "type"])
    tmp = df[[src_col, list_col]].copy()
    tmp[src_col] = tmp[src_col].apply(graph_norm_doc_id)
    tmp[list_col] = tmp[list_col].apply(graph_safe_list)
    tmp = tmp[tmp[src_col] != ""].explode(list_col)
    if tmp.empty:
        return pd.DataFrame(columns=["src", "dst", "type"])
    tmp["dst"] = tmp[list_col].apply(dst_normalizer)
    tmp = tmp[tmp["dst"] != ""]
    edges = pd.DataFrame({
        "src": tmp[src_col].astype(str),
        "dst": tmp["dst"].astype(str),
        "type": edge_type,
    })
    return edges.drop_duplicates(subset=["src", "dst", "type"]).reset_index(drop=True)


def build_graph_parquet_if_needed():
    if graph_parquet_ready(GRAPH_DIR):
        print(f"[GRAPH] graph parquet exists: {GRAPH_DIR}")
        return
    if not GRAPH_BUILD_IF_MISSING:
        print(f"[GRAPH] missing graph parquet and GRAPH_BUILD_IF_MISSING=false: {GRAPH_DIR}")
        return
    if not GRAPH_SOURCE_PARQUET_DIR.exists():
        print(f"[GRAPH] source parquet dir missing: {GRAPH_SOURCE_PARQUET_DIR}")
        return

    dirs = {
        "edges_citation": GRAPH_DIR / "edges_citation",
        "edges_inventor": GRAPH_DIR / "edges_inventor",
        "edges_assignee": GRAPH_DIR / "edges_assignee",
        "edges_ipc": GRAPH_DIR / "edges_ipc",
        "logs": GRAPH_DIR / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    shards = sorted(GRAPH_SOURCE_PARQUET_DIR.glob("es_ready_part_*.parquet"))
    if not shards:
        shards = sorted(GRAPH_SOURCE_PARQUET_DIR.glob("*.parquet"))
    if not shards:
        print(f"[GRAPH] no parquet shards found in {GRAPH_SOURCE_PARQUET_DIR}")
        return

    print(f"[GRAPH] building graph parquet from {len(shards)} shards -> {GRAPH_DIR}")
    summary = {
        "source_parquet_dir": str(GRAPH_SOURCE_PARQUET_DIR),
        "graph_dir": str(GRAPH_DIR),
        "total_shards": len(shards),
        "total_edges_citation": 0,
        "total_edges_inventor": 0,
        "total_edges_assignee": 0,
        "total_edges_ipc": 0,
    }

    required_cols = ["doc_id", "citations", "inventors", "assignees", "ipc_codes"]
    for idx, shard in enumerate(tqdm(shards, desc="Building graph parquet")):
        df = pd.read_parquet(shard)
        for col in required_cols:
            if col not in df.columns:
                df[col] = None
        df = df[required_cols]

        citation_edges = graph_build_edges(df, "doc_id", "citations", "CITES", graph_norm_doc_id)
        inventor_edges = graph_build_edges(df, "doc_id", "inventors", "HAS_INVENTOR", graph_norm_name)
        assignee_edges = graph_build_edges(df, "doc_id", "assignees", "HAS_ASSIGNEE", graph_norm_name)
        ipc_edges = graph_build_edges(df, "doc_id", "ipc_codes", "HAS_IPC", graph_norm_ipc)

        citation_edges.to_parquet(dirs["edges_citation"] / f"edges_citation_{idx:05d}.parquet", index=False, compression="zstd")
        inventor_edges.to_parquet(dirs["edges_inventor"] / f"edges_inventor_{idx:05d}.parquet", index=False, compression="zstd")
        assignee_edges.to_parquet(dirs["edges_assignee"] / f"edges_assignee_{idx:05d}.parquet", index=False, compression="zstd")
        ipc_edges.to_parquet(dirs["edges_ipc"] / f"edges_ipc_{idx:05d}.parquet", index=False, compression="zstd")

        summary["total_edges_citation"] += int(len(citation_edges))
        summary["total_edges_inventor"] += int(len(inventor_edges))
        summary["total_edges_assignee"] += int(len(assignee_edges))
        summary["total_edges_ipc"] += int(len(ipc_edges))

        del df, citation_edges, inventor_edges, assignee_edges, ipc_edges
        if idx % 50 == 0:
            gc.collect()

    with open(dirs["logs"] / "graph_build_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print("[GRAPH] build summary")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


class GraphRetriever:
    def __init__(self, graph_dir, es=None, index_name=None, query_doc_ids=None):
        self.graph_dir = Path(graph_dir)
        self.available = bool(GRAPH_ENABLED and graph_parquet_ready(self.graph_dir))
        self.valid_doc_ids = set()
        self.canonical_to_doc_ids = defaultdict(list)
        self.query_doc_ids = {graph_norm_doc_id(x) for x in (query_doc_ids or []) if graph_norm_doc_id(x)}
        self.query_canonical_doc_ids = {canonical_doc_id(x) for x in self.query_doc_ids if canonical_doc_id(x)}
        self.doc_dates = {}
        self.citation_adj = defaultdict(set)
        self.citation_rev = defaultdict(set)
        self.ipc_adj = defaultdict(set)
        self.ipc_rev = defaultdict(set)
        self.assignee_adj = defaultdict(set)
        self.assignee_rev = defaultdict(set)
        self.inventor_adj = defaultdict(set)
        self.inventor_rev = defaultdict(set)
        if not self.available:
            print(f"[GRAPH] unavailable: {self.graph_dir}")
            return
        if GRAPH_FILTER_TO_BM25_INDEX and es is not None and index_name:
            self._load_index_catalog(es, index_name)
        self._load_graph_edges()
        print("[GRAPH] ready", json.dumps(self.stats(), ensure_ascii=False))

    def stats(self):
        return {
            "graph_dir": str(self.graph_dir),
            "available": self.available,
            "filter_to_bm25_index": GRAPH_FILTER_TO_BM25_INDEX,
            "valid_index_docs": len(self.valid_doc_ids),
            "query_doc_ids": len(self.query_doc_ids),
            "citation_src": len(self.citation_adj),
            "citation_dst": len(self.citation_rev),
            "ipc_nodes": len(self.ipc_rev),
            "assignee_nodes": len(self.assignee_rev),
            "inventor_nodes": len(self.inventor_rev),
            "seed_top_k": GRAPH_SEED_TOP_K,
            "with_citation_seed_top_k": GRAPH_WITH_CITATION_SEED_TOP_K,
            "no_citation_seed_top_k": GRAPH_NO_CITATION_SEED_TOP_K,
            "candidate_top_k": GRAPH_CANDIDATE_TOP_K,
            "adaptive_with_citation_weight": GRAPH_ADAPTIVE_WITH_CITATION_WEIGHT,
            "adaptive_no_citation_weight": GRAPH_ADAPTIVE_NO_CITATION_WEIGHT,
            "adaptive_with_citation_rrf_k": GRAPH_ADAPTIVE_WITH_CITATION_RRF_K,
            "adaptive_no_citation_rrf_k": GRAPH_ADAPTIVE_NO_CITATION_RRF_K,
            "generation_hybrid_keep": GRAPH_GENERATION_HYBRID_KEEP,
            "generation_graph_keep": GRAPH_GENERATION_GRAPH_KEEP,
            "rag_second_hop_top_k": GRAPH_RAG_SECOND_HOP_TOP_K,
            "rag_backfill_top_k": GRAPH_RAG_BACKFILL_TOP_K,
            "rag_preserve_top_20": GRAPH_RAG_PRESERVE_TOP_20,
            "rag_preserve_top_50": GRAPH_RAG_PRESERVE_TOP_50,
            "graph_only_rag_seed_top_k": GRAPH_ONLY_RAG_SEED_TOP_K,
            "graph_only_rag_second_hop_top_k": GRAPH_ONLY_RAG_SECOND_HOP_TOP_K,
            "graph_only_rag_query_weight": GRAPH_ONLY_RAG_QUERY_WEIGHT,
            "graph_only_rag_hop1_weight": GRAPH_ONLY_RAG_HOP1_WEIGHT,
            "graph_only_rag_hop2_weight": GRAPH_ONLY_RAG_HOP2_WEIGHT,
            "graph_hybrid_rag_weight": GRAPH_HYBRID_RAG_WEIGHT,
            "graph_hybrid_rag_rrf_k": GRAPH_HYBRID_RAG_RRF_K,
            "graph_only_balanced_preserve_top": GRAPH_ONLY_BALANCED_PRESERVE_TOP,
            "graph_only_balanced_backfill_top": GRAPH_ONLY_BALANCED_BACKFILL_TOP,
            "graph_safe_rerank_preserve_top": GRAPH_SAFE_RERANK_PRESERVE_TOP,
            "graph_safe_rerank_top_k": GRAPH_SAFE_RERANK_TOP_K,
            "graph_safe_rerank_weight": GRAPH_SAFE_RERANK_WEIGHT,
            "graph_path_rag_candidate_top_k": GRAPH_PATH_RAG_CANDIDATE_TOP_K,
            "graph_path_rag_seed_top_k": GRAPH_PATH_RAG_SEED_TOP_K,
            "graph_path_rag_base_weight": GRAPH_PATH_RAG_BASE_WEIGHT,
            "graph_path_rag_evidence_weight": GRAPH_PATH_RAG_EVIDENCE_WEIGHT,
            "graph_path_rag_diversity_weight": GRAPH_PATH_RAG_DIVERSITY_WEIGHT,
            "graph_path_rerank_weight": GRAPH_PATH_RERANK_WEIGHT,
            "graph_ppr_steps": GRAPH_PPR_STEPS,
            "graph_ppr_alpha": GRAPH_PPR_ALPHA,
            "graph_ppr_frontier_top_k": GRAPH_PPR_FRONTIER_TOP_K,
            "graph_ppr_candidate_top_k": GRAPH_PPR_CANDIDATE_TOP_K,
            "graph_ppr_max_citation_degree": GRAPH_PPR_MAX_CITATION_DEGREE,
            "graph_ppr_max_entity_degree": GRAPH_PPR_MAX_ENTITY_DEGREE,
            "graph_ppr_query_weight": GRAPH_PPR_QUERY_WEIGHT,
            "graph_ppr_path_weight": GRAPH_PPR_PATH_WEIGHT,
            "graph_ppr_backfill_preserve_top": GRAPH_PPR_BACKFILL_PRESERVE_TOP,
            "graph_ppr_backfill_until": GRAPH_PPR_BACKFILL_UNTIL,
            "graph_ppr_rerank_weight": GRAPH_PPR_RERANK_WEIGHT,
            "graph_confident_expand_preserve_top": GRAPH_CONFIDENT_EXPAND_PRESERVE_TOP,
            "graph_confident_expand_until": GRAPH_CONFIDENT_EXPAND_UNTIL,
            "graph_confident_graph_top_k": GRAPH_CONFIDENT_GRAPH_TOP_K,
            "graph_confident_direct_top_k": GRAPH_CONFIDENT_DIRECT_TOP_K,
            "graph_confident_min_sources": GRAPH_CONFIDENT_MIN_SOURCES,
            "graph_confident_max_inserts": GRAPH_CONFIDENT_MAX_INSERTS,
            "graph_rescorer_seed_top_k": GRAPH_RESCORER_SEED_TOP_K,
            "graph_rescorer_expand_top_k": GRAPH_RESCORER_EXPAND_TOP_K,
            "graph_rescorer_preserve_top": GRAPH_RESCORER_PRESERVE_TOP,
            "graph_rescorer_base_weight": GRAPH_RESCORER_BASE_WEIGHT,
            "graph_rescorer_graph_weight": GRAPH_RESCORER_GRAPH_WEIGHT,
            "graph_rescorer_evidence_weight": GRAPH_RESCORER_EVIDENCE_WEIGHT,
            "graph_rescorer_new_doc_weight": GRAPH_RESCORER_NEW_DOC_WEIGHT,
            "graph_rank_rescorer_graph_weight": GRAPH_RANK_RESCORER_GRAPH_WEIGHT,
            "graph_rank_rescorer_evidence_weight": GRAPH_RANK_RESCORER_EVIDENCE_WEIGHT,
            "graph_rank_rescorer_new_doc_weight": GRAPH_RANK_RESCORER_NEW_DOC_WEIGHT,
            "graph_rank_rescorer_evidence_gate_floor": GRAPH_RANK_RESCORER_EVIDENCE_GATE_FLOOR,
        }

    def _load_index_catalog(self, es, index_name):
        query = {"query": {"match_all": {}}, "_source": ["doc_id", "canonical_doc_id", "publication_date"]}
        for hit in helpers.scan(es.options(request_timeout=180), index=index_name, query=query, size=1000, scroll="10m", preserve_order=False):
            src = hit.get("_source", {}) or {}
            doc_id = graph_norm_doc_id(src.get("doc_id") or hit.get("_id"))
            if not doc_id:
                continue
            canonical_id = canonical_doc_id(src.get("canonical_doc_id") or doc_id)
            self.valid_doc_ids.add(doc_id)
            if canonical_id:
                self.canonical_to_doc_ids[canonical_id].append(doc_id)
            pub_date = graph_date_key(src.get("publication_date"))
            if pub_date:
                self.doc_dates[doc_id] = pub_date

    def _load_edge_dir(self, edge_dir, handler):
        for file_path in sorted(Path(edge_dir).glob("*.parquet")):
            try:
                df = pd.read_parquet(file_path, columns=["src", "dst"])
            except Exception:
                df = pd.read_parquet(file_path)
                if not {"src", "dst"}.issubset(df.columns):
                    continue
                df = df[["src", "dst"]]
            for row in df.itertuples(index=False):
                handler(row.src, row.dst)
            del df

    def _src_in_index_scope(self, src):
        src = graph_norm_doc_id(src)
        src_canonical = canonical_doc_id(src)
        if src in self.query_doc_ids or src_canonical in self.query_canonical_doc_ids:
            return True
        if not self.valid_doc_ids:
            return True
        if src in self.valid_doc_ids:
            return True
        return src_canonical in self.canonical_to_doc_ids

    def _add_doc_keyed_edge(self, forward, reverse, src, dst):
        src = graph_norm_doc_id(src)
        dst = graph_norm_text(dst)
        if not src or not dst or not self._src_in_index_scope(src):
            return
        src_canonical = canonical_doc_id(src)
        forward[src].add(dst)
        if src_canonical and src_canonical != src:
            forward[src_canonical].add(dst)
        reverse[dst].add(src)

    def _add_citation_edge(self, src, dst):
        src = graph_norm_doc_id(src)
        dst = graph_norm_doc_id(dst)
        if not src or not dst or not self._src_in_index_scope(src):
            return
        src_canonical = canonical_doc_id(src)
        dst_canonical = canonical_doc_id(dst)
        self.citation_adj[src].add(dst)
        if src_canonical and src_canonical != src:
            self.citation_adj[src_canonical].add(dst)
        self.citation_rev[dst].add(src)
        if dst_canonical and dst_canonical != dst:
            self.citation_rev[dst_canonical].add(src)

    def _add_ipc_edge(self, src, dst):
        src = graph_norm_doc_id(src)
        if not src or not self._src_in_index_scope(src):
            return
        src_canonical = canonical_doc_id(src)
        for ipc_key in graph_ipc_level_keys(dst):
            self.ipc_adj[src].add(ipc_key)
            if src_canonical and src_canonical != src:
                self.ipc_adj[src_canonical].add(ipc_key)
            self.ipc_rev[ipc_key].add(src)

    def _load_graph_edges(self):
        self._load_edge_dir(self.graph_dir / "edges_citation", self._add_citation_edge)
        self._load_edge_dir(self.graph_dir / "edges_ipc", self._add_ipc_edge)
        self._load_edge_dir(self.graph_dir / "edges_assignee", lambda src, dst: self._add_doc_keyed_edge(self.assignee_adj, self.assignee_rev, src, graph_norm_name(dst)))
        self._load_edge_dir(self.graph_dir / "edges_inventor", lambda src, dst: self._add_doc_keyed_edge(self.inventor_adj, self.inventor_rev, src, graph_norm_name(dst)))

    def resolve_index_doc_ids(self, value, max_variants=3):
        doc_id = graph_norm_doc_id(value)
        if not doc_id:
            return []
        if self.valid_doc_ids:
            if doc_id in self.valid_doc_ids:
                return [doc_id]
            return self.canonical_to_doc_ids.get(canonical_doc_id(doc_id), [])[:max_variants]
        return [doc_id]

    def candidate_allowed(self, doc_id, plan):
        doc_id = graph_norm_doc_id(doc_id)
        if not doc_id:
            return False
        if self.valid_doc_ids and doc_id not in self.valid_doc_ids:
            return False
        filter_conf = parse_json_object(plan.get("filter_config_json"), {})
        excluded = canonical_doc_id(filter_conf.get("exclude_canonical_doc_id") or filter_conf.get("exclude_doc_id"))
        if excluded and canonical_doc_id(doc_id) == excluded:
            return False
        if filter_conf.get("use_date_filter") and filter_conf.get("prior_art_cutoff_date"):
            cutoff = graph_date_key(filter_conf.get("prior_art_cutoff_date"))
            doc_date = self.doc_dates.get(doc_id)
            if cutoff and doc_date and not (doc_date < cutoff):
                return False
        return True

    def _add_candidate_score(self, scores, raw_doc_id, score, plan):
        for doc_id in self.resolve_index_doc_ids(raw_doc_id):
            if self.candidate_allowed(doc_id, plan):
                scores[doc_id] += float(score)

    def _add_entity_neighbors(self, scores, entity_value, reverse, max_degree, edge_weight, plan):
        entity = graph_norm_text(entity_value)
        if not entity:
            return
        neighbors = reverse.get(entity, set())
        degree = len(neighbors)
        if degree <= 0 or degree > max_degree:
            return
        score = float(edge_weight) * graph_degree_weight(degree)
        for raw_doc_id in neighbors:
            self._add_candidate_score(scores, raw_doc_id, score, plan)

    def _ipc_degree_cap(self, ipc_key):
        if ipc_key.startswith("IPC_FULL:"):
            return GRAPH_MAX_IPC_DEGREE
        if ipc_key.startswith("IPC_MAIN:"):
            return GRAPH_MAX_IPC_MAIN_DEGREE
        if ipc_key.startswith("IPC_SUBCLASS:"):
            return GRAPH_MAX_IPC_SUBCLASS_DEGREE
        if ipc_key.startswith("IPC_CLASS:"):
            return GRAPH_MAX_IPC_CLASS_DEGREE
        if ipc_key.startswith("IPC_SECTION:"):
            return GRAPH_MAX_IPC_SECTION_DEGREE
        return GRAPH_MAX_IPC_DEGREE

    def _ipc_level_weight(self, ipc_key):
        if ipc_key.startswith("IPC_FULL:"):
            return 0.42
        if ipc_key.startswith("IPC_MAIN:"):
            return 0.28
        if ipc_key.startswith("IPC_SUBCLASS:"):
            return 0.14
        if ipc_key.startswith("IPC_CLASS:"):
            return 0.06
        if ipc_key.startswith("IPC_SECTION:"):
            return 0.02
        return 0.10

    def _plan_ipc_seed_specs(self, plan, boost_conf):
        raw_values = []
        ipc_fields = boost_conf.get("ipc_boost_fields") or {}
        raw_values.extend(ipc_fields.get("ipc_codes_norm") or [])
        for key in ["ipc_group", "ipc_main_group", "ipc_subclass", "ipc_class", "ipc_section"]:
            value = plan.get(key) or ipc_fields.get(key)
            if value:
                raw_values.append(value)

        specs = []
        seen = set()
        for value in raw_values:
            for ipc_key in graph_ipc_level_keys(value):
                if ipc_key in seen:
                    continue
                seen.add(ipc_key)
                cap = self._ipc_degree_cap(ipc_key)
                if cap <= 0:
                    continue
                specs.append((ipc_key, self._ipc_level_weight(ipc_key), cap))
        return specs

    def _plan_ipc_seeds(self, plan, boost_conf):
        return [ipc_key for ipc_key, _, _ in self._plan_ipc_seed_specs(plan, boost_conf)]

    def _query_doc_keys(self, plan):
        keys = []
        for value in [plan.get("query_doc_id"), plan.get("topic_id")]:
            doc_id = graph_norm_doc_id(value)
            canonical_id = canonical_doc_id(doc_id)
            if doc_id:
                keys.append(doc_id)
            if canonical_id:
                keys.append(canonical_id)
        seen = set()
        return [x for x in keys if x and not (x in seen or seen.add(x))]

    def inject_query_plan_edges(self, plan, boost_conf=None):
        """Add the query patent as an ephemeral graph node from extracted metadata.

        Benchmark query patents are intentionally not in the candidate index. They may also be
        absent from graph_parquet. This injects query -> citation/IPC/assignee/inventor edges
        at runtime so graph-only can still traverse from the query node to indexed candidates.
        """
        boost_conf = boost_conf or parse_json_object(plan.get("boost_config_json"), {})
        query_keys = self._query_doc_keys(plan)
        if not query_keys:
            return {"citations": 0, "ipc": 0, "assignees": 0, "inventors": 0}

        citation_ids = []
        for citation in boost_conf.get("query_citation_seeds") or []:
            citation_id = graph_norm_doc_id(citation)
            if citation_id:
                citation_ids.append(citation_id)

        ipc_ids = self._plan_ipc_seeds(plan, boost_conf)
        assignees = [graph_norm_name(x) for x in (boost_conf.get("assignees_clean") or []) if graph_norm_name(x)]
        inventors = [graph_norm_name(x) for x in (boost_conf.get("inventors_clean") or []) if graph_norm_name(x)]

        for key in query_keys:
            for citation_id in citation_ids:
                self.citation_adj[key].add(citation_id)
                self.citation_rev[citation_id].add(key)
                citation_canonical = canonical_doc_id(citation_id)
                if citation_canonical and citation_canonical != citation_id:
                    self.citation_rev[citation_canonical].add(key)
            for ipc in ipc_ids:
                self.ipc_adj[key].add(ipc)
                self.ipc_rev[ipc].add(key)
            for assignee in assignees:
                self.assignee_adj[key].add(assignee)
                self.assignee_rev[assignee].add(key)
            for inventor in inventors:
                self.inventor_adj[key].add(inventor)
                self.inventor_rev[inventor].add(key)

        return {
            "citations": len(set(citation_ids)),
            "ipc": len(set(ipc_ids)),
            "assignees": len(set(assignees)),
            "inventors": len(set(inventors)),
        }

    def rank_query_graph(self, plan, topic_views=None, top_k=None):
        if not self.available:
            return []
        top_k = int(top_k or GRAPH_CANDIDATE_TOP_K)
        boost_conf = parse_json_object(plan.get("boost_config_json"), {})
        self.inject_query_plan_edges(plan, boost_conf)
        scores = defaultdict(float)

        # Direct graph evidence from the injected/query patent node. This is the real graph-only signal.
        for key in self._query_doc_keys(plan):
            for cited_doc_id in self.citation_adj.get(key, set()):
                self._add_candidate_score(scores, cited_doc_id, 1.60, plan)
            reverse_neighbors = self.citation_rev.get(key, set())
            degree = len(reverse_neighbors)
            if 0 < degree <= GRAPH_MAX_CITATION_DEGREE:
                for raw_doc_id in reverse_neighbors:
                    self._add_candidate_score(scores, raw_doc_id, 0.12 * graph_degree_weight(degree), plan)
            for ipc in self.ipc_adj.get(key, set()):
                self._add_entity_neighbors(scores, ipc, self.ipc_rev, self._ipc_degree_cap(ipc), 0.35 * self._ipc_level_weight(ipc), plan)
            for assignee in self.assignee_adj.get(key, set()):
                self._add_entity_neighbors(scores, assignee, self.assignee_rev, GRAPH_MAX_ASSIGNEE_DEGREE, 0.18, plan)
            for inventor in self.inventor_adj.get(key, set()):
                self._add_entity_neighbors(scores, inventor, self.inventor_rev, GRAPH_MAX_INVENTOR_DEGREE, 0.22, plan)

        # Metadata fallback, useful when the query doc itself is not present in the graph shards.
        for citation in boost_conf.get("query_citation_seeds") or []:
            citation_id = graph_norm_doc_id(citation)
            if not citation_id:
                continue
            self._add_candidate_score(scores, citation_id, 1.20, plan)
            for key in {citation_id, canonical_doc_id(citation_id)}:
                neighbors = self.citation_rev.get(key, set())
                degree = len(neighbors)
                if degree <= 0 or degree > GRAPH_MAX_CITATION_DEGREE:
                    continue
                for raw_doc_id in neighbors:
                    self._add_candidate_score(scores, raw_doc_id, 0.20 * graph_degree_weight(degree), plan)
        for ipc_key, ipc_weight, ipc_cap in self._plan_ipc_seed_specs(plan, boost_conf):
            self._add_entity_neighbors(scores, ipc_key, self.ipc_rev, ipc_cap, ipc_weight, plan)
        for assignee in boost_conf.get("assignees_clean") or []:
            self._add_entity_neighbors(scores, graph_norm_name(assignee), self.assignee_rev, GRAPH_MAX_ASSIGNEE_DEGREE, 0.12, plan)
        for inventor in boost_conf.get("inventors_clean") or []:
            self._add_entity_neighbors(scores, graph_norm_name(inventor), self.inventor_rev, GRAPH_MAX_INVENTOR_DEGREE, 0.16, plan)
        ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
        return [(doc_id, float(score)) for doc_id, score in ranked[:top_k]]

    def _doc_key_set(self, doc_id):
        doc_id = graph_norm_doc_id(doc_id)
        canonical_id = canonical_doc_id(doc_id)
        return {key for key in [doc_id, canonical_id] if key}

    def _doc_entity_sets(self, doc_id):
        ipcs, assignees, inventors, citations = set(), set(), set(), set()
        for key in self._doc_key_set(doc_id):
            ipcs.update(self.ipc_adj.get(key, set()))
            assignees.update(self.assignee_adj.get(key, set()))
            inventors.update(self.inventor_adj.get(key, set()))
            citations.update(self.citation_adj.get(key, set()))
        return {
            "ipcs": ipcs,
            "assignees": assignees,
            "inventors": inventors,
            "citations": citations,
        }

    def _candidate_path_evidence_score(self, doc_id, plan, boost_conf):
        doc_id = graph_norm_doc_id(doc_id)
        canonical_id = canonical_doc_id(doc_id)
        if not doc_id:
            return 0.0, []

        entities = self._doc_entity_sets(doc_id)
        evidence = []
        score = 0.0

        query_citations = {
            graph_norm_doc_id(x)
            for x in graph_safe_list(boost_conf.get("query_citation_seeds"))
            if graph_norm_doc_id(x)
        }
        query_citation_canonicals = {canonical_doc_id(x) for x in query_citations if canonical_doc_id(x)}
        if canonical_id and canonical_id in query_citation_canonicals:
            score += 3.00
            evidence.append("direct_query_citation")

        shared_citations = {
            canonical_doc_id(x)
            for x in entities["citations"]
            if canonical_doc_id(x) in query_citation_canonicals
        }
        if shared_citations:
            score += min(1.50, 0.55 * len(shared_citations))
            evidence.append("bibliographic_coupling")

        for key in self._doc_key_set(doc_id):
            for citation in query_citations:
                citation_keys = {citation, canonical_doc_id(citation)}
                if self.citation_adj.get(citation, set()) & {key, canonical_doc_id(key)}:
                    score += 0.70
                    evidence.append("query_citation_points_to_candidate")
                    break
                if self.citation_rev.get(citation, set()) & {key, canonical_doc_id(key)}:
                    score += 0.35
                    evidence.append("candidate_cites_query_citation")
                    break

        query_ipc_specs = self._plan_ipc_seed_specs(plan, boost_conf)
        query_ipcs = {ipc_key for ipc_key, _weight, _cap in query_ipc_specs}
        shared_ipcs = entities["ipcs"] & query_ipcs
        if shared_ipcs:
            ipc_score = 0.0
            for ipc_key in shared_ipcs:
                ipc_score += 0.90 * self._ipc_level_weight(ipc_key) * graph_degree_weight(len(self.ipc_rev.get(ipc_key, set())))
            score += min(1.20, ipc_score)
            evidence.append("shared_ipc")

        query_assignees = {graph_norm_name(x) for x in graph_safe_list(boost_conf.get("assignees_clean")) if graph_norm_name(x)}
        shared_assignees = entities["assignees"] & query_assignees
        if shared_assignees:
            assignee_score = 0.0
            for assignee in shared_assignees:
                degree = len(self.assignee_rev.get(assignee, set()))
                if 0 < degree <= GRAPH_MAX_ASSIGNEE_DEGREE:
                    assignee_score += 0.45 * graph_degree_weight(degree)
            score += min(0.75, assignee_score)
            evidence.append("same_assignee")

        query_inventors = {graph_norm_name(x) for x in graph_safe_list(boost_conf.get("inventors_clean")) if graph_norm_name(x)}
        shared_inventors = entities["inventors"] & query_inventors
        if shared_inventors:
            inventor_score = 0.0
            for inventor in shared_inventors:
                degree = len(self.inventor_rev.get(inventor, set()))
                if 0 < degree <= GRAPH_MAX_INVENTOR_DEGREE:
                    inventor_score += 0.55 * graph_degree_weight(degree)
            score += min(0.80, inventor_score)
            evidence.append("same_inventor")

        evidence_diversity = len(set(evidence))
        if evidence_diversity:
            score += GRAPH_PATH_RAG_DIVERSITY_WEIGHT * min(5, evidence_diversity)
        return float(score), evidence

    def rank_graph_path_rag(self, plan, top_k=None, candidate_top_k=None):
        if not self.available:
            return []
        top_k = int(top_k or GRAPH_CANDIDATE_TOP_K)
        candidate_top_k = int(candidate_top_k or GRAPH_PATH_RAG_CANDIDATE_TOP_K)
        boost_conf = parse_json_object(plan.get("boost_config_json"), {})
        query_graph = self.rank_query_graph(plan, top_k=candidate_top_k)
        hop1 = self.rank_hybrid_expansion(
            query_graph,
            plan,
            top_k=candidate_top_k,
            seed_top_k=GRAPH_PATH_RAG_SEED_TOP_K,
        )
        hop2 = self.rank_hybrid_expansion(
            hop1,
            plan,
            top_k=candidate_top_k,
            seed_top_k=max(20, GRAPH_PATH_RAG_SEED_TOP_K // 2),
        )

        base_scores = defaultdict(float)
        for source_weight, ranked in [(1.00, query_graph), (0.70, hop1), (0.35, hop2)]:
            for rank, (doc_id, score) in enumerate(ranked, start=1):
                canonical_id = canonical_doc_id(doc_id)
                if not canonical_id:
                    continue
                base_scores[canonical_id] += source_weight * graph_rank_weight(rank) * max(0.05, float(score))

        rows = []
        seen_doc_for_canonical = {}
        for ranked in [query_graph, hop1, hop2]:
            for doc_id, _score in ranked:
                canonical_id = canonical_doc_id(doc_id)
                if canonical_id and canonical_id not in seen_doc_for_canonical:
                    seen_doc_for_canonical[canonical_id] = str(doc_id)

        for canonical_id, base_score in base_scores.items():
            doc_id = seen_doc_for_canonical.get(canonical_id, canonical_id)
            evidence_score, evidence = self._candidate_path_evidence_score(doc_id, plan, boost_conf)
            final_score = GRAPH_PATH_RAG_BASE_WEIGHT * float(base_score) + GRAPH_PATH_RAG_EVIDENCE_WEIGHT * float(evidence_score)
            if final_score <= 0:
                continue
            rows.append((doc_id, final_score, len(set(evidence))))

        rows.sort(key=lambda item: (-item[1], -item[2], item[0]))
        return [(doc_id, float(score)) for doc_id, score, _evidence_count in rows[:top_k]]

    def _ppr_seed_nodes(self, plan, boost_conf):
        seeds = defaultdict(float)
        self.inject_query_plan_edges(plan, boost_conf)
        for key in self._query_doc_keys(plan):
            seeds[('doc', key)] += 3.00
        for citation in graph_safe_list(boost_conf.get('query_citation_seeds')):
            citation_id = graph_norm_doc_id(citation)
            if citation_id:
                seeds[('doc', citation_id)] += 2.40
                canonical_id = canonical_doc_id(citation_id)
                if canonical_id:
                    seeds[('doc', canonical_id)] += 1.20
        for ipc_key, ipc_weight, _cap in self._plan_ipc_seed_specs(plan, boost_conf):
            seeds[('ipc', ipc_key)] += 1.15 * float(ipc_weight)
        for assignee in graph_safe_list(boost_conf.get('assignees_clean')):
            name = graph_norm_name(assignee)
            if name:
                seeds[('assignee', name)] += 0.55
        for inventor in graph_safe_list(boost_conf.get('inventors_clean')):
            name = graph_norm_name(inventor)
            if name:
                seeds[('inventor', name)] += 0.65
        return seeds

    def _ppr_add_candidate_mass(self, doc_scores, raw_doc_id, mass, plan):
        for doc_id in self.resolve_index_doc_ids(raw_doc_id):
            if self.candidate_allowed(doc_id, plan):
                doc_scores[doc_id] += float(mass)

    def _ppr_doc_transitions(self, doc_id):
        transitions = []
        for key in self._doc_key_set(doc_id):
            cited_docs = self.citation_adj.get(key, set())
            if cited_docs:
                weight = 1.35 / max(1, len(cited_docs))
                transitions.extend((('doc', cited_doc_id), weight) for cited_doc_id in cited_docs)
            reverse_docs = self.citation_rev.get(key, set())
            degree = len(reverse_docs)
            if 0 < degree <= GRAPH_PPR_MAX_CITATION_DEGREE:
                weight = 0.45 * graph_degree_weight(degree) / max(1, degree)
                transitions.extend((('doc', raw_doc_id), weight) for raw_doc_id in reverse_docs)
            for ipc in self.ipc_adj.get(key, set()):
                transitions.append((('ipc', ipc), 0.95 * self._ipc_level_weight(ipc)))
            for assignee in self.assignee_adj.get(key, set()):
                transitions.append((('assignee', assignee), 0.55))
            for inventor in self.inventor_adj.get(key, set()):
                transitions.append((('inventor', inventor), 0.65))
        return transitions

    def _ppr_entity_transitions(self, node_type, value):
        if node_type == 'ipc':
            reverse = self.ipc_rev
            cap = min(self._ipc_degree_cap(value), GRAPH_PPR_MAX_ENTITY_DEGREE)
            base_weight = 0.80 * self._ipc_level_weight(value)
        elif node_type == 'assignee':
            reverse = self.assignee_rev
            cap = min(GRAPH_MAX_ASSIGNEE_DEGREE, GRAPH_PPR_MAX_ENTITY_DEGREE)
            base_weight = 0.45
        elif node_type == 'inventor':
            reverse = self.inventor_rev
            cap = min(GRAPH_MAX_INVENTOR_DEGREE, GRAPH_PPR_MAX_ENTITY_DEGREE)
            base_weight = 0.55
        else:
            return []
        neighbors = reverse.get(value, set())
        degree = len(neighbors)
        if degree <= 0 or degree > cap:
            return []
        weight = float(base_weight) * graph_degree_weight(degree) / max(1, degree)
        return [(('doc', raw_doc_id), weight) for raw_doc_id in neighbors]

    def rank_graph_ppr(self, plan, top_k=None, candidate_top_k=None):
        if not self.available:
            return []
        top_k = int(top_k or GRAPH_CANDIDATE_TOP_K)
        candidate_top_k = int(candidate_top_k or GRAPH_PPR_CANDIDATE_TOP_K)
        boost_conf = parse_json_object(plan.get('boost_config_json'), {})
        seed_nodes = self._ppr_seed_nodes(plan, boost_conf)
        total_seed = sum(seed_nodes.values())
        if total_seed <= 0:
            return []
        frontier = {node: float(score) / total_seed for node, score in seed_nodes.items()}
        doc_scores = defaultdict(float)
        query_keys = set(self._query_doc_keys(plan))
        query_canonicals = {canonical_doc_id(x) for x in query_keys if canonical_doc_id(x)}
        for step in range(max(1, GRAPH_PPR_STEPS)):
            next_frontier = defaultdict(float)
            for node, mass in frontier.items():
                node_type, value = node
                if node_type == 'doc':
                    if value not in query_keys and canonical_doc_id(value) not in query_canonicals:
                        self._ppr_add_candidate_mass(doc_scores, value, (1.0 - GRAPH_PPR_ALPHA) * mass * (1.0 + 0.20 * step), plan)
                    transitions = self._ppr_doc_transitions(value)
                else:
                    transitions = self._ppr_entity_transitions(node_type, value)
                weight_sum = sum(weight for _next_node, weight in transitions)
                if weight_sum <= 0:
                    continue
                for next_node, weight in transitions:
                    next_frontier[next_node] += GRAPH_PPR_ALPHA * mass * float(weight) / weight_sum
            if not next_frontier:
                break
            frontier = dict(sorted(next_frontier.items(), key=lambda item: -item[1])[:GRAPH_PPR_FRONTIER_TOP_K])

        rows = []
        for doc_id, ppr_score in doc_scores.items():
            evidence_score, evidence = self._candidate_path_evidence_score(doc_id, plan, boost_conf)
            final_score = float(ppr_score) + 0.18 * float(evidence_score)
            rows.append((doc_id, final_score, len(set(evidence))))
        rows.sort(key=lambda item: (-item[1], -item[2], item[0]))
        return [(doc_id, float(score)) for doc_id, score, _evidence_count in rows[:min(top_k, candidate_top_k)]]

    def rank_hybrid_expansion(self, hybrid_items, plan, top_k=None, seed_top_k=None):
        if not self.available or not hybrid_items:
            return []
        top_k = int(top_k or GRAPH_CANDIDATE_TOP_K)
        seed_top_k = int(seed_top_k or GRAPH_SEED_TOP_K)
        scores = defaultdict(float)
        seed_items = hybrid_items[:seed_top_k]
        for rank, (seed_doc_id, _) in enumerate(seed_items, start=1):
            seed_doc_id = graph_norm_doc_id(seed_doc_id)
            seed_keys = {seed_doc_id, canonical_doc_id(seed_doc_id)}
            seed_weight = graph_rank_weight(rank)
            for key in seed_keys:
                for cited_doc_id in self.citation_adj.get(key, set()):
                    self._add_candidate_score(scores, cited_doc_id, 1.00 * seed_weight, plan)
                reverse_neighbors = self.citation_rev.get(key, set())
                degree = len(reverse_neighbors)
                if 0 < degree <= GRAPH_MAX_CITATION_DEGREE:
                    for raw_doc_id in reverse_neighbors:
                        self._add_candidate_score(scores, raw_doc_id, 0.18 * seed_weight * graph_degree_weight(degree), plan)
                for ipc in self.ipc_adj.get(key, set()):
                    self._add_entity_neighbors(scores, ipc, self.ipc_rev, self._ipc_degree_cap(ipc), 0.08 * seed_weight * self._ipc_level_weight(ipc), plan)
                for assignee in self.assignee_adj.get(key, set()):
                    self._add_entity_neighbors(scores, assignee, self.assignee_rev, GRAPH_MAX_ASSIGNEE_DEGREE, 0.05 * seed_weight, plan)
                for inventor in self.inventor_adj.get(key, set()):
                    self._add_entity_neighbors(scores, inventor, self.inventor_rev, GRAPH_MAX_INVENTOR_DEGREE, 0.06 * seed_weight, plan)
        seed_doc_ids = {graph_norm_doc_id(doc_id) for doc_id, _ in seed_items}
        ranked = [(doc_id, score) for doc_id, score in sorted(scores.items(), key=lambda item: (-item[1], item[0])) if graph_norm_doc_id(doc_id) not in seed_doc_ids]
        return [(doc_id, float(score)) for doc_id, score in ranked[:top_k]]


def plan_has_query_citations(plan):
    boost_conf = parse_json_object(plan.get("boost_config_json"), {})
    if graph_safe_list(boost_conf.get("query_citation_seeds")):
        return True
    if graph_safe_list(plan.get("citations_raw")) or graph_safe_list(plan.get("citations")):
        return True
    try:
        return int(plan.get("num_citations") or 0) > 0
    except Exception:
        return False


def graph_adaptive_params(plan):
    has_citations = plan_has_query_citations(plan)
    if has_citations:
        return {
            "query_has_citations": True,
            "seed_top_k": GRAPH_WITH_CITATION_SEED_TOP_K,
            "graph_weight": GRAPH_ADAPTIVE_WITH_CITATION_WEIGHT,
            "rrf_k": GRAPH_ADAPTIVE_WITH_CITATION_RRF_K,
        }
    return {
        "query_has_citations": False,
        "seed_top_k": GRAPH_NO_CITATION_SEED_TOP_K,
        "graph_weight": GRAPH_ADAPTIVE_NO_CITATION_WEIGHT,
        "rrf_k": GRAPH_ADAPTIVE_NO_CITATION_RRF_K,
    }


def graph_fill_ranked_items(primary_ranked, fill_ranked, primary_keep=200, fill_keep=100, top_k=300, fill_score_offset=-10.0):
    merged = []
    seen = set()
    for doc_id, score in primary_ranked[:int(primary_keep)]:
        doc_id = str(doc_id)
        if doc_id in seen:
            continue
        seen.add(doc_id)
        merged.append((doc_id, float(score)))
    for doc_id, score in fill_ranked[:int(fill_keep)]:
        if len(merged) >= int(top_k):
            break
        doc_id = str(doc_id)
        if doc_id in seen:
            continue
        seen.add(doc_id)
        # Keep fill candidates behind preserved primary candidates while preserving fill order.
        merged.append((doc_id, float(score) + float(fill_score_offset)))
    return merged[:int(top_k)]


def graph_minmax_scores(ranked_items):
    if not ranked_items:
        return {}
    scores = []
    best = {}
    for rank, (doc_id, score) in enumerate(ranked_items, start=1):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id:
            continue
        score = float(score)
        current = best.get(canonical_id)
        if current is None or score > current["score"] or (score == current["score"] and rank < current["rank"]):
            best[canonical_id] = {"doc_id": str(doc_id), "score": score, "rank": rank}
            scores.append(score)
    if not best:
        return {}
    min_score = min(item["score"] for item in best.values())
    max_score = max(item["score"] for item in best.values())
    denom = max_score - min_score
    out = {}
    for canonical_id, item in best.items():
        norm = ((item["score"] - min_score) / denom) if denom > 1e-12 else (1.0 / (1.0 + item["rank"] - 1))
        out[canonical_id] = {"doc_id": item["doc_id"], "score": float(norm), "rank": int(item["rank"])}
    return out


def graph_safe_rerank_base_with_graph(base_ranked, graph_ranked, preserve_top=10, rerank_top_k=100, top_k=300, graph_weight=0.12):
    """Use graph as a weak rerank signal inside the base pool, not as a hard replacement.

    This is the safest graph retrieval mode for top-rank metrics: preserve the strongest base prefix,
    rerank only the next window using normalized graph evidence, then append the rest of base order.
    """
    top_k = int(top_k)
    preserve_top = max(0, int(preserve_top))
    rerank_top_k = max(preserve_top, min(int(rerank_top_k), top_k))
    graph_scores = graph_minmax_scores(graph_ranked)
    output = []
    seen = set()

    def add(doc_id, score):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            return False
        seen.add(canonical_id)
        output.append((str(doc_id), float(score)))
        return True

    for doc_id, score in base_ranked[:preserve_top]:
        add(doc_id, score)

    candidates = []
    window = base_ranked[preserve_top:rerank_top_k]
    for offset, (doc_id, score) in enumerate(window, start=1):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            continue
        base_component = 1.0 / (float(offset) + 1.0)
        graph_component = graph_scores.get(canonical_id, {}).get("score", 0.0)
        final_score = base_component + float(graph_weight) * graph_component
        candidates.append((str(doc_id), float(final_score), offset))
    candidates.sort(key=lambda item: (-item[1], item[2], item[0]))
    for doc_id, score, _offset in candidates:
        add(doc_id, score)

    for doc_id, score in base_ranked[rerank_top_k:]:
        if len(output) >= top_k:
            break
        add(doc_id, float(score) - 20.0)

    return output[:top_k]


def graph_rag_backfill_ranked_items(base_ranked, graph_ranked, preserve_top=50, backfill_until=100, top_k=300):
    """Preserve the strongest base ranks, then use graph as a GraphRAG recall backfill.

    This avoids the top-rank damage seen with full graph fusion while still giving graph a
    chance to insert new candidates before rank@100.
    """
    top_k = int(top_k)
    preserve_top = max(0, int(preserve_top))
    backfill_until = max(preserve_top, min(int(backfill_until), top_k))
    merged = []
    seen_canonicals = set()

    def add_item(doc_id, score):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen_canonicals:
            return False
        seen_canonicals.add(canonical_id)
        merged.append((str(doc_id), float(score)))
        return True

    for doc_id, score in base_ranked[:preserve_top]:
        add_item(doc_id, score)

    for doc_id, score in graph_ranked:
        if len(merged) >= backfill_until:
            break
        # Negative offset keeps graph backfill behind the preserved base prefix.
        add_item(doc_id, float(score) - 10.0)

    for doc_id, score in base_ranked[preserve_top:]:
        if len(merged) >= top_k:
            break
        add_item(doc_id, float(score) - 20.0)

    return merged[:top_k]


def graph_best_by_canonical(ranked_items):
    best = {}
    for rank, (doc_id, score) in enumerate(ranked_items or [], start=1):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id:
            continue
        score = float(score)
        current = best.get(canonical_id)
        if current is None or score > current["raw_score"] or (score == current["raw_score"] and rank < current["rank"]):
            best[canonical_id] = {"doc_id": str(doc_id), "raw_score": score, "rank": rank}
    return best


def graph_normalized_canonical_scores(ranked_items):
    best = graph_best_by_canonical(ranked_items)
    if not best:
        return {}
    raw_scores = [item["raw_score"] for item in best.values()]
    min_score = min(raw_scores)
    max_score = max(raw_scores)
    denom = max_score - min_score
    out = {}
    for canonical_id, item in best.items():
        if denom > 1e-12:
            norm = (item["raw_score"] - min_score) / denom
        else:
            norm = 1.0 / float(item["rank"] + 1)
        out[canonical_id] = {"doc_id": item["doc_id"], "score": float(norm), "rank": int(item["rank"])}
    return out


def hybrid_graph_rescore_expand_base(base_ranked, graph_results, plan, graph_retriever, top_k=300):
    """Two-stage retrieval: hybrid base -> graph expansion -> normalized re-score."""
    if not base_ranked:
        return []
    top_k = int(top_k)
    preserve_top = max(0, min(int(GRAPH_RESCORER_PRESERVE_TOP), top_k))
    boost_conf = parse_json_object(plan.get("boost_config_json"), {})
    seed_graph = graph_retriever.rank_hybrid_expansion(
        base_ranked,
        plan,
        top_k=GRAPH_RESCORER_EXPAND_TOP_K,
        seed_top_k=GRAPH_RESCORER_SEED_TOP_K,
    ) if graph_retriever is not None and graph_retriever.available else []

    graph_sources = [
        seed_graph,
        graph_results.get("graph_ppr_rag", []),
        graph_results.get("graph_path_rag", []),
        graph_results.get("graph_query", []),
    ]
    graph_fused = fuse_ranked_items(
        [(f"graph_source_{idx}", ranked, 1.0) for idx, ranked in enumerate(graph_sources) if ranked],
        rrf_k=50,
        top_k=GRAPH_RESCORER_EXPAND_TOP_K,
    )

    base_scores = graph_normalized_canonical_scores(base_ranked)
    graph_scores = graph_normalized_canonical_scores(graph_fused)
    candidate_canonicals = set(base_scores) | set(graph_scores)

    output = []
    preserved = set()
    for doc_id, score in base_ranked[:preserve_top]:
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in preserved:
            continue
        preserved.add(canonical_id)
        output.append((str(doc_id), float(score) + 100.0))

    rows = []
    for canonical_id in candidate_canonicals:
        if canonical_id in preserved:
            continue
        base_item = base_scores.get(canonical_id)
        graph_item = graph_scores.get(canonical_id)
        doc_id = (base_item or graph_item or {}).get("doc_id", canonical_id)
        base_score = 0.0 if base_item is None else float(base_item["score"])
        graph_score = 0.0 if graph_item is None else float(graph_item["score"])
        evidence_score = 0.0
        if graph_retriever is not None and graph_retriever.available:
            evidence_score, _evidence = graph_retriever._candidate_path_evidence_score(doc_id, plan, boost_conf)
            evidence_score = min(1.0, float(evidence_score) / 4.0)
        new_doc_bonus = GRAPH_RESCORER_NEW_DOC_WEIGHT if base_item is None and graph_item is not None else 0.0
        final_score = (
            GRAPH_RESCORER_BASE_WEIGHT * base_score
            + GRAPH_RESCORER_GRAPH_WEIGHT * graph_score
            + GRAPH_RESCORER_EVIDENCE_WEIGHT * evidence_score
            + new_doc_bonus
        )
        base_rank = 10**9 if base_item is None else int(base_item["rank"])
        graph_rank = 10**9 if graph_item is None else int(graph_item["rank"])
        rows.append((str(doc_id), float(final_score), base_rank, graph_rank, canonical_id))

    rows.sort(key=lambda item: (-item[1], item[2], item[3], item[4]))
    seen = set(preserved)
    for doc_id, score, _base_rank, _graph_rank, canonical_id in rows:
        if len(output) >= top_k:
            break
        if canonical_id in seen:
            continue
        seen.add(canonical_id)
        output.append((doc_id, score))
    return output[:top_k]


def hybrid_graph_rank_rescore_expand_base(
    base_ranked,
    graph_results,
    plan,
    graph_retriever,
    top_k=300,
    graph_weight=None,
    evidence_weight=None,
    new_doc_weight=None,
    evidence_gate_floor=None,
):
    """Two-stage retrieval with rank-based scoring instead of min-max score fusion."""
    if not base_ranked:
        return []
    top_k = int(top_k)
    preserve_top = max(0, min(int(GRAPH_RESCORER_PRESERVE_TOP), top_k))
    graph_weight = GRAPH_RANK_RESCORER_GRAPH_WEIGHT if graph_weight is None else float(graph_weight)
    evidence_weight = GRAPH_RANK_RESCORER_EVIDENCE_WEIGHT if evidence_weight is None else float(evidence_weight)
    new_doc_weight = GRAPH_RANK_RESCORER_NEW_DOC_WEIGHT if new_doc_weight is None else float(new_doc_weight)
    evidence_gate_floor = GRAPH_RANK_RESCORER_EVIDENCE_GATE_FLOOR if evidence_gate_floor is None else float(evidence_gate_floor)
    boost_conf = parse_json_object(plan.get("boost_config_json"), {})
    seed_graph = graph_retriever.rank_hybrid_expansion(
        base_ranked,
        plan,
        top_k=GRAPH_RESCORER_EXPAND_TOP_K,
        seed_top_k=GRAPH_RESCORER_SEED_TOP_K,
    ) if graph_retriever is not None and graph_retriever.available else []

    graph_sources = [
        seed_graph,
        graph_results.get("graph_ppr_rag", []),
        graph_results.get("graph_path_rag", []),
        graph_results.get("graph_query", []),
    ]
    graph_fused = fuse_ranked_items(
        [(f"graph_source_{idx}", ranked, 1.0) for idx, ranked in enumerate(graph_sources) if ranked],
        rrf_k=50,
        top_k=GRAPH_RESCORER_EXPAND_TOP_K,
    )

    base_best = graph_best_by_canonical(base_ranked)
    graph_best = graph_best_by_canonical(graph_fused)
    candidate_canonicals = set(base_best) | set(graph_best)

    output = []
    preserved = set()
    for doc_id, score in base_ranked[:preserve_top]:
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in preserved:
            continue
        preserved.add(canonical_id)
        output.append((str(doc_id), float(score) + 100.0))

    rows = []
    for canonical_id in candidate_canonicals:
        if canonical_id in preserved:
            continue
        base_item = base_best.get(canonical_id)
        graph_item = graph_best.get(canonical_id)
        doc_id = (base_item or graph_item or {}).get("doc_id", canonical_id)
        base_rank = 10**9 if base_item is None else int(base_item["rank"])
        graph_rank = 10**9 if graph_item is None else int(graph_item["rank"])
        base_rank_score = 0.0 if base_item is None else graph_rank_weight(base_rank)
        graph_rank_score = 0.0 if graph_item is None else graph_rank_weight(graph_rank)

        evidence_score = 0.0
        if graph_retriever is not None and graph_retriever.available:
            evidence_score, _evidence = graph_retriever._candidate_path_evidence_score(doc_id, plan, boost_conf)
            evidence_score = min(1.0, float(evidence_score) / 4.0)
        evidence_gate = max(evidence_gate_floor, evidence_score)
        new_doc_bonus = new_doc_weight if base_item is None and graph_item is not None else 0.0
        final_score = (
            base_rank_score
            + graph_weight * graph_rank_score * evidence_gate
            + evidence_weight * evidence_score
            + new_doc_bonus
        )
        rows.append((str(doc_id), float(final_score), base_rank, graph_rank, canonical_id))

    rows.sort(key=lambda item: (-item[1], item[2], item[3], item[4]))
    seen = set(preserved)
    for doc_id, score, _base_rank, _graph_rank, canonical_id in rows:
        if len(output) >= top_k:
            break
        if canonical_id in seen:
            continue
        seen.add(canonical_id)
        output.append((doc_id, score))
    return output[:top_k]


def graph_confident_expand_base(base_ranked, graph_results, top_k=300):
    """Keep hybrid as the authority, then add only high-confidence graph candidates.

    Unlike broad RRF fusion, this does not let graph rewrite the strong hybrid prefix.
    A graph candidate can enter the expansion window only if direct query-graph ranks it
    very high or multiple independent graph views agree on it.
    """
    top_k = int(top_k)
    preserve_top = max(0, min(int(GRAPH_CONFIDENT_EXPAND_PRESERVE_TOP), top_k))
    expand_until = max(preserve_top, min(int(GRAPH_CONFIDENT_EXPAND_UNTIL), top_k))
    max_inserts = max(0, int(GRAPH_CONFIDENT_MAX_INSERTS))
    output = []
    seen = set()

    def add(doc_id, score):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            return False
        seen.add(canonical_id)
        output.append((str(doc_id), float(score)))
        return True

    base_rank_by_canonical = {}
    base_score_by_canonical = {}
    for rank, (doc_id, score) in enumerate(base_ranked, start=1):
        canonical_id = canonical_doc_id(doc_id)
        if canonical_id and canonical_id not in base_rank_by_canonical:
            base_rank_by_canonical[canonical_id] = rank
            base_score_by_canonical[canonical_id] = float(score)

    for doc_id, score in base_ranked[:preserve_top]:
        add(doc_id, score)

    source_specs = [
        ("graph_query", 1.40),
        ("graph_ppr", 1.10),
        ("graph_ppr_rag", 1.05),
        ("graph_path_rag", 1.05),
        ("graph_only_balanced", 0.95),
        ("graph_rag_expanded", 0.75),
    ]
    candidates = {}
    for source_name, source_weight in source_specs:
        ranked = graph_results.get(source_name, []) or []
        for rank, (doc_id, score) in enumerate(ranked[:GRAPH_CONFIDENT_GRAPH_TOP_K], start=1):
            canonical_id = canonical_doc_id(doc_id)
            if not canonical_id or canonical_id in seen:
                continue
            item = candidates.setdefault(canonical_id, {
                "doc_id": str(doc_id),
                "sources": set(),
                "score": 0.0,
                "best_rank": 10**9,
                "direct_rank": 10**9,
            })
            item["sources"].add(source_name)
            item["score"] += float(source_weight) / (rank + 8.0)
            item["best_rank"] = min(item["best_rank"], rank)
            if source_name == "graph_query":
                item["direct_rank"] = min(item["direct_rank"], rank)

    rows = []
    for canonical_id, item in candidates.items():
        source_count = len(item["sources"])
        direct_ok = item["direct_rank"] <= GRAPH_CONFIDENT_DIRECT_TOP_K
        consensus_ok = source_count >= GRAPH_CONFIDENT_MIN_SOURCES
        if not (direct_ok or consensus_ok):
            continue
        base_rank = base_rank_by_canonical.get(canonical_id, 10**9)
        base_bonus = 0.0 if base_rank == 10**9 else 0.05 / math.sqrt(float(base_rank))
        new_doc_bonus = 0.015 if base_rank == 10**9 else 0.0
        final_score = float(item["score"]) + base_bonus + new_doc_bonus + 0.01 * source_count
        rows.append((item["doc_id"], final_score, source_count, item["direct_rank"], item["best_rank"], base_rank))

    rows.sort(key=lambda item: (-item[1], -item[2], item[3], item[4], item[5], item[0]))
    inserts = 0
    for doc_id, score, _source_count, _direct_rank, _best_rank, _base_rank in rows:
        if len(output) >= expand_until or inserts >= max_inserts:
            break
        if add(doc_id, score - 10.0):
            inserts += 1

    for doc_id, score in base_ranked[preserve_top:]:
        if len(output) >= top_k:
            break
        add(doc_id, float(score) - 20.0)
    return output[:top_k]


def build_graph_results_for_topic(hybrid_ranked, plan, topic_views, graph_retriever):
    top_k = retrieval_pool_top_k(plan)
    empty = {
        "graph_query": [],
        "graph_ppr": [],
        "graph_ppr_rag": [],
        "graph_expanded": [],
        "graph_generation_pool": [],
    }
    if graph_retriever is None or not graph_retriever.available:
        return empty

    params = graph_adaptive_params(plan)
    graph_query = graph_retriever.rank_query_graph(plan, topic_views, top_k=top_k)
    graph_only_hop1 = graph_retriever.rank_hybrid_expansion(
        graph_query,
        plan,
        top_k=top_k,
        seed_top_k=GRAPH_ONLY_RAG_SEED_TOP_K,
    )
    graph_only_hop2 = graph_retriever.rank_hybrid_expansion(
        graph_only_hop1,
        plan,
        top_k=top_k,
        seed_top_k=GRAPH_ONLY_RAG_SECOND_HOP_TOP_K,
    )
    graph_path_rag = graph_retriever.rank_graph_path_rag(
        plan,
        top_k=top_k,
        candidate_top_k=GRAPH_PATH_RAG_CANDIDATE_TOP_K,
    )
    graph_ppr = graph_retriever.rank_graph_ppr(
        plan,
        top_k=top_k,
        candidate_top_k=GRAPH_PPR_CANDIDATE_TOP_K,
    )

    graph_only_rag = fuse_ranked_items(
        [
            ("query_graph", graph_query, GRAPH_ONLY_RAG_QUERY_WEIGHT),
            ("graph_only_hop1", graph_only_hop1, GRAPH_ONLY_RAG_HOP1_WEIGHT),
            ("graph_only_hop2", graph_only_hop2, GRAPH_ONLY_RAG_HOP2_WEIGHT),
        ],
        rrf_k=50,
        top_k=top_k,
    )
    graph_ppr_rag = fuse_ranked_items(
        [
            ("query_graph", graph_query, GRAPH_PPR_QUERY_WEIGHT),
            ("graph_ppr", graph_ppr, 1.00),
            ("graph_path_rag", graph_path_rag, GRAPH_PPR_PATH_WEIGHT),
            ("graph_only_rag", graph_only_rag, 0.45),
        ],
        rrf_k=45,
        top_k=top_k,
    )
    graph_only_balanced = graph_rag_backfill_ranked_items(
        graph_query,
        graph_only_rag,
        preserve_top=GRAPH_ONLY_BALANCED_PRESERVE_TOP,
        backfill_until=GRAPH_ONLY_BALANCED_BACKFILL_TOP,
        top_k=top_k,
    )

    graph_from_hybrid = graph_retriever.rank_hybrid_expansion(
        hybrid_ranked,
        plan,
        top_k=top_k,
        seed_top_k=params["seed_top_k"],
    )
    graph_second_hop = graph_retriever.rank_hybrid_expansion(
        graph_from_hybrid,
        plan,
        top_k=top_k,
        seed_top_k=GRAPH_RAG_SECOND_HOP_TOP_K,
    )
    graph_rag_expanded = fuse_ranked_items(
        [
            ("query_graph", graph_query, 1.00),
            ("graph_ppr_rag", graph_ppr_rag, 1.05),
            ("graph_path_rag", graph_path_rag, 0.90),
            ("graph_only_rag", graph_only_rag, 0.80),
            ("hybrid_seed_graph", graph_from_hybrid, 0.85),
            ("second_hop_graph", graph_second_hop, 0.35),
        ],
        rrf_k=50,
        top_k=top_k,
    )

    if params["query_has_citations"]:
        # Query citations are strong graph evidence; preserve query graph first, then expand from hybrid.
        graph_expanded = graph_fill_ranked_items(
            graph_query,
            graph_from_hybrid,
            primary_keep=100,
            fill_keep=200,
            top_k=top_k,
            fill_score_offset=-10.0,
        )
    else:
        # No query citation: treat graph as expansion from strong hybrid seeds, with metadata graph as backfill.
        graph_expanded = graph_fill_ranked_items(
            graph_from_hybrid,
            graph_query,
            primary_keep=200,
            fill_keep=100,
            top_k=top_k,
            fill_score_offset=-10.0,
        )

    graph_generation_pool = graph_fill_ranked_items(
        hybrid_ranked,
        graph_rag_expanded,
        primary_keep=GRAPH_GENERATION_HYBRID_KEEP,
        fill_keep=GRAPH_GENERATION_GRAPH_KEEP,
        top_k=top_k,
    )
    return {
        "graph_query": graph_query,
        "graph_only_rag": graph_only_rag,
        "graph_only_balanced": graph_only_balanced,
        "graph_path_rag": graph_path_rag,
        "graph_ppr": graph_ppr,
        "graph_ppr_rag": graph_ppr_rag,
        "graph_expanded": graph_expanded,
        "graph_rag_expanded": graph_rag_expanded,
        "graph_generation_pool": graph_generation_pool,
    }


def build_hybrid_graph_results_for_topic(hybrid_ranked, graph_results, plan, graph_retriever=None):
    top_k = retrieval_pool_top_k(plan)
    params = graph_adaptive_params(plan)
    graph_ranked = graph_results.get("graph_expanded", []) or graph_results.get("graph_query", [])
    graph_rag_ranked = graph_results.get("graph_rag_expanded", []) or graph_ranked
    graph_ppr_ranked = graph_results.get("graph_ppr_rag", []) or graph_results.get("graph_ppr", []) or graph_rag_ranked
    results = {
        "hybrid_graph_adaptive": fuse_ranked_items(
            [("hybrid", hybrid_ranked, 1.0), ("graph", graph_ranked, params["graph_weight"])],
            rrf_k=params["rrf_k"],
            top_k=top_k,
        ),
        "hybrid_graph_safe_rerank": graph_safe_rerank_base_with_graph(
            hybrid_ranked,
            graph_rag_ranked,
            preserve_top=GRAPH_SAFE_RERANK_PRESERVE_TOP,
            rerank_top_k=GRAPH_SAFE_RERANK_TOP_K,
            top_k=top_k,
            graph_weight=GRAPH_SAFE_RERANK_WEIGHT,
        ),
        "hybrid_graph_path_rerank": graph_safe_rerank_base_with_graph(
            hybrid_ranked,
            graph_results.get("graph_path_rag", []) or graph_rag_ranked,
            preserve_top=GRAPH_SAFE_RERANK_PRESERVE_TOP,
            rerank_top_k=GRAPH_SAFE_RERANK_TOP_K,
            top_k=top_k,
            graph_weight=GRAPH_PATH_RERANK_WEIGHT,
        ),
        "hybrid_graph_confident_expand": graph_confident_expand_base(
            hybrid_ranked,
            graph_results,
            top_k=top_k,
        ),
        "hybrid_graph_rescore_expand": hybrid_graph_rescore_expand_base(
            hybrid_ranked,
            graph_results,
            plan,
            graph_retriever,
            top_k=top_k,
        ),
        "hybrid_graph_rank_rescore_expand": hybrid_graph_rank_rescore_expand_base(
            hybrid_ranked,
            graph_results,
            plan,
            graph_retriever,
            top_k=top_k,
        ),
        "hybrid_graph_ppr_rerank": graph_safe_rerank_base_with_graph(
            hybrid_ranked,
            graph_ppr_ranked,
            preserve_top=GRAPH_SAFE_RERANK_PRESERVE_TOP,
            rerank_top_k=GRAPH_SAFE_RERANK_TOP_K,
            top_k=top_k,
            graph_weight=GRAPH_PPR_RERANK_WEIGHT,
        ),
        "hybrid_graph_ppr_backfill": graph_rag_backfill_ranked_items(
            hybrid_ranked,
            graph_ppr_ranked,
            preserve_top=GRAPH_PPR_BACKFILL_PRESERVE_TOP,
            backfill_until=GRAPH_PPR_BACKFILL_UNTIL,
            top_k=top_k,
        ),
        "hybrid_graph_hybrid_rag": fuse_ranked_items(
            [("hybrid", hybrid_ranked, 1.0), ("graph_rag", graph_rag_ranked, GRAPH_HYBRID_RAG_WEIGHT)],
            rrf_k=GRAPH_HYBRID_RAG_RRF_K,
            top_k=top_k,
        ),
        "hybrid_graph_rag_backfill_top20": graph_rag_backfill_ranked_items(
            hybrid_ranked,
            graph_rag_ranked,
            preserve_top=GRAPH_RAG_PRESERVE_TOP_20,
            backfill_until=GRAPH_RAG_BACKFILL_TOP_K,
            top_k=top_k,
        ),
        "hybrid_graph_rag_backfill_top50": graph_rag_backfill_ranked_items(
            hybrid_ranked,
            graph_rag_ranked,
            preserve_top=GRAPH_RAG_PRESERVE_TOP_50,
            backfill_until=GRAPH_RAG_BACKFILL_TOP_K,
            top_k=top_k,
        ),
    }
    for method_name, graph_weight, evidence_weight, new_doc_weight, gate_floor in GRAPH_RANK_RESCORER_VARIANTS:
        results[method_name] = hybrid_graph_rank_rescore_expand_base(
            hybrid_ranked,
            graph_results,
            plan,
            graph_retriever,
            top_k=top_k,
            graph_weight=graph_weight,
            evidence_weight=evidence_weight,
            new_doc_weight=new_doc_weight,
            evidence_gate_floor=gate_floor,
        )
    return results


def load_baseline_hybrid_rankings():
    if "rankings_df" in globals() and isinstance(rankings_df, pd.DataFrame) and len(rankings_df):
        baseline = rankings_df.copy()
    elif RESULTS_OUT.exists():
        baseline = pd.read_parquet(RESULTS_OUT)
    else:
        raise FileNotFoundError(f"Run the BM25/KNN/hybrid retrieval cell first. Missing: {RESULTS_OUT}")

    hybrid_rows = baseline[baseline["method"] == "hybrid_deep_norm_knn99_bm2501"].copy()
    if hybrid_rows.empty:
        raise ValueError("Baseline rankings do not contain method='hybrid_deep_norm_knn99_bm2501'.")
    hybrid_rows = hybrid_rows.sort_values(["topic_id", "rank"])
    return {
        str(topic_id): [(str(row.doc_id), float(row.score)) for row in group.itertuples(index=False)]
        for topic_id, group in hybrid_rows.groupby("topic_id")
    }


def run_graph_retrieval_experiments():
    if not GRAPH_ENABLED:
        raise RuntimeError("GRAPH_ENABLED=false")

    plans, views, qrels, relevant_by_topic, _topic_groups = load_retrieval_inputs()
    topic_to_views = {topic_id: group.copy() for topic_id, group in views.groupby("topic_id")}
    hybrid_by_topic = load_baseline_hybrid_rankings()

    build_graph_parquet_if_needed()

    es = get_es_client()
    if not es.ping():
        raise RuntimeError("Cannot connect to Elasticsearch.")
    query_doc_ids = plans["query_doc_id"].astype(str).tolist() if "query_doc_id" in plans.columns else plans["topic_id"].astype(str).tolist()
    graph_retriever = GraphRetriever(GRAPH_DIR, es=es, index_name=BM25_INDEX, query_doc_ids=query_doc_ids)

    all_rank_rows = []
    metric_rows = []
    t0 = time.time()

    for plan in tqdm(plans.to_dict(orient="records"), desc="Graph retrieval experiments"):
        topic_id = str(plan["topic_id"])
        topic_views = topic_to_views.get(topic_id, pd.DataFrame())
        hybrid_ranked = hybrid_by_topic.get(topic_id, [])
        graph_results = build_graph_results_for_topic(hybrid_ranked, plan, topic_views, graph_retriever)
        graph_results.update(build_hybrid_graph_results_for_topic(hybrid_ranked, graph_results, plan, graph_retriever))
        relevant = relevant_by_topic.get(topic_id, set())

        for method in GRAPH_EVAL_METHODS:
            ranked_items = graph_results.get(method, [])
            all_rank_rows.extend(ranking_rows(topic_id, method, ranked_items))
            metric_rows.append(evaluate_ranking(topic_id, method, ranked_items, relevant))

    graph_rankings = pd.DataFrame(all_rank_rows)
    graph_topic_metrics = pd.DataFrame(metric_rows)

    if TOPIC_METRICS_OUT.exists():
        baseline_topic_metrics = pd.read_parquet(TOPIC_METRICS_OUT)
        combined_topic_metrics = pd.concat([baseline_topic_metrics, graph_topic_metrics], ignore_index=True)
    else:
        combined_topic_metrics = graph_topic_metrics

    summary = summarize_metrics(combined_topic_metrics)

    graph_rankings.to_parquet(GRAPH_RANKINGS_OUT, index=False, compression="zstd")
    combined_topic_metrics.to_parquet(GRAPH_TOPIC_METRICS_OUT, index=False, compression="zstd")
    summary.to_csv(GRAPH_SUMMARY_OUT, index=False)

    payload = {
        "baseline_methods": METHODS,
        "graph_methods": GRAPH_EVAL_METHODS,
        "eval_ks": EVAL_KS,
        "num_topics": int(len(relevant_by_topic)),
        "num_graph_rank_rows": int(len(graph_rankings)),
        "elapsed_sec": round(time.time() - t0, 2),
        "graph_rankings_out": str(GRAPH_RANKINGS_OUT),
        "graph_topic_metrics_out": str(GRAPH_TOPIC_METRICS_OUT),
        "graph_summary_out": str(GRAPH_SUMMARY_OUT),
        "graph_runtime_config": graph_retriever.stats(),
        "summary": summary.to_dict(orient="records"),
    }
    with open(GRAPH_SUMMARY_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("\n[GRAPH SUMMARY]")
    display_cols = [
        "method", "num_topics", "avg_retrieved_per_topic", "avg_retrieved_canonical_per_topic", "avg_duplicate_variants_removed",
        "hit@1", "precision@1", "recall@1", "f1@1", "hit@3", "precision@3", "recall@3", "f1@3",
        "hit@5", "precision@5", "recall@5", "f1@5", "hit@10", "precision@10", "recall@10", "f1@10",
        "hit@50", "recall@50", "f1@50", "hit@100", "precision@100", "recall@100", "f1@100",
    ]
    display_cols = [c for c in display_cols if c in summary.columns]
    print(summary[display_cols].to_string(index=False))

    graph_summary = summary[summary["method"].isin(GRAPH_METHODS)].copy()
    if len(graph_summary):
        print(f"[BEST_GRAPH] {graph_summary.iloc[0]['method']}")
    hybrid_graph_summary = summary[summary["method"].isin(HYBRID_GRAPH_METHODS)].copy()
    if len(hybrid_graph_summary):
        print(f"[BEST_HYBRID_GRAPH] {hybrid_graph_summary.iloc[0]['method']}")
    print("[SAVED]", GRAPH_RANKINGS_OUT)
    print("[SAVED]", GRAPH_TOPIC_METRICS_OUT)
    print("[SAVED]", GRAPH_SUMMARY_OUT)

    return graph_rankings, combined_topic_metrics, summary


graph_rankings_df, graph_topic_metrics_df, graph_summary_df = run_graph_retrieval_experiments()


[GRAPH] GRAPH_DIR=/kaggle/input/datasets/djnhngocduc/graph-parquet/graph_parquet
[GRAPH] graph parquet exists: /kaggle/input/datasets/djnhngocduc/graph-parquet/graph_parquet
[GRAPH] ready {"graph_dir": "/kaggle/input/datasets/djnhngocduc/graph-parquet/graph_parquet", "available": true, "filter_to_bm25_index": true, "valid_index_docs": 2752, "query_doc_ids": 200, "citation_src": 2690, "citation_dst": 14473, "ipc_nodes": 3995, "assignee_nodes": 5433, "inventor_nodes": 10810, "seed_top_k": 75, "with_citation_seed_top_k": 50, "no_citation_seed_top_k": 100, "candidate_top_k": 300, "adaptive_with_citation_weight": 0.15, "adaptive_no_citation_weight": 0.05, "adaptive_with_citation_rrf_k": 40, "adaptive_no_citation_rrf_k": 20, "generation_hybrid_keep": 150, "generation_graph_keep": 150, "rag_second_hop_top_k": 50, "rag_backfill_top_k": 100, "rag_preserve_top_20": 20, "rag_preserve_top_50": 50, "graph_only_rag_seed_top_k": 80, "graph_only_rag_second_hop_top_k": 60, "graph_only_rag_query_weight"

Graph retrieval experiments:   0%|          | 0/200 [00:00<?, ?it/s]


[GRAPH SUMMARY]
                                     method  num_topics  avg_retrieved_per_topic  avg_retrieved_canonical_per_topic  avg_duplicate_variants_removed  hit@1  precision@1  recall@1   f1@1  hit@3  precision@3  recall@3     f1@3  hit@5  precision@5  recall@5     f1@5  hit@10  precision@10  recall@10    f1@10  hit@50  recall@50    f1@50  hit@100  precision@100  recall@100   f1@100
              hybrid_graph_confident_expand         200                  300.000                            300.000                           0.000  0.825        0.825  0.264167 0.3995  0.925     0.670000  0.641667 0.653810  0.965        0.476  0.756250 0.582639   0.985        0.2650   0.840833 0.402088   1.000   0.940833 0.111488    1.000        0.03040    0.964583 0.058920
hybrid_graph_rank_rescore_g030_e015_gate010         200                  300.000                            300.000                           0.000  0.825        0.825  0.264167 0.3995  0.925     0.670000  0.641667 0.653810  0.

In [41]:
# ================= FETCH TEXT =================
def normalize_doc_text_value(value):
    if value is None:
        return ""
    return " ".join(str(value).split()).strip()


def limit_words(value, max_words):
    text = normalize_doc_text_value(value)
    if not text:
        return ""
    if max_words is None or int(max_words) <= 0:
        return text
    words = text.split()
    return " ".join(words[:int(max_words)])


def chunk_words(value, chunk_words, overlap_words=0, max_chunks=None):
    text = normalize_doc_text_value(value)
    if not text:
        return []
    words = text.split()
    chunk_words = max(1, int(chunk_words))
    overlap_words = max(0, min(int(overlap_words), chunk_words - 1))
    step = max(1, chunk_words - overlap_words)
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_words]
        if not piece:
            break
        chunks.append(" ".join(piece))
        if start + chunk_words >= len(words):
            break
        if max_chunks and len(chunks) >= int(max_chunks):
            break
    return chunks


def join_assignees(value):
    if value is None:
        return ""
    if isinstance(value, list):
        parts = [normalize_doc_text_value(v) for v in value if normalize_doc_text_value(v)]
        return " ; ".join(parts)
    return normalize_doc_text_value(value)


def derive_publication_year(publication_date):
    publication_date = normalize_doc_text_value(publication_date)
    if len(publication_date) >= 4 and publication_date[:4].isdigit():
        return publication_date[:4]
    return ""


def build_rerank_doc_text(doc):
    # Avoid duplicating retrieval_text because it already repeats title/abstract/claims.
    parts = [
        f"[TITLE] {limit_words(doc.get('title'), 80)}" if doc.get("title") else "",
        f"[ABSTRACT] {limit_words(doc.get('abstract'), 260)}" if doc.get("abstract") else "",
        f"[CLAIMS] {limit_words(doc.get('claims'), 850)}" if doc.get("claims") else "",
        f"[DESCRIPTION] {limit_words(doc.get('description'), 220)}" if doc.get("description") else "",
    ]
    return "\n\n".join(part for part in parts if part)


def fetch_doc_texts(es, doc_ids):
    if not doc_ids:
        return {}

    body = {
        "query": {
            "terms": {
                "doc_id": doc_ids
            }
        },
        "_source": [
            "doc_id",
            "title",
            "assignees",
            "inventors",
            "ipc_codes",
            "citations",
            "publication_date",
            "abstract",
            "claims",
            "description",
            "retrieval_text"
        ],
        "size": len(doc_ids)
    }

    resp = es.search(index=BM25_INDEX, body=body)

    fetched = {}

    for hit in resp["hits"]["hits"]:
        src = hit["_source"]
        doc_id = src["doc_id"]

        doc = {
            "title": normalize_doc_text_value(src.get("title")),
            "assignee": join_assignees(src.get("assignees")),
            "assignees": src.get("assignees") or [],
            "inventors": src.get("inventors") or [],
            "ipc_codes": src.get("ipc_codes") or [],
            "citations": src.get("citations") or [],
            "year": derive_publication_year(src.get("publication_date")),
            "publication_date": normalize_doc_text_value(src.get("publication_date")),
            "abstract": normalize_doc_text_value(src.get("abstract")),
            "claims": normalize_doc_text_value(src.get("claims")),
            "description": normalize_doc_text_value(src.get("description")),
            "retrieval_text": normalize_doc_text_value(src.get("retrieval_text")),
        }
        doc["text"] = doc["retrieval_text"] or doc["description"] or doc["abstract"] or doc["claims"]
        doc["rerank_text"] = build_rerank_doc_text(doc)
        fetched[doc_id] = doc

    return {doc_id: fetched[doc_id] for doc_id in doc_ids if doc_id in fetched}

# ================= RERANK =================
RERANK_MODEL_NAME = os.getenv("RERANK_MODEL_NAME", "jinaai/jina-reranker-v2-base-multilingual")
RERANK_MAX_LENGTH = int(os.getenv("RERANK_MAX_LENGTH", "1024"))
RERANK_BATCH_SIZE = int(os.getenv("RERANK_BATCH_SIZE", "8" if DEVICE == "cuda" else "1"))
RERANK_QUERY_MAX_WORDS = int(os.getenv("RERANK_QUERY_MAX_WORDS", "320"))
RERANK_QUERY_VIEWS = [x.strip() for x in os.getenv("RERANK_QUERY_VIEWS", "combined_short,title_abstract_full,claims_short").split(",") if x.strip()]
RERANK_USE_FP16 = os.getenv("RERANK_USE_FP16", "true").lower() == "true" and DEVICE == "cuda"
RERANK_TRUST_REMOTE_CODE = os.getenv("RERANK_TRUST_REMOTE_CODE", "true").lower() == "true"
RERANK_MULTI_CHUNK = os.getenv("RERANK_MULTI_CHUNK", "true").lower() == "true"
RERANK_DOC_CHUNK_WORDS = int(os.getenv("RERANK_DOC_CHUNK_WORDS", "360"))
RERANK_DOC_CHUNK_OVERLAP = int(os.getenv("RERANK_DOC_CHUNK_OVERLAP", "80"))
RERANK_MAX_CHUNKS_PER_DOC = int(os.getenv("RERANK_MAX_CHUNKS_PER_DOC", "4"))
RERANK_CHUNK_AGG = os.getenv("RERANK_CHUNK_AGG", "max").lower()
RERANK_MEAN_WEIGHT = float(os.getenv("RERANK_MEAN_WEIGHT", "0.15"))
RERANK_ADAPTIVE_CHUNKS = os.getenv("RERANK_ADAPTIVE_CHUNKS", "true").lower() == "true"
RERANK_FULL_CHUNK_TOP_K = int(os.getenv("RERANK_FULL_CHUNK_TOP_K", "80"))
RERANK_HEAD_CHUNKS_PER_DOC = int(os.getenv("RERANK_HEAD_CHUNKS_PER_DOC", str(RERANK_MAX_CHUNKS_PER_DOC)))
RERANK_TAIL_CHUNKS_PER_DOC = int(os.getenv("RERANK_TAIL_CHUNKS_PER_DOC", "1"))

print(
    f"Rerank setup default_model={RERANK_MODEL_NAME} | device={DEVICE} | fp16={RERANK_USE_FP16} | "
    f"max_length={RERANK_MAX_LENGTH} | batch={RERANK_BATCH_SIZE} | multi_chunk={RERANK_MULTI_CHUNK} | "
    f"doc_chunk_words={RERANK_DOC_CHUNK_WORDS} | max_chunks={RERANK_MAX_CHUNKS_PER_DOC} | agg={RERANK_CHUNK_AGG} | "
    f"adaptive={RERANK_ADAPTIVE_CHUNKS} | full_top_k={RERANK_FULL_CHUNK_TOP_K} | "
    f"head_chunks={RERANK_HEAD_CHUNKS_PER_DOC} | tail_chunks={RERANK_TAIL_CHUNKS_PER_DOC}"
)
RERANK_MODEL_CACHE = {}


def get_rerank_components(model_name=None, trust_remote_code=None):
    model_name = model_name or RERANK_MODEL_NAME
    trust_remote_code = RERANK_TRUST_REMOTE_CODE if trust_remote_code is None else bool(trust_remote_code)
    cache_key = (str(model_name), bool(trust_remote_code), bool(RERANK_USE_FP16), str(DEVICE))
    if cache_key in RERANK_MODEL_CACHE:
        return RERANK_MODEL_CACHE[cache_key]
    print(f"[RERANK MODEL] loading {model_name} | trust_remote_code={trust_remote_code}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=trust_remote_code)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if RERANK_USE_FP16 else torch.float32,
        trust_remote_code=trust_remote_code,
    ).to(DEVICE)
    model.eval()
    RERANK_MODEL_CACHE[cache_key] = (tokenizer, model)
    return tokenizer, model


def build_rerank_query(topic_views):
    preferred_views = RERANK_QUERY_VIEWS

    queries = []
    seen = set()

    for view_name in preferred_views:
        matches = topic_views[topic_views["query_view"] == view_name]
        for _, row in matches.iterrows():
            text = normalize_doc_text_value(row.get("query_text"))
            if text and text not in seen:
                seen.add(text)
                queries.append(text)

    if not queries:
        for _, row in topic_views.iterrows():
            text = normalize_doc_text_value(row.get("query_text"))
            if text and text not in seen:
                seen.add(text)
                queries.append(text)

    return limit_words(" ".join(queries), RERANK_QUERY_MAX_WORDS)


def build_rerank_doc_chunks(doc, max_chunks=None):
    max_chunks = int(max_chunks or RERANK_MAX_CHUNKS_PER_DOC)
    max_chunks = max(1, max_chunks)

    title = normalize_doc_text_value(doc.get("title"))
    abstract = normalize_doc_text_value(doc.get("abstract"))
    claims = normalize_doc_text_value(doc.get("claims"))
    description = normalize_doc_text_value(doc.get("description"))

    prefix_parts = []
    if title:
        prefix_parts.append(f"[TITLE] {limit_words(title, 80)}")
    if abstract:
        prefix_parts.append(f"[ABSTRACT] {limit_words(abstract, 220)}")
    prefix = "\n".join(prefix_parts).strip()

    chunks = []
    seen = set()

    def add_chunk(label, text):
        text = normalize_doc_text_value(text)
        if not text:
            return
        chunk = f"{prefix}\n[{label}] {text}" if prefix else f"[{label}] {text}"
        key = chunk[:500]
        if key in seen:
            return
        seen.add(key)
        chunks.append(chunk)

    # First chunk is high-signal summary.
    add_chunk("SUMMARY", " ".join([
        limit_words(title, 80),
        limit_words(abstract, 260),
        limit_words(claims, 260),
    ]))

    for part in chunk_words(claims, RERANK_DOC_CHUNK_WORDS, RERANK_DOC_CHUNK_OVERLAP, max_chunks):
        add_chunk("CLAIMS", part)

    remaining = max(0, max_chunks - len(chunks))
    if remaining > 0:
        for part in chunk_words(description, RERANK_DOC_CHUNK_WORDS, RERANK_DOC_CHUNK_OVERLAP, remaining):
            add_chunk("DESCRIPTION", part)

    if not chunks:
        fallback = doc.get("rerank_text") or doc.get("text") or ""
        add_chunk("TEXT", limit_words(fallback, RERANK_DOC_CHUNK_WORDS))

    return chunks[:max_chunks]


def aggregate_chunk_scores(scores):
    if not scores:
        return float("-inf")
    scores = [float(s) for s in scores]
    max_score = max(scores)
    if RERANK_CHUNK_AGG == "max":
        return max_score
    if RERANK_CHUNK_AGG == "mean":
        return float(np.mean(scores))
    # max_mean rewards one very strong chunk while mildly rewarding repeated evidence.
    return float(max_score + RERANK_MEAN_WEIGHT * np.mean(scores))


@torch.no_grad()
def rerank_with_cross_encoder(query, docs, top_k=300, batch_size=None, max_length=None, model_name=None, trust_remote_code=None):
    if not query or not docs:
        return []

    batch_size = int(batch_size or RERANK_BATCH_SIZE)
    max_length = int(max_length or RERANK_MAX_LENGTH)
    rerank_tokenizer, rerank_model = get_rerank_components(model_name=model_name, trust_remote_code=trust_remote_code)
    doc_ids = list(docs.keys())

    pairs = []
    pair_doc_ids = []
    for rank_idx, doc_id in enumerate(doc_ids, start=1):
        if RERANK_MULTI_CHUNK:
            if RERANK_ADAPTIVE_CHUNKS:
                max_doc_chunks = RERANK_HEAD_CHUNKS_PER_DOC if rank_idx <= RERANK_FULL_CHUNK_TOP_K else RERANK_TAIL_CHUNKS_PER_DOC
            else:
                max_doc_chunks = RERANK_MAX_CHUNKS_PER_DOC
            chunks = build_rerank_doc_chunks(docs[doc_id], max_chunks=max_doc_chunks)
        else:
            chunks = [docs[doc_id].get("rerank_text") or docs[doc_id].get("text") or ""]
        for chunk in chunks:
            if not chunk:
                continue
            pairs.append([query, chunk])
            pair_doc_ids.append(doc_id)

    if not pairs:
        return []

    doc_scores = defaultdict(list)
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i + batch_size]
        batch_doc_ids = pair_doc_ids[i:i + batch_size]
        inputs = rerank_tokenizer(
            batch_pairs,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=max_length,
        ).to(DEVICE)

        if RERANK_USE_FP16:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = rerank_model(**inputs).logits.squeeze(-1)
        else:
            logits = rerank_model(**inputs).logits.squeeze(-1)
        batch_scores = logits.float().cpu().numpy().tolist()
        if not isinstance(batch_scores, list):
            batch_scores = [batch_scores]

        for doc_id, score in zip(batch_doc_ids, batch_scores):
            doc_scores[str(doc_id)].append(float(score))

    ranked = [(doc_id, aggregate_chunk_scores(scores)) for doc_id, scores in doc_scores.items()]
    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked[:int(top_k)]


def rerank_with_bge(query, docs, top_k=300, batch_size=None, max_length=None):
    return rerank_with_cross_encoder(query, docs, top_k=top_k, batch_size=batch_size, max_length=max_length)


def normalize_scores(ranked_docs):
    if not ranked_docs:
        return []

    scores = [s for _, s in ranked_docs]
    min_s = min(scores)
    max_s = max(scores)

    def norm(s):
        return 100 * (s - min_s) / (max_s - min_s + 1e-6)

    return [(doc_id, s, norm(s)) for doc_id, s in ranked_docs]


def minmax_score_dict(ranked_docs):
    if not ranked_docs:
        return {}
    scores = [float(score) for _doc_id, score in ranked_docs]
    min_score = min(scores)
    max_score = max(scores)
    denom = max_score - min_score
    out = {}
    for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
        if denom > 1e-12:
            norm = (float(score) - min_score) / denom
        else:
            norm = 1.0 / float(rank + 1)
        out[str(doc_id)] = float(norm)
    return out


def safe_bge_fused_rerank(base_ranked, bge_ranked, candidate_top_k=100, preserve_top=10, rerank_until=100, output_top_k=300, bge_weight=0.18):
    """Preserve strong retrieval prefix, then use BGE as a weak signal in a bounded window."""
    if not base_ranked:
        return []
    output_top_k = int(output_top_k)
    candidate_top_k = max(1, int(candidate_top_k))
    preserve_top = max(0, min(int(preserve_top), output_top_k))
    rerank_until = max(preserve_top, min(int(rerank_until), output_top_k))
    bge_norm = minmax_score_dict(bge_ranked)
    output = []
    seen = set()

    def add(doc_id, score):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            return False
        seen.add(canonical_id)
        output.append((str(doc_id), float(score)))
        return True

    for doc_id, score in base_ranked[:preserve_top]:
        add(doc_id, float(score) + 100.0)

    candidates = []
    for rank, (doc_id, score) in enumerate(base_ranked[preserve_top:candidate_top_k], start=preserve_top + 1):
        canonical_id = canonical_doc_id(doc_id)
        if not canonical_id or canonical_id in seen:
            continue
        base_rank_score = graph_rank_weight(rank) if "graph_rank_weight" in globals() else 1.0 / math.log2(rank + 2.0)
        ce_score = bge_norm.get(str(doc_id), 0.0)
        final_score = base_rank_score + float(bge_weight) * ce_score
        candidates.append((str(doc_id), float(final_score), rank))
    candidates.sort(key=lambda item: (-item[1], item[2], item[0]))

    for doc_id, score, _rank in candidates:
        if len(output) >= rerank_until:
            break
        add(doc_id, score)

    for doc_id, score in base_ranked[preserve_top:]:
        if len(output) >= output_top_k:
            break
        add(doc_id, float(score) - 20.0)
    return output[:output_top_k]


def evidence_safe_list(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, set)):
        return [str(v).strip() for v in value if str(v).strip()]
    text = str(value).strip()
    return [text] if text else []


def evidence_norm_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip().upper()


def evidence_norm_ipc(value):
    return re.sub(r"\s+", "", evidence_norm_text(value))


def evidence_norm_doc(value):
    return evidence_norm_text(value).replace("_", "-")


def evidence_norm_name(value):
    if "clean_party_name" in globals():
        cleaned = clean_party_name(value)
        return evidence_norm_text(cleaned)
    return evidence_norm_text(value)


def build_candidate_graph_evidence(plan, doc):
    boost_conf = parse_json_object(plan.get("boost_config_json"), {}) if "parse_json_object" in globals() else {}
    query_citations = {evidence_norm_doc(v) for v in evidence_safe_list(boost_conf.get("query_citation_seeds"))}
    query_ipcs = {evidence_norm_ipc(v) for v in evidence_safe_list((boost_conf.get("ipc_boost_fields") or {}).get("ipc_codes_norm"))}
    query_assignees = {evidence_norm_name(v) for v in evidence_safe_list(boost_conf.get("assignees_clean"))}
    query_inventors = {evidence_norm_name(v) for v in evidence_safe_list(boost_conf.get("inventors_clean"))}

    doc_citations = {evidence_norm_doc(v) for v in evidence_safe_list(doc.get("citations"))}
    doc_ipcs = {evidence_norm_ipc(v) for v in evidence_safe_list(doc.get("ipc_codes"))}
    doc_assignees = {evidence_norm_name(v) for v in evidence_safe_list(doc.get("assignees"))}
    doc_inventors = {evidence_norm_name(v) for v in evidence_safe_list(doc.get("inventors"))}

    evidence = []
    shared = sorted(query_citations & doc_citations)[:5]
    if shared:
        evidence.append("shared_citations=" + ", ".join(shared))
    shared = sorted(query_ipcs & doc_ipcs)[:8]
    if shared:
        evidence.append("shared_ipc=" + ", ".join(shared))
    shared = sorted(query_assignees & doc_assignees)[:5]
    if shared:
        evidence.append("same_assignee=" + ", ".join(shared))
    shared = sorted(query_inventors & doc_inventors)[:5]
    if shared:
        evidence.append("same_inventor=" + ", ".join(shared))
    if not evidence:
        evidence.append("graph_expansion_candidate")
    return evidence


def load_rankings_for_topic(topic_id, preferred_methods=None):
    preferred_methods = preferred_methods or [
        "hybrid_graph_rank_rescore_g040_e020_gate010", # ← đưa lên đầu
        "hybrid_graph_confident_expand",
        "hybrid_graph_rank_rescore_expand",
        "hybrid_deep_norm_knn99_bm2501",
        "knn",
    ]
    frames = []
    if "graph_rankings_df" in globals() and isinstance(graph_rankings_df, pd.DataFrame) and len(graph_rankings_df):
        frames.append(graph_rankings_df)
    if "rankings_df" in globals() and isinstance(rankings_df, pd.DataFrame) and len(rankings_df):
        frames.append(rankings_df)
    for path_name in ["GRAPH_RANKINGS_OUT", "RESULTS_OUT"]:
        path = globals().get(path_name)
        if path is not None and Path(path).exists():
            frames.append(pd.read_parquet(path))
    if not frames:
        return [], ""
    rows = pd.concat(frames, ignore_index=True)
    rows["topic_id"] = rows["topic_id"].astype(str)
    rows["method"] = rows["method"].astype(str)
    for method in preferred_methods:
        method_rows = rows[(rows["topic_id"] == str(topic_id)) & (rows["method"] == method)].copy()
        if method_rows.empty:
            continue
        method_rows = method_rows.sort_values("rank")
        return [(str(row.doc_id), float(row.score)) for row in method_rows.itertuples(index=False)], method
    return [], ""


def run_final_retrieval_for_topic(es, plan, topic_views):
    topic_id = str(plan.get("topic_id"))
    ranked, method = load_rankings_for_topic(topic_id)
    if ranked:
        return {"final_method": method, "final": ranked}

    result = run_knn_for_topic(es, plan, topic_views)
    bm25_ablation = run_bm25_ablation_for_topic(es, plan, topic_views)
    result.update(bm25_ablation)
    result.update(build_hybrid_ablation_results(result, bm25_ablation, plan))
    return {
        "final_method": "hybrid_deep_norm_knn99_bm2501",
        "final": result.get("hybrid_deep_norm_knn99_bm2501", []),
        "bm25": result.get("bm25_enhanced_2view_claims_full_rrf", []),
        "knn": result.get("knn", []),
        "hybrid": result.get("hybrid_deep_norm_knn99_bm2501", []),
    }


FINAL_RERANK_INPUT_TOP_K = int(os.getenv("FINAL_RERANK_INPUT_TOP_K", "200"))
FINAL_RERANK_OUTPUT_TOP_K = int(os.getenv("FINAL_RERANK_OUTPUT_TOP_K", "300"))
RERANK_PRESERVE_TOP_K = int(os.getenv("RERANK_PRESERVE_TOP_K", "30"))
RERANK_SAFE_WINDOW_TOP_K = int(os.getenv("RERANK_SAFE_WINDOW_TOP_K", "100"))
RERANK_EXPAND_WINDOW_TOP_K = int(os.getenv("RERANK_EXPAND_WINDOW_TOP_K", "200"))
RERANK_RERANK_UNTIL = int(os.getenv("RERANK_RERANK_UNTIL", "100"))
RERANK_BGE_WEIGHT = float(os.getenv("RERANK_BGE_WEIGHT", "0.05"))

# ================= FULL PIPELINE =================
def run_full_pipeline_rerank(es, plan, topic_views):

    # ===== final retrieval: hybrid_graph_adaptive if graph cell ran, fallback to baseline hybrid =====
    result = run_final_retrieval_for_topic(es, plan, topic_views)
    final_ranking = result["final"]

    # ===== bounded candidate pool for cross-encoder =====
    top_docs = [doc for doc, _ in final_ranking[:FINAL_RERANK_INPUT_TOP_K]]

    # ===== fetch =====
    doc_texts = fetch_doc_texts(es, top_docs)
    for doc_id, doc in doc_texts.items():
        doc["graph_evidence"] = build_candidate_graph_evidence(plan, doc)

    # ===== query =====
    query = build_rerank_query(topic_views)

    # ===== safe rerank =====
    bge_scores = rerank_with_bge(query, doc_texts, top_k=len(doc_texts))
    reranked = safe_bge_fused_rerank(
        final_ranking,
        bge_scores,
        candidate_top_k=min(FINAL_RERANK_INPUT_TOP_K, RERANK_SAFE_WINDOW_TOP_K),
        preserve_top=RERANK_PRESERVE_TOP_K,
        rerank_until=RERANK_RERANK_UNTIL,
        output_top_k=FINAL_RERANK_OUTPUT_TOP_K,
        bge_weight=RERANK_BGE_WEIGHT,
    )

    return {
        "final_method": result.get("final_method"),
        "bm25": result.get("bm25", []),
        "knn": result.get("knn", []),
        "hybrid": result.get("hybrid", final_ranking),
        "final": final_ranking,
        "reranked": reranked,
        "doc_texts": doc_texts,
    }


Rerank setup default_model=jinaai/jina-reranker-v2-base-multilingual | device=cuda | fp16=True | max_length=1024 | batch=8 | multi_chunk=True | doc_chunk_words=360 | max_chunks=4 | agg=max | adaptive=True | full_top_k=80 | head_chunks=4 | tail_chunks=1


In [28]:
# # ================= RERANK EXPERIMENTS: MULTI-STRATEGY =================
# # Run after baseline retrieval + graph retrieval + FETCH TEXT/RERANK cell.
# # FIX: Old code called rerank_with_cross_encoder directly (pure CE),
# # which DESTROYED retrieval ranking. New code adds fusion strategies.


# def rrf_fusion_rerank(retrieval_ranked, ce_ranked, k=60, output_top_k=300):
#     """Reciprocal Rank Fusion between retrieval ranking and cross-encoder ranking."""
#     rrf_scores = {}
#     for rank, (doc_id, _score) in enumerate(retrieval_ranked, 1):
#         did = str(doc_id)
#         rrf_scores[did] = rrf_scores.get(did, 0.0) + 1.0 / (k + rank)
#     for rank, (doc_id, _score) in enumerate(ce_ranked, 1):
#         did = str(doc_id)
#         rrf_scores[did] = rrf_scores.get(did, 0.0) + 1.0 / (k + rank)
#     result = sorted(rrf_scores.items(), key=lambda x: -x[1])
#     return [(doc_id, score) for doc_id, score in result[:output_top_k]]


# def interpolate_rerank(retrieval_ranked, ce_ranked, alpha=0.7, output_top_k=300):
#     """Score interpolation: alpha * retrieval_norm + (1-alpha) * ce_norm."""
#     r_norm = minmax_score_dict(retrieval_ranked)
#     c_norm = minmax_score_dict(ce_ranked)
#     all_docs = set(r_norm) | set(c_norm)
#     scores = {}
#     for doc_id in all_docs:
#         scores[doc_id] = alpha * r_norm.get(doc_id, 0.0) + (1.0 - alpha) * c_norm.get(doc_id, 0.0)
#     result = sorted(scores.items(), key=lambda x: -x[1])
#     return [(doc_id, score) for doc_id, score in result[:output_top_k]]


# # --- Variant configs: each model_name gets multiple fusion strategies ---
# _BGE_MODEL = os.getenv("BGE_RERANK_MODEL_NAME", "BAAI/bge-reranker-v2-m3")
# _BGE_TRUST = os.getenv("BGE_RERANK_TRUST_REMOTE_CODE", "false").lower() == "true"
# _JINA_MODEL = os.getenv("JINA_RERANK_MODEL_NAME", "jinaai/jina-reranker-v2-base-multilingual")
# _JINA_TRUST = os.getenv("JINA_RERANK_TRUST_REMOTE_CODE", "true").lower() == "true"

# RERANK_VARIANT_CONFIGS = {
#     # --- BGE strategies ---
#     "bge_pure_top200": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "pure",
#     },
#     "bge_safe_p10_w020": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "safe",
#         "preserve_top": 10, "rerank_until": 100, "bge_weight": 0.20,
#     },
#     "bge_safe_p10_w035": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "safe",
#         "preserve_top": 10, "rerank_until": 150, "bge_weight": 0.35,
#     },
#     "bge_safe_p30_w015": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "safe",
#         "preserve_top": 30, "rerank_until": 100, "bge_weight": 0.15,
#     },
#     "bge_safe_p05_w025": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "safe",
#         "preserve_top": 5, "rerank_until": 100, "bge_weight": 0.25,
#     },
#     "bge_rrf_k60": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "rrf", "rrf_k": 60,
#     },
#     "bge_rrf_k30": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "rrf", "rrf_k": 30,
#     },
#     "bge_interp_a08": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "interpolate", "alpha": 0.80,
#     },
#     "bge_interp_a07": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "interpolate", "alpha": 0.70,
#     },
#     "bge_interp_a06": {
#         "model_name": _BGE_MODEL, "trust_remote_code": _BGE_TRUST,
#         "candidate_top_k": 200, "mode": "interpolate", "alpha": 0.60,
#     },
#     # --- Jina strategies ---
#     "jina_safe_p10_w020": {
#         "model_name": _JINA_MODEL, "trust_remote_code": _JINA_TRUST,
#         "candidate_top_k": 200, "mode": "safe",
#         "preserve_top": 10, "rerank_until": 100, "bge_weight": 0.20,
#     },
#     "jina_rrf_k60": {
#         "model_name": _JINA_MODEL, "trust_remote_code": _JINA_TRUST,
#         "candidate_top_k": 200, "mode": "rrf", "rrf_k": 60,
#     },
#     "jina_interp_a07": {
#         "model_name": _JINA_MODEL, "trust_remote_code": _JINA_TRUST,
#         "candidate_top_k": 200, "mode": "interpolate", "alpha": 0.70,
#     },
# }
# RERANK_METHODS = list(RERANK_VARIANT_CONFIGS)
# RERANK_RANKINGS_OUT = OUT_DIR / "rerank_rankings_multi_strategy.parquet"
# RERANK_TOPIC_METRICS_OUT = OUT_DIR / "rerank_topic_metrics_multi_strategy.parquet"
# RERANK_SUMMARY_OUT = OUT_DIR / "rerank_summary_multi_strategy.csv"
# RERANK_SUMMARY_JSON_OUT = OUT_DIR / "rerank_summary_multi_strategy.json"


# def _apply_fusion(mode, config, final_ranked, raw_ce_scores, output_top_k):
#     """Apply fusion strategy based on mode."""
#     if mode == "pure":
#         return raw_ce_scores[:output_top_k]
#     elif mode == "safe":
#         return safe_bge_fused_rerank(
#             final_ranked, raw_ce_scores,
#             candidate_top_k=int(config.get("candidate_top_k", 200)),
#             preserve_top=int(config.get("preserve_top", 10)),
#             rerank_until=int(config.get("rerank_until", 100)),
#             output_top_k=output_top_k,
#             bge_weight=float(config.get("bge_weight", 0.18)),
#         )
#     elif mode == "rrf":
#         return rrf_fusion_rerank(
#             final_ranked, raw_ce_scores,
#             k=int(config.get("rrf_k", 60)),
#             output_top_k=output_top_k,
#         )
#     elif mode == "interpolate":
#         return interpolate_rerank(
#             final_ranked, raw_ce_scores,
#             alpha=float(config.get("alpha", 0.7)),
#             output_top_k=output_top_k,
#         )
#     else:
#         return raw_ce_scores[:output_top_k]


# def run_bge_rerank_experiments():
#     plans, views, qrels, relevant_by_topic, _topic_groups = load_retrieval_inputs()
#     topic_to_views = {topic_id: group.copy() for topic_id, group in views.groupby("topic_id")}

#     es = get_es_client()
#     if not es.ping():
#         raise RuntimeError("Cannot connect to Elasticsearch.")

#     # Group configs by model to avoid redundant CE scoring
#     model_groups = {}
#     for method, config in RERANK_VARIANT_CONFIGS.items():
#         key = (config["model_name"], bool(config.get("trust_remote_code", False)),
#                int(config.get("candidate_top_k", 200)))
#         model_groups.setdefault(key, []).append((method, config))

#     print("[RERANK INPUT]")
#     print("plans:", plans.shape, "| views:", views.shape, "| qrels:", qrels.shape)
#     print("num_methods:", len(RERANK_METHODS))
#     print("model_groups:", len(model_groups), "unique (model, top_k) combos")
#     print("methods:", RERANK_METHODS)

#     all_rank_rows = []
#     metric_rows = []
#     runtime_rows = []
#     t0 = time.time()

#     for plan in tqdm(plans.to_dict(orient="records"), desc="Rerank topics"):
#         topic_id = str(plan["topic_id"])
#         topic_views = topic_to_views.get(topic_id, pd.DataFrame())
#         if topic_views.empty:
#             continue

#         retrieval_result = run_final_retrieval_for_topic(es, plan, topic_views)
#         final_ranked = retrieval_result.get("final", [])
#         candidate_doc_ids = [doc_id for doc_id, _ in final_ranked[:FINAL_RERANK_INPUT_TOP_K]]
#         doc_texts = fetch_doc_texts(es, candidate_doc_ids)
#         for doc_id, doc in doc_texts.items():
#             doc["graph_evidence"] = build_candidate_graph_evidence(plan, doc)
#         query = build_rerank_query(topic_views)

#         rt0 = time.time()

#         # Score each unique (model, candidate_top_k) combo ONCE, then apply all fusion modes
#         ce_cache = {}
#         for (model_name, trust_rc, cand_k), _methods in model_groups.items():
#             method_ids = candidate_doc_ids[:cand_k]
#             method_docs = {did: doc_texts[did] for did in method_ids if did in doc_texts}
#             ce_scores = rerank_with_cross_encoder(
#                 query, method_docs,
#                 top_k=len(method_docs),
#                 model_name=model_name,
#                 trust_remote_code=trust_rc,
#             )
#             ce_cache[(model_name, trust_rc, cand_k)] = ce_scores

#         rerank_variants = {}
#         for method, config in RERANK_VARIANT_CONFIGS.items():
#             key = (config["model_name"], bool(config.get("trust_remote_code", False)),
#                    int(config.get("candidate_top_k", 200)))
#             raw_ce = ce_cache[key]
#             mode = config.get("mode", "pure")
#             rerank_variants[method] = _apply_fusion(
#                 mode, config, final_ranked, raw_ce, FINAL_RERANK_OUTPUT_TOP_K
#             )

#         runtime_rows.append({
#             "topic_id": topic_id,
#             "base_method": retrieval_result.get("final_method", ""),
#             "num_input_candidates": len(candidate_doc_ids),
#             "num_fetched_docs": len(doc_texts),
#             "num_reranked": max((len(v) for v in rerank_variants.values()), default=0),
#             "elapsed_sec": round(time.time() - rt0, 3),
#         })

#         relevant = relevant_by_topic.get(topic_id, set())
#         for method, reranked in rerank_variants.items():
#             all_rank_rows.extend(ranking_rows(topic_id, method, reranked))
#             metric_rows.append(evaluate_ranking(topic_id, method, reranked, relevant))

#     rerank_rankings = pd.DataFrame(all_rank_rows)
#     rerank_topic_metrics = pd.DataFrame(metric_rows)

#     combined_metrics = rerank_topic_metrics
#     graph_metrics_path = globals().get("GRAPH_TOPIC_METRICS_OUT")
#     if graph_metrics_path is not None and Path(graph_metrics_path).exists():
#         combined_metrics = pd.concat([pd.read_parquet(graph_metrics_path), rerank_topic_metrics], ignore_index=True)
#     elif TOPIC_METRICS_OUT.exists():
#         combined_metrics = pd.concat([pd.read_parquet(TOPIC_METRICS_OUT), rerank_topic_metrics], ignore_index=True)

#     summary = summarize_metrics(combined_metrics)
#     runtime_df = pd.DataFrame(runtime_rows)

#     rerank_rankings.to_parquet(RERANK_RANKINGS_OUT, index=False, compression="zstd")
#     rerank_topic_metrics.to_parquet(RERANK_TOPIC_METRICS_OUT, index=False, compression="zstd")
#     summary.to_csv(RERANK_SUMMARY_OUT, index=False)

#     payload = {
#         "methods": RERANK_METHODS,
#         "num_topics": int(len(relevant_by_topic)),
#         "elapsed_sec": round(time.time() - t0, 2),
#         "avg_topic_rerank_sec": float(runtime_df["elapsed_sec"].mean()) if len(runtime_df) else 0.0,
#         "rankings_out": str(RERANK_RANKINGS_OUT),
#         "topic_metrics_out": str(RERANK_TOPIC_METRICS_OUT),
#         "summary_out": str(RERANK_SUMMARY_OUT),
#         "variant_configs": {k: str(v) for k, v in RERANK_VARIANT_CONFIGS.items()},
#     }
#     with open(RERANK_SUMMARY_JSON_OUT, "w", encoding="utf-8") as f:
#         json.dump(payload, f, ensure_ascii=False, indent=2)

#     print("\n[RERANK SUMMARY]")
#     display_cols = [
#         "method", "num_topics", "avg_retrieved_per_topic", "avg_retrieved_canonical_per_topic", "avg_duplicate_variants_removed",
#         "hit@1", "precision@1", "recall@1", "f1@1", "hit@3", "precision@3", "recall@3", "f1@3",
#         "hit@5", "precision@5", "recall@5", "f1@5", "hit@10", "precision@10", "recall@10", "f1@10",
#         "hit@50", "recall@50", "f1@50", "hit@100", "precision@100", "recall@100", "f1@100",
#     ]
#     display_cols = [c for c in display_cols if c in summary.columns]
#     print(summary[display_cols].to_string(index=False))
#     if len(runtime_df):
#         print("\n[RERANK RUNTIME]")
#         print(runtime_df[["num_input_candidates", "num_fetched_docs", "num_reranked", "elapsed_sec"]].describe().to_string())
#     print("[SAVED]", RERANK_RANKINGS_OUT)
#     print("[SAVED]", RERANK_TOPIC_METRICS_OUT)
#     print("[SAVED]", RERANK_SUMMARY_OUT)
#     return rerank_rankings, rerank_topic_metrics, summary


# rerank_rankings_df, rerank_topic_metrics_df, rerank_summary_df = run_bge_rerank_experiments()


[RERANK INPUT]
plans: (200, 36) | views: (1000, 16) | qrels: (630, 10)
num_methods: 13
model_groups: 2 unique (model, top_k) combos
methods: ['bge_pure_top200', 'bge_safe_p10_w020', 'bge_safe_p10_w035', 'bge_safe_p30_w015', 'bge_safe_p05_w025', 'bge_rrf_k60', 'bge_rrf_k30', 'bge_interp_a08', 'bge_interp_a07', 'bge_interp_a06', 'jina_safe_p10_w020', 'jina_rrf_k60', 'jina_interp_a07']


Rerank topics:   0%|          | 0/200 [00:00<?, ?it/s]


[RERANK SUMMARY]
                                     method  num_topics  avg_retrieved_per_topic  avg_retrieved_canonical_per_topic  avg_duplicate_variants_removed  hit@1  precision@1  recall@1   f1@1  hit@3  precision@3  recall@3     f1@3  hit@5  precision@5  recall@5     f1@5  hit@10  precision@10  recall@10    f1@10  hit@50  recall@50    f1@50  hit@100  precision@100  recall@100   f1@100
                             bge_interp_a06         200                  300.000                            300.000                           0.000  0.485        0.485  0.155417 0.2350  0.690     0.330000  0.315000 0.321429  0.735        0.247  0.392917 0.302500   0.855        0.1715   0.547083 0.260604   1.000   0.950000 0.112614    1.000        0.03040    0.964583 0.058920
                             bge_interp_a07         200                  300.000                            300.000                           0.000  0.520        0.520  0.166667 0.2520  0.700     0.345000  0.330000 0.336429  0

In [18]:
!pip install openai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
# GENERATION
from openai import OpenAI
import time
import re
import json

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.shopaikey.com/v1")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
OPENAI_TIMEOUT = int(os.getenv("OPENAI_TIMEOUT", "180"))
OPENAI_MAX_COMPLETION_TOKENS = int(os.getenv("OPENAI_MAX_COMPLETION_TOKENS", "3000"))
OPENAI_TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.1"))
GENERATION_MAX_DOCS = int(os.getenv("GENERATION_MAX_DOCS", "5"))
GENERATION_ABSTRACT_WORDS = int(os.getenv("GENERATION_ABSTRACT_WORDS", "0"))
GENERATION_CLAIMS_WORDS = int(os.getenv("GENERATION_CLAIMS_WORDS", "0"))
GENERATION_CONTENT_WORDS = int(os.getenv("GENERATION_CONTENT_WORDS", "260"))
GENERATION_MAX_RELATIONSHIPS = int(os.getenv("GENERATION_MAX_RELATIONSHIPS", "30"))
GENERATION_SINGLE_CALL_MODE = os.getenv("GENERATION_SINGLE_CALL_MODE", "true").lower() == "true"
GENERATION_RERANK_IF_MISSING = os.getenv("GENERATION_RERANK_IF_MISSING", "false").lower() == "true"

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    timeout=OPENAI_TIMEOUT
)


In [30]:
def generation_limit_words(text, max_words):
    if max_words is None or int(max_words) <= 0:
        return normalize_doc_text_value(text)
    return limit_words(text, int(max_words))


def generation_safe_list(value, limit=None):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, set)):
        items = [str(x).strip() for x in value if str(x).strip()]
    else:
        text = str(value).strip()
        if not text or text.lower() in {"nan", "none", "null", "[]"}:
            items = []
        else:
            items = [text]
    return items[:limit] if limit else items


def generation_list_preview(value, max_items=8):
    items = generation_safe_list(value, limit=max_items)
    return "; ".join(items) if items else "null"


def generation_doc_canonical(doc_id):
    if "canonical_doc_id" in globals():
        return canonical_doc_id(doc_id)
    text = str(doc_id or "").strip().upper().replace("_", "-")
    return re.sub(r"-[A-Z]\d?$", "", text)


def generation_norm_token(value):
    return re.sub(r"\s+", " ", str(value or "").strip().lower())


def generation_doc_year(doc):
    year = doc.get("year")
    if year is not None and str(year).strip():
        return str(year)
    pub = str(doc.get("publication_date") or "")
    m = re.search(r"(19|20)\d{2}", pub)
    return m.group(0) if m else None


def generation_doc_relationships(doc_data, ranked_docs, max_docs=None):
    max_docs = max_docs or GENERATION_MAX_DOCS
    selected = [doc_id for doc_id, *_ in ranked_docs[:max_docs]]
    relationships = []

    for i, left_id in enumerate(selected):
        left = doc_data.get(left_id, {})
        left_ipc = set(generation_safe_list(left.get("ipc_codes")))
        left_assignees = {generation_norm_token(x) for x in generation_safe_list(left.get("assignees"))}
        left_inventors = {generation_norm_token(x) for x in generation_safe_list(left.get("inventors"))}
        left_citations = {generation_doc_canonical(x) for x in generation_safe_list(left.get("citations"), limit=200)}

        for right_id in selected[i + 1:]:
            right = doc_data.get(right_id, {})
            right_ipc = set(generation_safe_list(right.get("ipc_codes")))
            right_assignees = {generation_norm_token(x) for x in generation_safe_list(right.get("assignees"))}
            right_inventors = {generation_norm_token(x) for x in generation_safe_list(right.get("inventors"))}
            right_citations = {generation_doc_canonical(x) for x in generation_safe_list(right.get("citations"), limit=200)}
            left_can = generation_doc_canonical(left_id)
            right_can = generation_doc_canonical(right_id)

            shared_ipc = sorted(left_ipc & right_ipc)[:5]
            shared_assignees = sorted(x for x in (left_assignees & right_assignees) if x)[:3]
            shared_inventors = sorted(x for x in (left_inventors & right_inventors) if x)[:3]
            citation_links = []
            if right_can in left_citations:
                citation_links.append(f"{left_id} cites {right_id}")
            if left_can in right_citations:
                citation_links.append(f"{right_id} cites {left_id}")

            if citation_links:
                relationships.append({
                    "type": "citation_link",
                    "source": left_id,
                    "target": right_id,
                    "evidence": citation_links,
                })
            if shared_ipc:
                relationships.append({
                    "type": "shared_ipc",
                    "source": left_id,
                    "target": right_id,
                    "evidence": shared_ipc,
                })
            if shared_assignees:
                relationships.append({
                    "type": "same_assignee",
                    "source": left_id,
                    "target": right_id,
                    "evidence": shared_assignees,
                })
            if shared_inventors:
                relationships.append({
                    "type": "same_inventor",
                    "source": left_id,
                    "target": right_id,
                    "evidence": shared_inventors,
                })

            if len(relationships) >= GENERATION_MAX_RELATIONSHIPS:
                return relationships[:GENERATION_MAX_RELATIONSHIPS]

    return relationships[:GENERATION_MAX_RELATIONSHIPS]


def build_context_from_docs(doc_data, ranked_docs, max_docs=None):
    max_docs = max_docs or GENERATION_MAX_DOCS
    ranked_docs = normalize_scores(ranked_docs)
    relationships = generation_doc_relationships(doc_data, ranked_docs, max_docs=max_docs)

    parts = [
        "CANDIDATE_TO_CANDIDATE_GRAPH_RELATIONSHIPS:\n"
        + json.dumps(relationships, ensure_ascii=False, indent=2)
    ]

    for rank, (doc_id, raw_score, norm_score) in enumerate(ranked_docs[:max_docs], start=1):
        d = doc_data.get(doc_id, {})
        graph_evidence = generation_safe_list(d.get("graph_evidence"), limit=12)
        citations = generation_safe_list(d.get("citations"), limit=12)

        block = f"""
RANK: {rank}
PATENT_ID: {doc_id}
SIMILARITY_SCORE_PERCENT: {round(float(norm_score) * 100.0, 2)}
RAW_RERANK_SCORE: {raw_score}
TITLE: {d.get('title')}
ASSIGNEE: {generation_list_preview(d.get('assignees'), max_items=6)}
INVENTORS: {generation_list_preview(d.get('inventors'), max_items=8)}
YEAR: {generation_doc_year(d)}
PUBLICATION_DATE: {d.get('publication_date')}
IPC_CODES: {generation_list_preview(d.get('ipc_codes'), max_items=12)}
CITATIONS_SAMPLE: {generation_list_preview(citations, max_items=12)}
GRAPH_EVIDENCE_QUERY_TO_PATENT: {' ; '.join(graph_evidence) if graph_evidence else 'null'}

ABSTRACT_EXCERPT:
{generation_limit_words(d.get('abstract'), GENERATION_ABSTRACT_WORDS)}

CLAIMS_EXCERPT:
{generation_limit_words(d.get('claims'), GENERATION_CLAIMS_WORDS)}

DESCRIPTION_EXCERPT:
{generation_limit_words(d.get('description'), GENERATION_CONTENT_WORDS)}
"""
        parts.append(block.strip())

    return "\n\n=====\n\n".join(parts)


In [31]:
SYSTEM_PROMPT = (
    "Bạn là chuyên gia trong lĩnh vực tìm kiếm sáng chế và phân tích prior-art. Tất cả nội dung diễn giải trong JSON PHẢI bằng tiếng Việt có dấu.\n"
    "Nhiệm vụ: tạo báo cáo phân tích prior art từ QUERY và CONTEXT đã được RETRIEVAL + RERANKING.\n"
    "Thiết kế generation là single-call: mỗi query chỉ gọi API một lần, nên phải từ hoàn thiện báo cáo tốt nhất từ context hiện có.\n"
    "Chỉ dùng thông tin trong QUERY và CONTEXT. Không suy đoán, không dùng kiến thức ngoài context.\n"
    "Output bắt buộc là JSON hợp lệ, không thêm markdown, không thêm giải thích ngoài JSON.\n"
    "Tất cả giá trị dạng văn bản trong JSON phải viết tiếng Việt có dấu, trừ patent_id, mã IPC, tên riêng và trích dẫn kỹ thuật cần giữ nguyên.\n"
    "Nếu thiếu field thì dùng null hoặc mảng rỗng. Không tự bia assignee, inventor, year, citation hay quan hệ graph.\n"
    "Claims/abstract phải được tóm tắt ngắn gọn, không copy nguyên văn dài vào output.\n"
    "similarity_score phải là điểm tương đồng tương đối trong [0,100], lấy từ SIMILARITY_SCORE_PERCENT trong context; không gọi đây là xác suất.\n"
    "Chỉ đưa tối đa 5 prior art mạnh nhất vào prior_art_list, theo đúng thứ hạng trong context.\n"
    "coverage_assessment phải đánh giá độ tin cậy dựa trên score, độ khớp claim/metadata và graph evidence có trong context.\n"
)


def build_messages(query, context):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""
QUERY PATENT / SÁNG CHẾ:
{query}

RETRIEVED + RERANKING CONTEXT:
{context}

Trả về JSON theo đúng schema sau. Mỗi nội dung diễn giải phải bằng tiếng Việt có dấu:

{{
  "query_understanding": {{
    "input_type": "patent_document | idea | unknown",
    "technical_problem": string | null,
    "key_technical_features": [string],
    "metadata_filters_used": {{
      "date_cutoff": string | null,
      "ipc_codes": [string],
      "assignees": [string]
    }}
  }},
  "prior_art_list": [
    {{
      "rank": number,
      "patent_id": string,
      "title": string | null,
      "assignee": string | null,
      "year": string | null,
      "similarity_score": number,
      "abstract_summary": string | null,
      "key_claims": [string],
      "relevant_claim_elements": [string],
      "metadata": {{
        "publication_date": string | null,
        "ipc_codes": [string],
        "inventors": [string],
        "citations_sample": [string]
      }},
      "graph_relationships": [string],
      "relevance_explanation": string,
      "limitations": string | null
    }}
  ],
  "knowledge_graph_analysis": {{
    "query_to_candidate_relationships": [string],
    "candidate_to_candidate_relationships": [string],
    "important_ipc_clusters": [string],
    "assignee_or_inventor_links": [string]
  }},
  "coverage_assessment": {{
    "is_result_sufficient": boolean,
    "confidence": "high | medium | low",
    "coverage_notes": string,
    "recommended_next_actions": [string]
  }},
  "structured_report": {{
    "executive_summary": string,
    "strongest_prior_art": [string],
    "novelty_risk_factors": [string],
    "claim_overlap_summary": [string],
    "metadata_summary": string,
    "graph_evidence_summary": string
  }}
}}

QUY TẮC BẮT BUỘC:
- prior_art_list chỉ lấy tối đa 5 patent mạnh nhất.
- Không thêm patent không có trong context.
- graph_relationships chỉ được dùng GRAPH_EVIDENCE_QUERY_TO_PATENT hoặc CANDIDATE_TO_CANDIDATE_GRAPH_RELATIONSHIPS.
- Nếu graph evidence yếu hoặc không có, phải nói rõ trong graph_evidence_summary và limitations.
- Vì chỉ có một lần gọi API, không yêu cầu chạy lại, không yêu cầu bổ sung context; nếu thiếu dữ liệu thì ghi rõ trong limitations/coverage_notes.
- Không trả thêm text ngoài JSON.
"""
        }
    ]

In [32]:
def generate_answer(query: str, context: str):
    if not context.strip():
        return json.dumps({
            "prior_art_list": [],
            "coverage_assessment": {
                "is_result_sufficient": False,
                "confidence": "low",
                "coverage_notes": "Không có ngữ cảnh để phân tích.",
                "recommended_next_actions": ["Chạy lại retrieval/rerank để tạo candidate context."]
            }
        }, ensure_ascii=False), 0.0

    messages = build_messages(query, context)
    t0 = time.time()

    request = {
        "model": OPENAI_MODEL,
        "messages": messages,
        "max_completion_tokens": OPENAI_MAX_COMPLETION_TOKENS,
        "temperature": OPENAI_TEMPERATURE,
        "response_format": {"type": "json_object"},
    }

    try:
        resp = client.chat.completions.create(**request)
    except Exception as exc:
        # Mot so endpoint/model khong ho tro response_format hoac temperature.
        msg = str(exc).lower()
        if "response_format" in msg or "temperature" in msg:
            request.pop("response_format", None)
            if "temperature" in msg:
                request.pop("temperature", None)
            resp = client.chat.completions.create(**request)
        else:
            raise

    latency = time.time() - t0
    answer = resp.choices[0].message.content or ""
    return clean_answer(answer), latency


In [33]:
def clean_answer(text: str):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"assistant:", "", text, flags=re.IGNORECASE)
    return text.strip()

In [34]:
import json
import re

def extract_json(answer: str):
    match = re.search(r"\{.*\}", answer, re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except:
        return None

In [40]:
def load_existing_rerank_for_topic(topic_id, preferred_methods=None):
    preferred_methods = preferred_methods or [
        "bge_full_rerank_top200",
        "jina_full_rerank_top200",
        "bge_adaptive_rerank_top300",
        "bge_rerank_top300",
    ]

    frames = []
    if "rerank_rankings_df" in globals() and isinstance(rerank_rankings_df, pd.DataFrame) and len(rerank_rankings_df):
        frames.append(rerank_rankings_df)

    candidate_paths = []
    if "RERANK_RANKINGS_OUT" in globals():
        candidate_paths.append(Path(RERANK_RANKINGS_OUT))
    if "OUT_DIR" in globals():
        candidate_paths.extend([
            Path(OUT_DIR) / "rerank_rankings_bge_jina_full_top200.parquet",
            Path(OUT_DIR) / "rerank_rankings_cross_encoder_safe_fused.parquet",
            Path(OUT_DIR) / "rerank_rankings_bge_safe_fused.parquet",
            Path(OUT_DIR) / "rerank_rankings_bge_adaptive_top300.parquet",
            Path(OUT_DIR) / "rerank_rankings_bge_top300.parquet",
        ])

    seen_paths = set()
    for path in candidate_paths:
        path = Path(path)
        if path in seen_paths:
            continue
        seen_paths.add(path)
        if path.exists():
            frames.append(pd.read_parquet(path))

    if not frames:
        return [], ""

    rows = pd.concat(frames, ignore_index=True)
    if rows.empty or not {"topic_id", "method", "doc_id", "rank", "score"}.issubset(rows.columns):
        return [], ""

    rows["topic_id"] = rows["topic_id"].astype(str)
    rows["method"] = rows["method"].astype(str)

    for method in preferred_methods:
        method_rows = rows[(rows["topic_id"] == str(topic_id)) & (rows["method"] == method)].copy()
        if method_rows.empty:
            continue
        method_rows = method_rows.sort_values("rank")
        ranked = [(str(row.doc_id), float(row.score)) for row in method_rows.itertuples(index=False)]
        return ranked, method

    return [], ""


def load_best_retrieval_for_topic(topic_id, preferred_methods=None, dedup_canonical=True):
    preferred_methods = preferred_methods or ["hybrid_graph_rank_rescore_g040_e020_gate010", "hybrid_graph_confident_expand", "hybrid_graph_rank_rescore_expand", "hybrid_deep_norm_knn99_bm2501"]
    ranked, method = load_rankings_for_topic(str(topic_id), preferred_methods=preferred_methods)
    if not ranked:
        return [], ""
    if dedup_canonical and "canonical_dedup_ranked_items" in globals():
        ranked = canonical_dedup_ranked_items(ranked)
    return ranked, method


def run_full_pipeline_generation(es, plan, topic_views=None, use_rerank=False):

    if "prepare_agent_query_inputs" in globals():
        plan, topic_views = prepare_agent_query_inputs(plan, topic_views, topic_id=str(plan.get("topic_id", "idea_query")) if isinstance(plan, dict) else "idea_query")

    topic_id = str(plan.get("topic_id"))

    # ===== STEP 4 agent: build auditable execution plan =====
    if "run_planner_agent" in globals():
        planner_state = run_planner_agent(plan, topic_views)
        retrieval_execution_plan = planner_state.get("retrieval_plan", {})
        planner_audit_log = planner_state.get("audit_log", [])
        planner_langgraph_used = bool(planner_state.get("langgraph_used", False))
    else:
        retrieval_execution_plan = {}
        planner_audit_log = []
        planner_langgraph_used = False

    # ===== STEP 1: use best filtered retrieval for generation by default =====
    used_existing_rerank = False
    rerank_method = ""
    if use_rerank:
        reranked, rerank_method = load_existing_rerank_for_topic(topic_id)
        used_existing_rerank = bool(reranked)
    else:
        reranked, rerank_method = [], ""

    if used_existing_rerank:
        final_ranking = reranked
        final_method = rerank_method
        retrieval_result = {"final_method": rerank_method, "final": reranked}
    else:
        best_retrieval, best_retrieval_method = load_best_retrieval_for_topic(topic_id, dedup_canonical=True)
        if best_retrieval:
            final_ranking = best_retrieval
            final_method = f"{best_retrieval_method}_canonical_filtered"
            retrieval_result = {"final_method": final_method, "final": best_retrieval, "hybrid": best_retrieval}
        else:
            retrieval_result = run_final_retrieval_for_topic(es, plan, topic_views)
            final_ranking = retrieval_result["final"]
            final_method = retrieval_result.get("final_method")
            if "canonical_dedup_ranked_items" in globals():
                final_ranking = canonical_dedup_ranked_items(final_ranking)
                final_method = f"{final_method}_canonical_filtered"

        reranked = final_ranking

    normalized_reranked = normalize_scores(reranked)

    # ===== STEP 2: fetch only the docs needed for generation context =====
    generation_doc_ids = [doc for doc, _ in reranked[:GENERATION_MAX_DOCS]]
    doc_texts = fetch_doc_texts(es, generation_doc_ids)
    for doc_id, doc in doc_texts.items():
        doc["graph_evidence"] = build_candidate_graph_evidence(plan, doc)

    # ===== STEP 3: build query + context =====
    query = build_rerank_query(topic_views)
    context = build_context_from_docs(
        doc_texts,
        reranked,
        max_docs=GENERATION_MAX_DOCS,
    )

    # ===== STEP 4: single-call generation / analysis =====
    answer, latency = generate_answer(query, context)
    parsed = extract_json(answer)

    if "run_evaluator_agent" in globals():
        evaluator_state = run_evaluator_agent(
            plan,
            topic_views=topic_views,
            reranked=reranked,
            answer_json=parsed or {},
            context_doc_ids=generation_doc_ids,
        )
        agent_evaluation = evaluator_state.get("evaluation", {})
        agent_audit_log = evaluator_state.get("audit_log", [])
        evaluator_langgraph_used = bool(evaluator_state.get("langgraph_used", False))
    else:
        agent_evaluation = {}
        agent_audit_log = []
        evaluator_langgraph_used = False

    return {
        "final_method": final_method,
        "retrieval_execution_plan": retrieval_execution_plan,
        "planner_audit_log": planner_audit_log,
        "planner_langgraph_used": planner_langgraph_used,
        "used_existing_rerank": used_existing_rerank,
        "rerank_method": rerank_method,
        "bm25": retrieval_result.get("bm25", []),
        "knn": retrieval_result.get("knn", []),
        "hybrid": retrieval_result.get("hybrid", retrieval_result.get("final", [])),
        "final": final_ranking,
        "reranked": reranked,
        "reranked_normalized": normalized_reranked,
        "generation_context_docs": GENERATION_MAX_DOCS,
        "context_doc_ids": generation_doc_ids,
        "context": context,
        "answer_raw": answer,
        "answer_json": parsed,
        "agent_evaluation": agent_evaluation,
        "agent_audit_log": agent_audit_log,
        "evaluator_langgraph_used": evaluator_langgraph_used,
        "latency": latency,
    }


In [36]:
def render_generation_report(answer_json, topic_id=None):
    """Render generation JSON into a human-readable Vietnamese Markdown report."""
    if not isinstance(answer_json, dict):
        return "Kh\u00f4ng c\u00f3 JSON h\u1ee3p l\u1ec7 \u0111\u1ec3 hi\u1ec3n th\u1ecb b\u00e1o c\u00e1o."

    lines = []
    title = "B\u00e1o c\u00e1o ph\u00e2n t\u00edch prior art"
    if topic_id:
        title += f" cho {topic_id}"
    lines.append(f"# {title}")

    query = answer_json.get("query_understanding") or {}
    if query:
        lines.append("\n## 1. T\u00f3m t\u1eaft truy v\u1ea5n")
        problem = query.get("technical_problem")
        if problem:
            lines.append(str(problem))
        features = query.get("key_technical_features") or []
        if features:
            lines.append("\n**C\u00e1c \u0111\u1eb7c tr\u01b0ng k\u1ef9 thu\u1eadt ch\u00ednh:**")
            for item in features:
                lines.append(f"- {item}")
        filters = query.get("metadata_filters_used") or {}
        if filters:
            date_cutoff = filters.get("date_cutoff")
            ipc_codes = filters.get("ipc_codes") or []
            assignees = filters.get("assignees") or []
            lines.append("\n**B\u1ed9 l\u1ecdc metadata/filter \u0111\u01b0\u1ee3c ghi nh\u1eadn:**")
            missing_text = "Kh\u00f4ng c\u00f3 trong output"
            date_cutoff_text = date_cutoff if date_cutoff else missing_text
            ipc_text = ", ".join(ipc_codes) if ipc_codes else missing_text
            assignee_text = ", ".join(assignees) if assignees else missing_text
            lines.append(f"- M\u1ed1c th\u1eddi gian prior art: {date_cutoff_text}")
            lines.append(f"- IPC: {ipc_text}")
            lines.append(f"- Assignee: {assignee_text}")

    prior_art = answer_json.get("prior_art_list") or []
    lines.append("\n## 2. Danh s\u00e1ch prior art li\u00ean quan nh\u1ea5t")
    if not prior_art:
        lines.append("Kh\u00f4ng c\u00f3 prior art n\u00e0o \u0111\u01b0\u1ee3c tr\u00edch xu\u1ea5t.")
    for item in prior_art:
        if not isinstance(item, dict):
            continue
        rank = item.get("rank")
        patent_id = item.get("patent_id") or "N/A"
        patent_title = item.get("title") or "Kh\u00f4ng r\u00f5 ti\u00eau \u0111\u1ec1"
        assignee = item.get("assignee") or "Kh\u00f4ng r\u00f5 assignee"
        year = item.get("year") or "Kh\u00f4ng r\u00f5 n\u0103m"
        score = item.get("similarity_score")
        score_text = "Kh\u00f4ng c\u00f3 \u0111i\u1ec3m" if score is None else f"{float(score):.2f}"

        lines.append(f"\n### {rank}. {patent_id} - {patent_title}")
        lines.append(f"- **Assignee:** {assignee}")
        lines.append(f"- **N\u0103m:** {year}")
        lines.append(f"- **\u0110i\u1ec3m t\u01b0\u01a1ng \u0111\u1ed3ng t\u01b0\u01a1ng \u0111\u1ed1i:** {score_text}")

        abstract_summary = item.get("abstract_summary")
        if abstract_summary:
            lines.append(f"- **T\u00f3m t\u1eaft abstract:** {abstract_summary}")

        key_claims = item.get("key_claims") or []
        if key_claims:
            lines.append("- **Claim/\u0111\u1eb7c \u0111i\u1ec3m ch\u00ednh:**")
            for claim in key_claims[:6]:
                lines.append(f"  - {claim}")

        relevant_elements = item.get("relevant_claim_elements") or []
        if relevant_elements:
            lines.append("- **Y\u1ebfu t\u1ed1 claim tr\u00f9ng ho\u1eb7c g\u1ea7n v\u1edbi truy v\u1ea5n:**")
            for elem in relevant_elements[:6]:
                lines.append(f"  - {elem}")

        explanation = item.get("relevance_explanation")
        if explanation:
            lines.append(f"- **L\u00fd do li\u00ean quan:** {explanation}")

        limitations = item.get("limitations")
        if limitations:
            lines.append(f"- **H\u1ea1n ch\u1ebf/kh\u00e1c bi\u1ec7t:** {limitations}")

        graph_relationships = item.get("graph_relationships") or []
        if graph_relationships:
            lines.append("- **Quan h\u1ec7 \u0111\u1ed3 th\u1ecb:**")
            for rel in graph_relationships[:5]:
                lines.append(f"  - {rel}")

    kg = answer_json.get("knowledge_graph_analysis") or {}
    if kg:
        lines.append("\n## 3. Ph\u00e2n t\u00edch \u0111\u1ed3 th\u1ecb tri th\u1ee9c")
        sections = [
            ("Quan h\u1ec7 gi\u1eefa truy v\u1ea5n v\u00e0 \u1ee9ng vi\u00ean", kg.get("query_to_candidate_relationships") or []),
            ("Quan h\u1ec7 gi\u1eefa c\u00e1c prior art", kg.get("candidate_to_candidate_relationships") or []),
            ("C\u1ee5m IPC quan tr\u1ecdng", kg.get("important_ipc_clusters") or []),
            ("Li\u00ean k\u1ebft assignee/inventor", kg.get("assignee_or_inventor_links") or []),
        ]
        for heading, values in sections:
            if values:
                lines.append(f"\n**{heading}:**")
                for value in values[:8]:
                    lines.append(f"- {value}")

    report = answer_json.get("structured_report") or {}
    if report:
        lines.append("\n## 4. B\u00e1o c\u00e1o ph\u00e2n t\u00edch c\u00f3 c\u1ea5u tr\u00fac")
        if report.get("executive_summary"):
            lines.append(f"**T\u00f3m t\u1eaft:** {report['executive_summary']}")

        for key, heading in [
            ("strongest_prior_art", "Prior art m\u1ea1nh nh\u1ea5t"),
            ("novelty_risk_factors", "R\u1ee7i ro \u0111\u1ed1i v\u1edbi t\u00ednh m\u1edbi"),
            ("claim_overlap_summary", "T\u00f3m t\u1eaft m\u1ee9c \u0111\u1ed9 ch\u1ed3ng l\u1ea5n claim"),
        ]:
            values = report.get(key) or []
            if values:
                lines.append(f"\n**{heading}:**")
                for value in values:
                    lines.append(f"- {value}")

        if report.get("metadata_summary"):
            lines.append(f"\n**T\u00f3m t\u1eaft metadata:** {report['metadata_summary']}")
        if report.get("graph_evidence_summary"):
            lines.append(f"\n**T\u00f3m t\u1eaft graph evidence:** {report['graph_evidence_summary']}")

    coverage = answer_json.get("coverage_assessment") or {}
    if coverage:
        lines.append("\n## 5. \u0110\u00e1nh gi\u00e1 \u0111\u1ed9 ph\u1ee7 v\u00e0 \u0111\u1ed9 tin c\u1eady")
        sufficient = coverage.get("is_result_sufficient")
        confidence = coverage.get("confidence") or "unknown"
        lines.append(f"- **K\u1ebft qu\u1ea3 \u0111\u1ee7 d\u00f9ng:** {sufficient}")
        lines.append(f"- **\u0110\u1ed9 tin c\u1eady:** {confidence}")
        if coverage.get("coverage_notes"):
            lines.append(f"- **Nh\u1eadn x\u00e9t:** {coverage['coverage_notes']}")
        actions = coverage.get("recommended_next_actions") or []
        if actions:
            lines.append("- **Khuy\u1ebfn ngh\u1ecb:**")
            for action in actions:
                lines.append(f"  - {action}")

    return "\n".join(lines).strip()


In [ ]:
RUN_GENERATION_SMOKE_TEST = os.getenv("RUN_GENERATION_SMOKE_TEST", "true").lower() == "true"
SMOKE_USE_RERANK = os.getenv("SMOKE_USE_RERANK", "false").lower() == "true"

if RUN_GENERATION_SMOKE_TEST:
    sample_plans, sample_views, _, _, _ = load_retrieval_inputs()
    sample_plan = sample_plans.iloc[0].to_dict()
    sample_topic_id = str(sample_plan["topic_id"])
    sample_topic_views = sample_views[sample_views["topic_id"].astype(str) == sample_topic_id].copy()

    print("[GENERATION SMOKE TEST]")
    print("topic_id:", sample_topic_id)
    print("views:", sample_topic_views["query_view"].tolist())

    best_retrieval, best_retrieval_method = load_best_retrieval_for_topic(sample_topic_id, dedup_canonical=True)
    print("best_retrieval_method:", best_retrieval_method or None)
    print("best_retrieval_docs_after_filter:", len(best_retrieval))
    print("use_rerank:", SMOKE_USE_RERANK)

    if not best_retrieval and not SMOKE_USE_RERANK:
        raise RuntimeError(
            "No existing hybrid_graph_adaptive/hybrid retrieval result found for the first query. "
            "Run retrieval + graph retrieval cells first."
        )

    es = get_es_client()
    if not es.ping():
        raise RuntimeError("Cannot connect to Elasticsearch.")

    sample_generation_result = run_full_pipeline_generation(
        es,
        sample_plan,
        sample_topic_views,
        use_rerank=SMOKE_USE_RERANK,
    )

    print("final_method:", sample_generation_result.get("final_method"))
    print("used_existing_rerank:", sample_generation_result.get("used_existing_rerank"))
    print("rerank_method:", sample_generation_result.get("rerank_method"))
    print("final_top_docs_filtered:", sample_generation_result.get("context_doc_ids"))
    print("context_doc_ids:", sample_generation_result.get("context_doc_ids"))
    print("latency_sec:", round(float(sample_generation_result.get("latency", 0.0)), 2))

    if sample_generation_result.get("retrieval_execution_plan"):
        print("\n[PLANNER AGENT PLAN]")
        print(json.dumps(sample_generation_result["retrieval_execution_plan"], ensure_ascii=False, indent=2))

    if sample_generation_result.get("agent_evaluation"):
        print("\n[EVALUATOR AGENT]")
        print(json.dumps(sample_generation_result["agent_evaluation"], ensure_ascii=False, indent=2))

    print("\n[GENERATION REPORT]")
    if sample_generation_result.get("answer_json") is not None and "render_generation_report" in globals():
        print(render_generation_report(sample_generation_result["answer_json"], topic_id=sample_topic_id))
    else:
        print(sample_generation_result.get("answer_raw", ""))

    print("\n[GENERATION JSON - DEBUG]")
    if sample_generation_result.get("answer_json") is not None:
        print(json.dumps(sample_generation_result["answer_json"], ensure_ascii=False, indent=2))
    else:
        print(sample_generation_result.get("answer_raw", ""))
else:
    print("[SKIP] Set RUN_GENERATION_SMOKE_TEST=true to run one-query generation test.")


[GENERATION SMOKE TEST]
topic_id: EP-1225393-A2
views: ['claims_short', 'combined_short', 'embedding_text', 'metadata', 'title_abstract_full']
best_retrieval_method: hybrid_graph_rank_rescore_g040_e020_gate010
best_retrieval_docs_after_filter: 300
use_rerank: False
